<a href="https://colab.research.google.com/github/kavyad4/llm-reliability-structured-extraction/blob/main/llm_researchipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from pathlib import Path
import urllib.request

url = "https://raw.githubusercontent.com/wyim/aci-bench/main/data/challenge_data_json/clinicalnlp_taskB_test1.json"

input_path = Path("/content/clinicalnlp_taskB_test1.json")

urllib.request.urlretrieve(url, input_path)

with input_path.open("r", encoding="utf-8") as f:
    source_data = json.load(f)

print("Loaded successfully!")
print("Number of records:", len(source_data["data"]))

Loaded successfully!
Number of records: 40


In [ ]:
# Check the dataset structure

print("Top-level keys:", source_data.keys())
print("Number of cases:", len(source_data["data"]))

# Look at the first case
first_case = source_data["data"][0]

print("\nCase ID:", first_case["file"])
print("\nAvailable fields:", first_case.keys())

print("\nTranscript preview:")
print(first_case["src"][:1000])

Top-level keys: dict_keys(['data'])
Number of cases: 40

Case ID: D2N088-virtassist

Available fields: dict_keys(['src', 'tgt', 'file'])

Transcript preview:
[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yea

In [ ]:
# Check the dataset structure

print("Top-level keys:", source_data.keys())
print("Number of cases:", len(source_data["data"]))

# Look at the first case
first_case = source_data["data"][0]

print("\nCase ID:", first_case["file"])
print("\nAvailable fields:", first_case.keys())

print("\nTranscript preview:")
print(first_case["src"][:1000])

Top-level keys: dict_keys(['data'])
Number of cases: 40

Case ID: D2N088-virtassist

Available fields: dict_keys(['src', 'tgt', 'file'])

Transcript preview:
[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yea

In [ ]:
# Create a clean working dataset

research_cases = []

for item in source_data["data"]:
    research_cases.append({
        "case_id": item["file"],
        "transcript": item["src"],
        "reference_note": item["tgt"]
    })

print("Number of research cases:", len(research_cases))

# Look at the first case
print("\nCase ID:")
print(research_cases[0]["case_id"])

print("\nTranscript:")
print(research_cases[0]["transcript"][:1000])

print("\nReference note:")
print(research_cases[0]["reference_note"][:1000])

Number of research cases: 40

Case ID:
D2N088-virtassist

Transcript:
[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yeah , both .
[doctor] okay . all right . and , um , do you have any history of any seasona

In [ ]:
import os

print("Current folder:", os.getcwd())
print("\nFiles:")
print(os.listdir())

Current folder: /content

Files:
['.config', 'clinicalnlp_taskB_test1.json', 'sample_data']


In [ ]:
import glob

files = glob.glob("/content/**/*clinical*schema*.json", recursive=True)

for f in files:
    print(f)

In [ ]:
from pathlib import Path

schema_path = Path("https://raw.githubusercontent.com/wyim/aci-bench/main/data/challenge_data_json/clinicalnlp_taskB_test1.json")

In [ ]:
# Cell 4: Define the structured clinical extraction schema

clinical_schema = {
    "type": "object",
    "additionalProperties": False,

    "required": [
        "patient",
        "encounter",
        "medical_history",
        "symptoms",
        "medications",
        "physical_exam",
        "diagnostic_tests",
        "assessment",
        "plan",
        "follow_up"
    ],

    "properties": {

        "patient": {
            "type": "object",
            "additionalProperties": False,
            "required": ["age", "sex"],
            "properties": {
                "age": {
                    "type": ["integer", "null"]
                },
                "sex": {
                    "type": ["string", "null"]
                }
            }
        },

        "encounter": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "visit_reason",
                "chief_complaint"
            ],
            "properties": {
                "visit_reason": {
                    "type": ["string", "null"]
                },
                "chief_complaint": {
                    "type": ["string", "null"]
                }
            }
        },

        "medical_history": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "condition",
                    "status"
                ],
                "properties": {
                    "condition": {
                        "type": ["string", "null"]
                    },
                    "status": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "symptoms": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "name",
                    "status",
                    "body_site",
                    "severity",
                    "duration"
                ],
                "properties": {
                    "name": {
                        "type": ["string", "null"]
                    },
                    "status": {
                        "type": ["string", "null"]
                    },
                    "body_site": {
                        "type": ["string", "null"]
                    },
                    "severity": {
                        "type": ["string", "null"]
                    },
                    "duration": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "medications": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "name",
                    "dosage",
                    "frequency",
                    "action"
                ],
                "properties": {
                    "name": {
                        "type": ["string", "null"]
                    },
                    "dosage": {
                        "type": ["string", "null"]
                    },
                    "frequency": {
                        "type": ["string", "null"]
                    },
                    "action": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "physical_exam": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "system",
                    "finding",
                    "body_site"
                ],
                "properties": {
                    "system": {
                        "type": ["string", "null"]
                    },
                    "finding": {
                        "type": ["string", "null"]
                    },
                    "body_site": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "diagnostic_tests": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "test_name",
                    "status",
                    "result"
                ],
                "properties": {
                    "test_name": {
                        "type": ["string", "null"]
                    },
                    "status": {
                        "type": ["string", "null"]
                    },
                    "result": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "assessment": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "condition",
                    "certainty"
                ],
                "properties": {
                    "condition": {
                        "type": ["string", "null"]
                    },
                    "certainty": {
                        "type": ["string", "null"]
                    }
                }
            }
        },

        "plan": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "medication_changes",
                "tests_ordered",
                "procedures",
                "referrals",
                "patient_instructions"
            ],
            "properties": {
                "medication_changes": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "tests_ordered": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "procedures": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "referrals": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "patient_instructions": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            }
        },

        "follow_up": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "value",
                "unit",
                "condition"
            ],
            "properties": {
                "value": {
                    "type": ["integer", "null"]
                },
                "unit": {
                    "type": ["string", "null"]
                },
                "condition": {
                    "type": ["string", "null"]
                }
            }
        }
    }
}

print("Clinical schema created successfully.")
print("Top-level fields:")
print(list(clinical_schema["properties"].keys()))

Clinical schema created successfully.
Top-level fields:
['patient', 'encounter', 'medical_history', 'symptoms', 'medications', 'physical_exam', 'diagnostic_tests', 'assessment', 'plan', 'follow_up']


In [ ]:
# Cell 5: Select the first case for manual gold annotation

case_index = 0

current_case = research_cases[case_index]

print("=" * 80)
print("CASE ID:", current_case["case_id"])
print("=" * 80)

print("\nTRANSCRIPT:\n")
print(current_case["transcript"])

print("\n" + "=" * 80)
print("REFERENCE NOTE:")
print("=" * 80)

print(current_case["reference_note"])

CASE ID: D2N088-virtassist

TRANSCRIPT:

[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yeah , both .
[doctor] okay . all right . and , um , do you have any history of any seasonal allergies at all ?
[patient

In [ ]:
# Create the gold_annotation variable

gold_annotation = {
    "case_id": current_case["case_id"],

    "gold_output": {},

    "evidence": {},

    "review_status": "not_reviewed"
}

print("Created gold_annotation for:", gold_annotation["case_id"])

Created gold_annotation for: D2N088-virtassist


In [ ]:
# Cell 7: Gold annotation for D2N088-virtassist

gold_annotation["gold_output"] = {

    "patient": {
        "age": 59,
        "sex": "male"
    },

    "encounter": {
        "visit_reason": "upper respiratory infection",
        "chief_complaint": "upper respiratory infection"
    },

    "medical_history": [
        {
            "condition": "depression",
            "status": "current"
        },
        {
            "condition": "type 2 diabetes",
            "status": "current"
        },
        {
            "condition": "hypertension",
            "status": "current"
        }
    ],

    "symptoms": [
        {
            "name": "fatigue",
            "status": "present",
            "body_site": None,
            "severity": None,
            "duration": "about one week"
        },
        {
            "name": "shortness of breath",
            "status": "present",
            "body_site": None,
            "severity": None,
            "duration": "about one week"
        },
        {
            "name": "elbow pain",
            "status": "present",
            "body_site": "bilateral elbows",
            "severity": None,
            "duration": None
        },
        {
            "name": "knee discomfort",
            "status": "present",
            "body_site": "bilateral knees",
            "severity": None,
            "duration": None
        },
        {
            "name": "nausea",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        },
        {
            "name": "vomiting",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        },
        {
            "name": "diarrhea",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        }
    ],

    "medications": [
        {
            "name": "metformin",
            "dosage": "1000 mg",
            "frequency": "twice daily",
            "action": "increase"
        },
        {
            "name": "lisinopril",
            "dosage": "20 mg",
            "frequency": "once daily",
            "action": "continue"
        },
        {
            "name": "Robitussin",
            "dosage": None,
            "frequency": None,
            "action": "recommend"
        },
        {
            "name": "ibuprofen",
            "dosage": None,
            "frequency": None,
            "action": "recommend as needed for fever"
        },
        {
            "name": "Tylenol",
            "dosage": None,
            "frequency": None,
            "action": "recommend as needed for fever"
        }
    ],

    "physical_exam": [
        {
            "system": "respiratory",
            "finding": "scattered rhonchi bilaterally clearing with cough",
            "body_site": "lungs"
        },
        {
            "system": "musculoskeletal",
            "finding": "edema",
            "body_site": "bilateral lower extremities"
        },
        {
            "system": "musculoskeletal",
            "finding": "pain to palpation",
            "body_site": "bilateral elbows"
        }
    ],

    "diagnostic_tests": [
        {
            "test_name": "chest x-ray",
            "status": "reviewed",
            "result": "no airspace disease or pneumonia"
        },
        {
            "test_name": "hemoglobin A1c",
            "status": "reviewed",
            "result": "8"
        },
        {
            "test_name": "COVID-19 test",
            "status": "ordered",
            "result": None
        },
        {
            "test_name": "hemoglobin A1c",
            "status": "ordered",
            "result": None
        },
        {
            "test_name": "lipid panel",
            "status": "ordered",
            "result": None
        }
    ],

    "assessment": [
        {
            "condition": "viral syndrome",
            "certainty": "suspected"
        },
        {
            "condition": "depression",
            "certainty": "confirmed"
        },
        {
            "condition": "type 2 diabetes",
            "certainty": "confirmed"
        },
        {
            "condition": "hypertension",
            "certainty": "confirmed"
        }
    ],

    "plan": {
        "medication_changes": [
            "increase metformin to 1000 mg twice daily",
            "continue lisinopril 20 mg once daily",
            "recommend Robitussin",
            "use ibuprofen or Tylenol if fever develops"
        ],

        "tests_ordered": [
            "COVID-19 test",
            "hemoglobin A1c in four months",
            "lipid panel"
        ],

        "procedures": [],

        "referrals": [],

        "patient_instructions": [
            "monitor symptoms",
            "contact clinician if symptoms worsen"
        ]
    },

    "follow_up": {
        "value": 4,
        "unit": "months",
        "condition": None
    }
}

gold_annotation["review_status"] = "manually_annotated"

print(json.dumps(gold_annotation["gold_output"], indent=2))

{
  "patient": {
    "age": 59,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "upper respiratory infection",
    "chief_complaint": "upper respiratory infection"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current"
    },
    {
      "condition": "hypertension",
      "status": "current"
    }
  ],
  "symptoms": [
    {
      "name": "fatigue",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "about one week"
    },
    {
      "name": "shortness of breath",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "about one week"
    },
    {
      "name": "elbow pain",
      "status": "present",
      "body_site": "bilateral elbows",
      "severity": null,
      "duration": null
    },
    {
      "name": "knee discomfort",
      "status": "present",
      "body_site": "bil

In [ ]:
# Cell 8: Validate gold annotation against the JSON schema

!pip -q install jsonschema

from jsonschema import Draft202012Validator

validator = Draft202012Validator(clinical_schema)

errors = sorted(
    validator.iter_errors(gold_annotation["gold_output"]),
    key=lambda e: list(e.absolute_path)
)

if not errors:
    print("✅ Gold annotation is VALID.")
else:
    print(f"❌ Found {len(errors)} validation error(s):\n")

    for i, error in enumerate(errors, start=1):
        path = ".".join(
            str(x) for x in error.absolute_path
        )

        if not path:
            path = "<root>"

        print(f"{i}. Field: {path}")
        print(f"   Error: {error.message}")
        print()

✅ Gold annotation is VALID.


In [ ]:
!pip -q install -U google-genai

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client created successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 21.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
Gemini client created successfully


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("✅ Gemini API key found")

✅ Gemini API key found


In [ ]:
# Fix: force Gemini Developer API instead of Vertex AI

import os

# Remove old Vertex AI configuration from this Colab runtime
for variable in [
    "GOOGLE_GENAI_USE_VERTEXAI",
    "GOOGLE_GENAI_USE_ENTERPRISE",
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_LOCATION"
]:
    os.environ.pop(variable, None)

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=GEMINI_API_KEY,
    vertexai=False
)

print("Using Vertex AI:", client.vertexai)

Using Vertex AI: False


In [ ]:
# Fresh Gemini Developer API client
# Do NOT reuse the old "client" variable

import os
from google import genai
from google.colab import userdata

# Remove any Vertex/Cloud configuration left in this notebook
for var in [
    "GOOGLE_GENAI_USE_VERTEXAI",
    "GOOGLE_GENAI_USE_ENTERPRISE",
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_LOCATION"
]:
    os.environ.pop(var, None)


# Get Gemini API key from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")


# Create a NEW client name
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY,
    enterprise=False,
    vertexai=False
)


# IMPORTANT CHECK
print("Using Vertex AI:", gemini_client.vertexai)

assert gemini_client.vertexai is False, \
    "Client is still configured for Vertex AI"


print("✅ Gemini Developer API client created")

Using Vertex AI: False
✅ Gemini Developer API client created


In [ ]:
test_response = gemini_client.interactions.create(
    model="gemini-3.6-flash",
    input="Reply with exactly: API working"
)

print(test_response.output_text)

API working


In [ ]:
# Baseline prompt

baseline_prompt = f"""
You are extracting structured clinical information from a doctor-patient transcript.

Rules:
1. Use only information explicitly supported by the transcript.
2. Do not invent missing information.
3. If a scalar value is unknown, use null.
4. If there are no items for a list, use [].
5. Preserve negated symptoms using status "denied".
6. Distinguish current, historical, ordered, planned, and completed information.
7. Return JSON only.
8. Do not include explanations outside the JSON.

Return the information using this structure:

{{
  "patient": {{
    "age": null,
    "sex": null
  }},
  "encounter": {{
    "visit_reason": null,
    "chief_complaint": null
  }},
  "medical_history": [
    {{
      "condition": null,
      "status": null
    }}
  ],
  "symptoms": [
    {{
      "name": null,
      "status": null,
      "body_site": null,
      "severity": null,
      "duration": null
    }}
  ],
  "medications": [
    {{
      "name": null,
      "dosage": null,
      "frequency": null,
      "action": null
    }}
  ],
  "physical_exam": [
    {{
      "system": null,
      "finding": null,
      "body_site": null
    }}
  ],
  "diagnostic_tests": [
    {{
      "test_name": null,
      "status": null,
      "result": null
    }}
  ],
  "assessment": [
    {{
      "condition": null,
      "certainty": null
    }}
  ],
  "plan": {{
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  }},
  "follow_up": {{
    "value": null,
    "unit": null,
    "condition": null
  }}
}}

TRANSCRIPT:

{current_case["transcript"]}
"""

In [ ]:
# Cell 9: Baseline LLM structured extraction
# Gemini Developer API version

!pip -q install -U google-genai

import os
import json

from google import genai
from google.colab import userdata


# ---------------------------------
# REMOVE OLD VERTEX CONFIGURATION
# ---------------------------------

for var in [
    "GOOGLE_GENAI_USE_VERTEXAI",
    "GOOGLE_GENAI_USE_ENTERPRISE",
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_LOCATION"
]:
    os.environ.pop(var, None)


# ---------------------------------
# LOAD GEMINI API KEY
# ---------------------------------

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")


# Gemini Developer API client
# NOT Vertex AI
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ---------------------------------
# EMPTY OUTPUT STRUCTURE
# ---------------------------------

output_template = {

    "patient": {
        "age": None,
        "sex": None
    },

    "encounter": {
        "visit_reason": None,
        "chief_complaint": None
    },

    "medical_history": [],

    "symptoms": [],

    "medications": [],

    "physical_exam": [],

    "diagnostic_tests": [],

    "assessment": [],

    "plan": {
        "medication_changes": [],
        "tests_ordered": [],
        "procedures": [],
        "referrals": [],
        "patient_instructions": []
    },

    "follow_up": {
        "value": None,
        "unit": None,
        "condition": None
    }
}


# ---------------------------------
# BASELINE PROMPT
# ---------------------------------

baseline_prompt = f"""
Extract structured clinical information from the transcript.

Rules:
- Use only information supported by the transcript.
- Do not invent missing information.
- Use null when a scalar value is unknown.
- Use [] when no items are supported.
- Preserve negated symptoms using status = "denied".
- Distinguish current, historical, ordered, planned,
  and completed information.
- Return JSON only.
- Do not include explanations.

Required output structure:

{json.dumps(output_template, indent=2)}

TRANSCRIPT:

{current_case["transcript"]}
"""


# ---------------------------------
# CALL GEMINI
# ---------------------------------

response = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=baseline_prompt
)


baseline_raw = response.text


print("RAW MODEL RESPONSE:")
print(baseline_raw)

RAW MODEL RESPONSE:
{
  "patient": {
    "age": 59,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "upper respiratory infection",
    "chief_complaint": "shortness of breath and fatigue"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current"
    },
    {
      "condition": "hypertension",
      "status": "current"
    },
    {
      "condition": "seasonal allergies",
      "status": "denied"
    }
  ],
  "symptoms": [
    {
      "name": "fatigue",
      "status": "current"
    },
    {
      "name": "shortness of breath",
      "status": "current"
    },
    {
      "name": "coughing up sputum",
      "status": "denied"
    },
    {
      "name": "fever",
      "status": "denied"
    },
    {
      "name": "elbow pain",
      "status": "current"
    },
    {
      "name": "knee fatigue and tension",
      "status": "current"
    },
    {
      "name": "nausea

In [ ]:
# Cell 10: Compare baseline LLM output with gold annotation

import json
import re
from jsonschema import Draft202012Validator


# -----------------------------------
# 1. CLEAN MODEL RESPONSE
# -----------------------------------

clean_baseline = baseline_raw.strip()

# Remove ```json ... ``` if Gemini returned markdown fences
clean_baseline = re.sub(
    r"^```json\s*",
    "",
    clean_baseline,
    flags=re.IGNORECASE
)

clean_baseline = re.sub(
    r"\s*```$",
    "",
    clean_baseline
)


# -----------------------------------
# 2. PARSE JSON
# -----------------------------------

try:
    baseline_output = json.loads(clean_baseline)
    print("✅ Baseline response is valid JSON")
except json.JSONDecodeError as e:
    print("❌ Baseline response is NOT valid JSON")
    print(e)
    baseline_output = None


# -----------------------------------
# 3. VALIDATE AGAINST OUR SCHEMA
# -----------------------------------

if baseline_output is not None:

    validator = Draft202012Validator(
        clinical_schema
    )

    schema_errors = sorted(
        validator.iter_errors(baseline_output),
        key=lambda e: list(e.absolute_path)
    )

    print("\n" + "=" * 70)
    print("SCHEMA VALIDATION")
    print("=" * 70)

    if not schema_errors:
        print("✅ Baseline follows the clinical schema")
    else:
        print(
            f"❌ Baseline has "
            f"{len(schema_errors)} schema error(s)"
        )

        for i, error in enumerate(
            schema_errors,
            start=1
        ):

            path = ".".join(
                str(x)
                for x in error.absolute_path
            )

            if not path:
                path = "<root>"

            print(
                f"\n{i}. Path: {path}"
            )

            print(
                f"   Error: {error.message}"
            )


# -----------------------------------
# 4. GET GOLD OUTPUT
# -----------------------------------

gold_output = gold_annotation["gold_output"]


# -----------------------------------
# 5. COMPARE TOP-LEVEL SECTIONS
# -----------------------------------

print("\n" + "=" * 70)
print("GOLD VS BASELINE")
print("=" * 70)


for section in gold_output.keys():

    gold_section = gold_output.get(section)
    baseline_section = baseline_output.get(
        section
    )

    match = (
        gold_section == baseline_section
    )

    print(
        f"\n{section}: "
        f"{'✅ MATCH' if match else '❌ DIFFERENT'}"
    )


# -----------------------------------
# 6. FULL EXACT MATCH
# -----------------------------------

exact_match = (
    baseline_output == gold_output
)

print("\n" + "=" * 70)
print("EXACT RECORD MATCH")
print("=" * 70)

print(
    "✅ Exact match"
    if exact_match
    else "❌ Not an exact match"
)

✅ Baseline response is valid JSON

SCHEMA VALIDATION
❌ Baseline has 59 schema error(s)

1. Path: assessment.0
   Error: Additional properties are not allowed ('status' was unexpected)

2. Path: assessment.0
   Error: 'certainty' is a required property

3. Path: assessment.1
   Error: Additional properties are not allowed ('status' was unexpected)

4. Path: assessment.1
   Error: 'certainty' is a required property

5. Path: assessment.2
   Error: Additional properties are not allowed ('status' was unexpected)

6. Path: assessment.2
   Error: 'certainty' is a required property

7. Path: assessment.3
   Error: Additional properties are not allowed ('status' was unexpected)

8. Path: assessment.3
   Error: 'certainty' is a required property

9. Path: assessment.4
   Error: Additional properties are not allowed ('status' was unexpected)

10. Path: assessment.4
   Error: 'certainty' is a required property

11. Path: diagnostic_tests.0
   Error: Additional properties are not allowed ('test' w

In [ ]:
from collections import Counter
from jsonschema import Draft202012Validator
import json

# --------------------------------------------------
# 1. Validate the baseline output against our schema
# --------------------------------------------------

validator = Draft202012Validator(clinical_schema)

schema_errors = sorted(
    validator.iter_errors(baseline_output),
    key=lambda e: list(e.absolute_path)
)

# --------------------------------------------------
# 2. Count schema error types
# --------------------------------------------------

schema_error_types = Counter(
    error.validator for error in schema_errors
)

# --------------------------------------------------
# 3. Compare each top-level section with gold
# --------------------------------------------------

section_matches = {}

for section in gold_output.keys():
    section_matches[section] = (
        gold_output.get(section)
        == baseline_output.get(section)
    )

exact_section_matches = sum(section_matches.values())
total_sections = len(section_matches)

# --------------------------------------------------
# 4. Store Case 1 baseline metrics
# --------------------------------------------------

baseline_metrics_case1 = {
    "case_id": current_case["case_id"],
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    "json_valid": 1,

    "schema_valid": int(len(schema_errors) == 0),

    "schema_error_count": len(schema_errors),

    "required_field_errors":
        schema_error_types.get("required", 0),

    "additional_property_errors":
        schema_error_types.get(
            "additionalProperties", 0
        ),

    "type_errors":
        schema_error_types.get("type", 0),

    "exact_section_matches":
        exact_section_matches,

    "total_sections":
        total_sections,

    "exact_section_match_rate":
        exact_section_matches / total_sections,

    "exact_record_match":
        int(baseline_output == gold_output)
}

print("=" * 60)
print("CASE 1 BASELINE METRICS")
print("=" * 60)

print(
    json.dumps(
        baseline_metrics_case1,
        indent=2
    )
)

CASE 1 BASELINE METRICS
{
  "case_id": "D2N088-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 59,
  "required_field_errors": 44,
  "additional_property_errors": 10,
  "type_errors": 5,
  "exact_section_matches": 2,
  "total_sections": 10,
  "exact_section_match_rate": 0.2,
  "exact_record_match": 0
}


In [ ]:
# --------------------------------------------------
# CASE 1 - MANUAL SEMANTIC ERROR REVIEW
# --------------------------------------------------

semantic_errors_case1 = [
    {
        "field": "patient.age",
        "gold_value": 59,
        "model_value": 59,
        "classification": "correct_extraction",
        "notes": "Correctly extracted age"
    },
    {
        "field": "patient.sex",
        "gold_value": "male",
        "model_value": "male",
        "classification": "correct_extraction",
        "notes": "Correctly extracted sex"
    },
    {
        "field": "medications",
        "gold_value": None,
        "model_value": "COVID-19 vaccine",
        "classification": "mapping_error",
        "notes": "Vaccination information was placed under medications"
    },
    {
        "field": "diagnostic_tests.COVID-19 test",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "COVID-19 test was ordered but missing from diagnostic_tests"
    },
    {
        "field": "diagnostic_tests.lipid_panel",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "Lipid panel was ordered but missing from diagnostic_tests"
    },
    {
        "field": "assessment.viral_syndrome.certainty",
        "gold_value": "suspected",
        "model_value": None,
        "classification": "status_error",
        "notes": "Model did not preserve diagnostic certainty"
    },
    {
        "field": "plan.patient_instructions",
        "gold_value": None,
        "model_value": "continue monitoring blood pressure and blood sugar levels at home",
        "classification": "possible_hallucination",
        "notes": "Needs transcript evidence review before labeling as hallucination"
    }
]

for item in semantic_errors_case1:
    print(
        item["classification"],
        "->",
        item["field"]
    )

correct_extraction -> patient.age
correct_extraction -> patient.sex
mapping_error -> medications
omission -> diagnostic_tests.COVID-19 test
omission -> diagnostic_tests.lipid_panel
status_error -> assessment.viral_syndrome.certainty
possible_hallucination -> plan.patient_instructions


In [ ]:
# --------------------------------------------------
# CASE 1 - MANUAL SEMANTIC ERROR REVIEW
# --------------------------------------------------

semantic_errors_case1 = [
    {
        "field": "patient.age",
        "gold_value": 59,
        "model_value": 59,
        "classification": "correct_extraction",
        "notes": "Correctly extracted age"
    },
    {
        "field": "patient.sex",
        "gold_value": "male",
        "model_value": "male",
        "classification": "correct_extraction",
        "notes": "Correctly extracted sex"
    },
    {
        "field": "medications",
        "gold_value": None,
        "model_value": "COVID-19 vaccine",
        "classification": "mapping_error",
        "notes": "Vaccination information was placed under medications"
    },
    {
        "field": "diagnostic_tests.COVID-19 test",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "COVID-19 test was ordered but missing from diagnostic_tests"
    },
    {
        "field": "diagnostic_tests.lipid_panel",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "Lipid panel was ordered but missing from diagnostic_tests"
    },
    {
        "field": "assessment.viral_syndrome.certainty",
        "gold_value": "suspected",
        "model_value": None,
        "classification": "status_error",
        "notes": "Model did not preserve diagnostic certainty"
    },
    {
        "field": "plan.patient_instructions",
        "gold_value": None,
        "model_value": "continue monitoring blood pressure and blood sugar levels at home",
        "classification": "possible_hallucination",
        "notes": "Needs transcript evidence review before labeling as hallucination"
    }
]

for item in semantic_errors_case1:
    print(
        item["classification"],
        "->",
        item["field"]
    )

correct_extraction -> patient.age
correct_extraction -> patient.sex
mapping_error -> medications
omission -> diagnostic_tests.COVID-19 test
omission -> diagnostic_tests.lipid_panel
status_error -> assessment.viral_syndrome.certainty
possible_hallucination -> plan.patient_instructions


In [ ]:
import pandas as pd

semantic_review_df = pd.DataFrame(
    semantic_errors_case1
)

print("=" * 80)
print("CASE 1 SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df)

CASE 1 SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,59,59,correct_extraction,Correctly extracted age
1,patient.sex,male,male,correct_extraction,Correctly extracted sex
2,medications,None,COVID-19 vaccine,mapping_error,Vaccination information was placed under medic...
3,diagnostic_tests.COVID-19 test,ordered,None,omission,COVID-19 test was ordered but missing from dia...
4,diagnostic_tests.lipid_panel,ordered,None,omission,Lipid panel was ordered but missing from diagn...
5,assessment.viral_syndrome.certainty,suspected,None,status_error,Model did not preserve diagnostic certainty
6,plan.patient_instructions,None,continue monitoring blood pressure and blood s...,possible_hallucination,Needs transcript evidence review before labeli...


In [ ]:
# --------------------------------------------------
# CASE 1 - COMPLETE SEMANTIC REVIEW
# --------------------------------------------------

semantic_errors_case1 = [

    # -------------------------
    # PATIENT
    # -------------------------
    {
        "field": "patient.age",
        "gold_value": 59,
        "model_value": 59,
        "classification": "correct_extraction",
        "notes": "Correctly extracted age"
    },
    {
        "field": "patient.sex",
        "gold_value": "male",
        "model_value": "male",
        "classification": "correct_extraction",
        "notes": "Correctly extracted sex"
    },

    # -------------------------
    # MEDICAL HISTORY
    # -------------------------
    {
        "field": "medical_history.depression",
        "gold_value": "current",
        "model_value": "active",
        "classification": "status_label_difference",
        "notes": "Condition correctly extracted; status wording differs"
    },
    {
        "field": "medical_history.type_2_diabetes",
        "gold_value": "current",
        "model_value": "active",
        "classification": "status_label_difference",
        "notes": "Condition correctly extracted; status wording differs"
    },
    {
        "field": "medical_history.hypertension",
        "gold_value": "current",
        "model_value": "active",
        "classification": "status_label_difference",
        "notes": "Condition correctly extracted; status wording differs"
    },
    {
        "field": "medical_history.seasonal_allergies",
        "gold_value": None,
        "model_value": "denied",
        "classification": "mapping_error",
        "notes": "Denied allergy history was placed inside medical_history"
    },

    # -------------------------
    # SYMPTOMS
    # -------------------------
    {
        "field": "symptoms.fatigue",
        "gold_value": "present",
        "model_value": "reported",
        "classification": "correct_extraction",
        "notes": "Symptom correctly identified; label wording differs"
    },
    {
        "field": "symptoms.shortness_of_breath",
        "gold_value": "present",
        "model_value": "reported",
        "classification": "correct_extraction",
        "notes": "Correctly extracted symptom"
    },
    {
        "field": "symptoms.elbow_pain",
        "gold_value": "present",
        "model_value": "reported",
        "classification": "correct_extraction",
        "notes": "Correctly extracted symptom"
    },
    {
        "field": "symptoms.knee_discomfort",
        "gold_value": "present",
        "model_value": "reported",
        "classification": "correct_extraction",
        "notes": "Correct symptom concept despite wording difference"
    },
    {
        "field": "symptoms.nausea",
        "gold_value": "denied",
        "model_value": "denied",
        "classification": "correct_extraction",
        "notes": "Correctly preserved negation"
    },
    {
        "field": "symptoms.vomiting",
        "gold_value": "denied",
        "model_value": "denied",
        "classification": "correct_extraction",
        "notes": "Correctly preserved negation"
    },
    {
        "field": "symptoms.diarrhea",
        "gold_value": "denied",
        "model_value": "denied",
        "classification": "correct_extraction",
        "notes": "Correctly preserved negation"
    },
    {
        "field": "symptoms.fever",
        "gold_value": None,
        "model_value": "denied",
        "classification": "needs_review",
        "notes": "Transcript evidence should be checked before calling this hallucination"
    },

    # -------------------------
    # MEDICATIONS
    # -------------------------
    {
        "field": "medications.metformin",
        "gold_value": "increase to 1000 mg twice daily",
        "model_value": "current; dosage not correctly represented",
        "classification": "partial_extraction",
        "notes": "Medication identified but action/frequency representation is incomplete"
    },
    {
        "field": "medications.lisinopril",
        "gold_value": "continue 20 mg once daily",
        "model_value": "20 mg daily; current",
        "classification": "partial_extraction",
        "notes": "Medication and dose correct; action field missing"
    },
    {
        "field": "medications.COVID_vaccine",
        "gold_value": None,
        "model_value": "COVID-19 vaccine, 2 doses",
        "classification": "mapping_error",
        "notes": "Vaccination history incorrectly mapped to medications"
    },

    # -------------------------
    # PHYSICAL EXAM
    # -------------------------
    {
        "field": "physical_exam.respiratory",
        "gold_value": "scattered bilateral rhonchi clearing with cough",
        "model_value": "scattered rhonchi bilaterally, clears with cough",
        "classification": "correct_extraction",
        "notes": "Finding correctly extracted"
    },
    {
        "field": "physical_exam.lower_extremity_edema",
        "gold_value": "edema",
        "model_value": "mild lower extremity edema",
        "classification": "correct_extraction",
        "notes": "Finding correctly extracted"
    },
    {
        "field": "physical_exam.elbow_pain",
        "gold_value": "pain to palpation",
        "model_value": "pain to palpation of elbows bilaterally",
        "classification": "correct_extraction",
        "notes": "Finding correctly extracted"
    },

    # -------------------------
    # DIAGNOSTIC TESTS
    # -------------------------
    {
        "field": "diagnostic_tests.chest_xray",
        "gold_value": "no airspace disease or pneumonia",
        "model_value": "no airspace disease or pneumonia",
        "classification": "correct_extraction",
        "notes": "Correct test result"
    },
    {
        "field": "diagnostic_tests.a1c_result",
        "gold_value": "8",
        "model_value": "8%",
        "classification": "correct_extraction",
        "notes": "Same clinical result with minor formatting difference"
    },
    {
        "field": "diagnostic_tests.COVID_test",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "Ordered COVID-19 test missing from diagnostic_tests"
    },
    {
        "field": "diagnostic_tests.repeat_a1c",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "Repeat A1c missing from diagnostic_tests"
    },
    {
        "field": "diagnostic_tests.lipid_panel",
        "gold_value": "ordered",
        "model_value": None,
        "classification": "omission",
        "notes": "Lipid panel missing from diagnostic_tests"
    },

    # -------------------------
    # ASSESSMENT
    # -------------------------
    {
        "field": "assessment.viral_syndrome",
        "gold_value": "suspected",
        "model_value": "upper respiratory infection / viral syndrome",
        "classification": "status_error",
        "notes": "Condition identified but certainty was not preserved"
    },
    {
        "field": "assessment.depression",
        "gold_value": "confirmed",
        "model_value": "depression",
        "classification": "partial_extraction",
        "notes": "Condition identified but certainty omitted"
    },
    {
        "field": "assessment.type_2_diabetes",
        "gold_value": "confirmed",
        "model_value": "type 2 diabetes",
        "classification": "partial_extraction",
        "notes": "Condition identified but certainty omitted"
    },
    {
        "field": "assessment.hypertension",
        "gold_value": "confirmed",
        "model_value": "hypertension",
        "classification": "partial_extraction",
        "notes": "Condition identified but certainty omitted"
    },

    # -------------------------
    # PLAN
    # -------------------------
    {
        "field": "plan.metformin_change",
        "gold_value": "increase metformin to 1000 mg twice daily",
        "model_value": "increase metformin to 1000 mg twice daily",
        "classification": "correct_extraction",
        "notes": "Correct plan item"
    },
    {
        "field": "plan.lisinopril",
        "gold_value": "continue lisinopril 20 mg once daily",
        "model_value": "refill lisinopril 20 mg daily",
        "classification": "partial_extraction",
        "notes": "Medication and dose correct; action wording differs"
    },
    {
        "field": "plan.tests_ordered",
        "gold_value": ["COVID-19 test", "A1c", "lipid panel"],
        "model_value": ["COVID-19 test", "Hemoglobin A1c", "Lipid panel"],
        "classification": "correct_extraction",
        "notes": "Ordered tests correctly identified"
    },
    {
        "field": "plan.patient_instructions.monitoring",
        "gold_value": None,
        "model_value": "continue monitoring blood pressure and blood sugar levels at home",
        "classification": "needs_review",
        "notes": "May be inferred rather than explicitly stated; verify transcript evidence"
    },

    # -------------------------
    # FOLLOW-UP
    # -------------------------
    {
        "field": "follow_up",
        "gold_value": "4 months",
        "model_value": "4 months",
        "classification": "correct_extraction",
        "notes": "Exact follow-up match"
    }
]

semantic_review_df = pd.DataFrame(semantic_errors_case1)

display(semantic_review_df)

,field,gold_value,model_value,classification,notes
0,patient.age,59,59,correct_extraction,Correctly extracted age
1,patient.sex,male,male,correct_extraction,Correctly extracted sex
2,medical_history.depression,current,active,status_label_difference,Condition correctly extracted; status wording ...
3,medical_history.type_2_diabetes,current,active,status_label_difference,Condition correctly extracted; status wording ...
4,medical_history.hypertension,current,active,status_label_difference,Condition correctly extracted; status wording ...
5,medical_history.seasonal_allergies,None,denied,mapping_error,Denied allergy history was placed inside medic...
6,symptoms.fatigue,present,reported,correct_extraction,Symptom correctly identified; label wording di...
7,symptoms.shortness_of_breath,present,reported,correct_extraction,Correctly extracted symptom
8,symptoms.elbow_pain,present,reported,correct_extraction,Correctly extracted symptom
9,symptoms.knee_discomfort,present,reported,correct_extraction,Correct symptom concept despite wording differ...


In [ ]:
# --------------------------------------------------
# CASE 1 - NORMALIZE SEMANTIC REVIEW
# --------------------------------------------------

# Convert harmless wording differences into correct extractions
semantic_review_df.loc[
    semantic_review_df["classification"] == "status_label_difference",
    "classification"
] = "correct_extraction"

semantic_review_df.loc[
    semantic_review_df["classification"] == "status_label_difference",
    "notes"
] = "Correct clinical meaning after status normalization"

print("=" * 80)
print("CASE 1 SEMANTIC REVIEW AFTER NORMALIZATION")
print("=" * 80)

display(semantic_review_df)

CASE 1 SEMANTIC REVIEW AFTER NORMALIZATION


,field,gold_value,model_value,classification,notes
0,patient.age,59,59,correct_extraction,Correctly extracted age
1,patient.sex,male,male,correct_extraction,Correctly extracted sex
2,medical_history.depression,current,active,correct_extraction,Condition correctly extracted; status wording ...
3,medical_history.type_2_diabetes,current,active,correct_extraction,Condition correctly extracted; status wording ...
4,medical_history.hypertension,current,active,correct_extraction,Condition correctly extracted; status wording ...
5,medical_history.seasonal_allergies,None,denied,mapping_error,Denied allergy history was placed inside medic...
6,symptoms.fatigue,present,reported,correct_extraction,Symptom correctly identified; label wording di...
7,symptoms.shortness_of_breath,present,reported,correct_extraction,Correctly extracted symptom
8,symptoms.elbow_pain,present,reported,correct_extraction,Correctly extracted symptom
9,symptoms.knee_discomfort,present,reported,correct_extraction,Correct symptom concept despite wording differ...


In [ ]:
# --------------------------------------------------
# CASE 1 - RESOLVE NEEDS_REVIEW ITEMS
# --------------------------------------------------

# 1. Fever was explicitly denied in the transcript
semantic_review_df.loc[
    semantic_review_df["field"] == "symptoms.fever",
    "classification"
] = "correct_extraction"

semantic_review_df.loc[
    semantic_review_df["field"] == "symptoms.fever",
    "notes"
] = "Patient explicitly denied fever; correctly extracted"


# 2. Home monitoring existed in history,
# but was not explicitly given as a new plan instruction
semantic_review_df.loc[
    semantic_review_df["field"] ==
    "plan.patient_instructions.monitoring",
    "classification"
] = "unsupported_inference"

semantic_review_df.loc[
    semantic_review_df["field"] ==
    "plan.patient_instructions.monitoring",
    "notes"
] = (
    "Patient was already monitoring BP and glucose, "
    "but clinician did not explicitly instruct continued "
    "home monitoring in the final plan"
)


print("=" * 80)
print("CASE 1 FINAL SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df)

CASE 1 FINAL SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,59,59,correct_extraction,Correctly extracted age
1,patient.sex,male,male,correct_extraction,Correctly extracted sex
2,medical_history.depression,current,active,correct_extraction,Condition correctly extracted; status wording ...
3,medical_history.type_2_diabetes,current,active,correct_extraction,Condition correctly extracted; status wording ...
4,medical_history.hypertension,current,active,correct_extraction,Condition correctly extracted; status wording ...
5,medical_history.seasonal_allergies,None,denied,mapping_error,Denied allergy history was placed inside medic...
6,symptoms.fatigue,present,reported,correct_extraction,Symptom correctly identified; label wording di...
7,symptoms.shortness_of_breath,present,reported,correct_extraction,Correctly extracted symptom
8,symptoms.elbow_pain,present,reported,correct_extraction,Correctly extracted symptom
9,symptoms.knee_discomfort,present,reported,correct_extraction,Correct symptom concept despite wording differ...


In [ ]:
# --------------------------------------------------
# CASE 1 - SEMANTIC ERROR COUNTS
# --------------------------------------------------

semantic_counts = (
    semantic_review_df["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 1 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts.items():
    print(f"{category}: {count}")

CASE 1 SEMANTIC COUNTS
correct_extraction: 21
partial_extraction: 6
omission: 3
mapping_error: 2
status_error: 1
unsupported_inference: 1


In [ ]:
# --------------------------------------------------
# CASE 1 - FINAL BASELINE RESULT
# --------------------------------------------------

semantic_counts = (
    semantic_review_df["classification"]
    .value_counts()
    .to_dict()
)

case1_final_result = {

    # Identification
    "case_id": current_case["case_id"],
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    # -------------------------
    # STRUCTURAL RESULTS
    # -------------------------
    "json_valid": baseline_metrics_case1["json_valid"],
    "schema_valid": baseline_metrics_case1["schema_valid"],
    "schema_error_count":
        baseline_metrics_case1["schema_error_count"],

    "required_field_errors":
        baseline_metrics_case1["required_field_errors"],

    "additional_property_errors":
        baseline_metrics_case1["additional_property_errors"],

    "type_errors":
        baseline_metrics_case1["type_errors"],

    "exact_section_matches":
        baseline_metrics_case1["exact_section_matches"],

    "total_sections":
        baseline_metrics_case1["total_sections"],

    "exact_section_match_rate":
        baseline_metrics_case1["exact_section_match_rate"],

    "exact_record_match":
        baseline_metrics_case1["exact_record_match"],

    # -------------------------
    # SEMANTIC RESULTS
    # -------------------------
    "semantic_items_reviewed":
        len(semantic_review_df),

    "correct_extractions":
        semantic_counts.get("correct_extraction", 0),

    "partial_extractions":
        semantic_counts.get("partial_extraction", 0),

    "omissions":
        semantic_counts.get("omission", 0),

    "mapping_errors":
        semantic_counts.get("mapping_error", 0),

    "status_errors":
        semantic_counts.get("status_error", 0),

    "unsupported_inferences":
        semantic_counts.get("unsupported_inference", 0)
}

print("=" * 70)
print("CASE 1 FINAL BASELINE RESULT")
print("=" * 70)

print(json.dumps(
    case1_final_result,
    indent=2
))

CASE 1 FINAL BASELINE RESULT
{
  "case_id": "D2N088-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 59,
  "required_field_errors": 44,
  "additional_property_errors": 10,
  "type_errors": 5,
  "exact_section_matches": 2,
  "total_sections": 10,
  "exact_section_match_rate": 0.2,
  "exact_record_match": 0,
  "semantic_items_reviewed": 34,
  "correct_extractions": 21,
  "partial_extractions": 6,
  "omissions": 3,
  "mapping_errors": 2,
  "status_errors": 1,
  "unsupported_inferences": 1
}


In [ ]:
# --------------------------------------------------
# FREEZE REVIEWED CASE 1 RESULT
# --------------------------------------------------

case1_final_result = {
    "case_id": "D2N088-virtassist",
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    # Structural metrics from the SAME output
    # used for the manual semantic review
    "json_valid": 1,
    "schema_valid": 0,
    "schema_error_count": 70,
    "required_field_errors": 47,
    "additional_property_errors": 15,
    "type_errors": 8,

    "exact_section_matches": 2,
    "total_sections": 10,
    "exact_section_match_rate": 0.2,
    "exact_record_match": 0,

    # Semantic review of that same output
    "semantic_items_reviewed": 34,
    "correct_extractions": 21,
    "partial_extractions": 6,
    "omissions": 3,
    "mapping_errors": 2,
    "status_errors": 1,
    "unsupported_inferences": 1
}

print(json.dumps(case1_final_result, indent=2))

{
  "case_id": "D2N088-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 70,
  "required_field_errors": 47,
  "additional_property_errors": 15,
  "type_errors": 8,
  "exact_section_matches": 2,
  "total_sections": 10,
  "exact_section_match_rate": 0.2,
  "exact_record_match": 0,
  "semantic_items_reviewed": 34,
  "correct_extractions": 21,
  "partial_extractions": 6,
  "omissions": 3,
  "mapping_errors": 2,
  "status_errors": 1,
  "unsupported_inferences": 1
}


In [ ]:
# --------------------------------------------------
# SHOW CASES 2-5
# --------------------------------------------------

for i in range(1, 5):
    print(
        f"Case {i + 1}:",
        research_cases[i]["case_id"]
    )

Case 2: D2N089-virtassist
Case 3: D2N090-virtassist
Case 4: D2N091-virtassist
Case 5: D2N092-virtassist


In [ ]:
# --------------------------------------------------
# CASE 2 - SELECT AND INSPECT
# --------------------------------------------------

case_index = 1
current_case = research_cases[case_index]

print("=" * 80)
print("CASE 2")
print("=" * 80)

print("CASE ID:", current_case["case_id"])

print("\n" + "=" * 80)
print("TRANSCRIPT")
print("=" * 80)
print(current_case["transcript"])

print("\n" + "=" * 80)
print("REFERENCE NOTE")
print("=" * 80)
print(current_case["reference_note"])

CASE 2
CASE ID: D2N089-virtassist

TRANSCRIPT
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year-old female with a past medical history significant for rheumatoid arthritis , atrial fibrillation , and reflux who presents today for her annual exam . so andrea , it's been a year since i saw you . how are you doing ?
[patient] i'm doing well . so , i've been walking like you told me to and , um , exercising and doing yoga , and that's actually helped with my arthritis a lot , just the- the constant movement . so , i have n't had any joint pain recently .
[doctor] okay . good . so , no- no issues with any stiffness or pain or flare ups over the last year ?
[patient] no .
[doctor] okay . and i know that we have you on the methotrexate , are you still taking that once a week ?
[patient] yes , i am 

In [ ]:
# --------------------------------------------------
# CASE 2 - CREATE GOLD ANNOTATION CONTAINER
# --------------------------------------------------

gold_annotation_case2 = {
    "case_id": current_case["case_id"],

    "gold_output": {
        "patient": {
            "age": None,
            "sex": None
        },

        "encounter": {
            "visit_reason": None,
            "chief_complaint": None
        },

        "medical_history": [],

        "symptoms": [],

        "medications": [],

        "physical_exam": [],

        "diagnostic_tests": [],

        "assessment": [],

        "plan": {
            "medication_changes": [],
            "tests_ordered": [],
            "procedures": [],
            "referrals": [],
            "patient_instructions": []
        },

        "follow_up": {
            "value": None,
            "unit": None,
            "condition": None
        }
    },

    "evidence": {},

    "review_status": "not_reviewed"
}

print("Gold annotation created for:")
print(gold_annotation_case2["case_id"])

print("\nEmpty gold structure:")
print(
    json.dumps(
        gold_annotation_case2["gold_output"],
        indent=2
    )
)

Gold annotation created for:
D2N089-virtassist

Empty gold structure:
{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}


In [ ]:
# --------------------------------------------------
# CASE 2 - REVIEW SOURCE BEFORE GOLD ANNOTATION
# --------------------------------------------------

print("=" * 80)
print("CASE 2:", current_case["case_id"])
print("=" * 80)

print("\nTRANSCRIPT:")
print("-" * 80)
print(current_case["transcript"])

print("\nREFERENCE NOTE:")
print("-" * 80)
print(current_case["reference_note"])

CASE 2: D2N089-virtassist

TRANSCRIPT:
--------------------------------------------------------------------------------
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year-old female with a past medical history significant for rheumatoid arthritis , atrial fibrillation , and reflux who presents today for her annual exam . so andrea , it's been a year since i saw you . how are you doing ?
[patient] i'm doing well . so , i've been walking like you told me to and , um , exercising and doing yoga , and that's actually helped with my arthritis a lot , just the- the constant movement . so , i have n't had any joint pain recently .
[doctor] okay . good . so , no- no issues with any stiffness or pain or flare ups over the last year ?
[patient] no .
[doctor] okay . and i know that we have you on the me

In [ ]:
# --------------------------------------------------
# CASE 2 - FILL GOLD ANNOTATION
# D2N089-virtassist
# --------------------------------------------------

gold_annotation_case2["gold_output"] = {

    "patient": {
        "age": 52,
        "sex": "female"
    },

    "encounter": {
        "visit_reason": "annual exam",
        "chief_complaint": "annual exam"
    },

    "medical_history": [
        {
            "condition": "rheumatoid arthritis",
            "status": "current"
        },
        {
            "condition": "atrial fibrillation",
            "status": "current"
        },
        {
            "condition": "reflux",
            "status": "current"
        }
    ],

    "symptoms": [
        {
            "name": "joint pain",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": "over the last year"
        },
        {
            "name": "joint stiffness",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": "over the last year"
        },
        {
            "name": "palpitations",
            "status": "present",
            "body_site": None,
            "severity": None,
            "duration": "last episode about one week ago"
        },
        {
            "name": "nasal congestion",
            "status": "present",
            "body_site": "nose",
            "severity": None,
            "duration": None
        },
        {
            "name": "chest pain",
            "status": "denied",
            "body_site": "chest",
            "severity": None,
            "duration": None
        },
        {
            "name": "shortness of breath",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        },
        {
            "name": "nausea",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        },
        {
            "name": "vomiting",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        }
    ],

    "medications": [
        {
            "name": "methotrexate",
            "dosage": "2.5 mg",
            "frequency": "once weekly",
            "action": "continue and refill"
        },
        {
            "name": "Protonix",
            "dosage": "40 mg",
            "frequency": "once daily",
            "action": "continue"
        }
    ],

    "physical_exam": [
        {
            "system": "cardiovascular",
            "finding": "regular rate and rhythm with slight 2/6 systolic ejection murmur",
            "body_site": "heart"
        },
        {
            "system": "respiratory",
            "finding": "lungs clear bilaterally",
            "body_site": "lungs"
        },
        {
            "system": "musculoskeletal",
            "finding": "edema and erythema",
            "body_site": "right elbow"
        },
        {
            "system": "musculoskeletal",
            "finding": "pain to palpation",
            "body_site": "right elbow"
        },
        {
            "system": "musculoskeletal",
            "finding": "no edema",
            "body_site": "bilateral lower extremities"
        }
    ],

    "diagnostic_tests": [
        {
            "test_name": "cardiac event monitor",
            "status": "reviewed",
            "result": "intermittent atrial fibrillation with conversion pause"
        },
        {
            "test_name": "autoimmune panel",
            "status": "reviewed",
            "result": "normal; rheumatoid arthritis well controlled"
        }
    ],

    "assessment": [
        {
            "condition": "rheumatoid arthritis",
            "certainty": "confirmed"
        },
        {
            "condition": "atrial fibrillation",
            "certainty": "confirmed"
        },
        {
            "condition": "reflux",
            "certainty": "confirmed"
        }
    ],

    "plan": {

        "medication_changes": [
            "continue and refill methotrexate 2.5 mg once weekly",
            "continue Protonix 40 mg once daily"
        ],

        "tests_ordered": [],

        "procedures": [],

        "referrals": [
            "cardiology referral for cardiac ablation"
        ],

        "patient_instructions": [
            "continue dietary modifications for reflux",
            "avoid dietary triggers such as coffee and spicy foods",
            "contact clinician if reflux symptoms or other issues recur"
        ]
    },

    "follow_up": {
        "value": None,
        "unit": None,
        "condition": None
    }
}

gold_annotation_case2["review_status"] = "manually_annotated"


print("=" * 80)
print("CASE 2 GOLD ANNOTATION")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case2["gold_output"],
        indent=2
    )
)

CASE 2 GOLD ANNOTATION
{
  "patient": {
    "age": 52,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    {
      "condition": "rheumatoid arthritis",
      "status": "current"
    },
    {
      "condition": "atrial fibrillation",
      "status": "current"
    },
    {
      "condition": "reflux",
      "status": "current"
    }
  ],
  "symptoms": [
    {
      "name": "joint pain",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": "over the last year"
    },
    {
      "name": "joint stiffness",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": "over the last year"
    },
    {
      "name": "palpitations",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "last episode about one week ago"
    },
    {
      "name": "nasal congestion",
      "status": "present",

In [ ]:
# --------------------------------------------------
# CASE 2 - VALIDATE GOLD AGAINST SCHEMA
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator = Draft202012Validator(clinical_schema)

gold_errors_case2 = sorted(
    validator.iter_errors(
        gold_annotation_case2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 2 GOLD SCHEMA VALIDATION")
print("=" * 80)

if not gold_errors_case2:
    print("✅ Gold annotation is valid against clinical_schema")
else:
    print(
        f"❌ Gold annotation has "
        f"{len(gold_errors_case2)} schema errors\n"
    )

    for i, error in enumerate(
        gold_errors_case2,
        start=1
    ):
        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. {path or '<root>'}: "
            f"{error.message}"
        )

CASE 2 GOLD SCHEMA VALIDATION
✅ Gold annotation is valid against clinical_schema


In [ ]:
# --------------------------------------------------
# CASE 2 - BUILD BASELINE PROMPT
# --------------------------------------------------

baseline_prompt_case2 = f"""
You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{{
  "patient": {{
    "age": null,
    "sex": null
  }},
  "encounter": {{
    "visit_reason": null,
    "chief_complaint": null
  }},
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {{
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  }},
  "follow_up": {{
    "value": null,
    "unit": null,
    "condition": null
  }}
}}

Extract information only from the transcript below.

TRANSCRIPT:
{current_case["transcript"]}
"""

print("=" * 80)
print("CASE 2 BASELINE PROMPT CREATED")
print("=" * 80)

print(baseline_prompt_case2[:2000])

CASE 2 BASELINE PROMPT CREATED

You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year-old female with a past medical histo

In [ ]:
# --------------------------------------------------
# CASE 2 - RUN BASELINE GEMINI ONCE AND FREEZE OUTPUT
# --------------------------------------------------

# Create storage dictionary if it does not already exist
if "baseline_outputs" not in globals():
    baseline_outputs = {}

baseline_response_case2 = gemini_client.interactions.create(
    model="gemini-3.5-flash",
    input=baseline_prompt_case2
)

# Save the exact raw output
baseline_raw_case2 = baseline_response_case2.output_text

# Freeze it under this case ID
baseline_outputs[current_case["case_id"]] = baseline_raw_case2

print("=" * 80)
print("CASE 2 BASELINE OUTPUT - FROZEN")
print("=" * 80)

print(baseline_raw_case2)

CASE 2 BASELINE OUTPUT - FROZEN
{
  "patient": {
    "age": 52,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    "Rheumatoid arthritis",
    "Atrial fibrillation",
    "Reflux",
    "Allergies"
  ],
  "symptoms": [
    "Palpitations",
    "Nasal congestion",
    "Right elbow pain"
  ],
  "medications": [
    "Methotrexate 2.5 mg once weekly",
    "Protonix 40 mg daily"
  ],
  "physical_exam": [
    "Heart: Regular rate and rhythm",
    "Heart: Slight 2/6 systolic ejection murmur",
    "Lungs: Clear",
    "Right elbow: Edema, erythema, and pain to palpation",
    "No lower extremity edema"
  ],
  "diagnostic_tests": [
    "Event monitor (results: shows patient is in and out of atrial fibrillation with a conversion pause)",
    "Autoimmune panel (results: normal / well controlled)"
  ],
  "assessment": [
    "Rheumatoid arthritis (stable and well-controlled on Methotrexate)",
    "Atrial fibr

In [ ]:
# --------------------------------------------------
# CASE 2 - PARSE + VALIDATE BASELINE OUTPUT
# --------------------------------------------------

import json
import re
from jsonschema import Draft202012Validator


# 1. Start from the exact frozen Case 2 response
clean_case2 = baseline_raw_case2.strip()


# 2. Remove markdown code fences if Gemini added them
clean_case2 = re.sub(
    r"^```(?:json)?\s*",
    "",
    clean_case2,
    flags=re.IGNORECASE
)

clean_case2 = re.sub(
    r"\s*```$",
    "",
    clean_case2
)


# 3. Parse JSON
try:
    baseline_output_case2 = json.loads(clean_case2)
    json_valid_case2 = 1
    print("✅ Case 2 JSON parsed successfully")

except json.JSONDecodeError as e:
    baseline_output_case2 = None
    json_valid_case2 = 0

    print("❌ Case 2 JSON parsing failed")
    print(e)


# 4. Validate against clinical_schema
schema_errors_case2 = []

if baseline_output_case2 is not None:

    validator = Draft202012Validator(
        clinical_schema
    )

    schema_errors_case2 = sorted(
        validator.iter_errors(
            baseline_output_case2
        ),
        key=lambda e: list(
            e.absolute_path
        )
    )


# 5. Count error types
required_errors_case2 = sum(
    1
    for e in schema_errors_case2
    if e.validator == "required"
)

additional_errors_case2 = sum(
    1
    for e in schema_errors_case2
    if e.validator == "additionalProperties"
)

type_errors_case2 = sum(
    1
    for e in schema_errors_case2
    if e.validator == "type"
)


schema_valid_case2 = int(
    json_valid_case2 == 1
    and len(schema_errors_case2) == 0
)


# 6. Print summary
print("\n" + "=" * 80)
print("CASE 2 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", json_valid_case2)
print("Schema valid:", schema_valid_case2)
print(
    "Total schema errors:",
    len(schema_errors_case2)
)
print(
    "Required-field errors:",
    required_errors_case2
)
print(
    "Additional-property errors:",
    additional_errors_case2
)
print(
    "Type errors:",
    type_errors_case2
)


# 7. Show detailed errors
if schema_errors_case2:

    print("\n" + "=" * 80)
    print("SCHEMA ERROR DETAILS")
    print("=" * 80)

    for i, error in enumerate(
        schema_errors_case2,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

✅ Case 2 JSON parsed successfully

CASE 2 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 0
Total schema errors: 19
Required-field errors: 0
Additional-property errors: 0
Type errors: 19

SCHEMA ERROR DETAILS
1. assessment.0: 'Rheumatoid arthritis (stable and well-controlled on Methotrexate)' is not of type 'object'
2. assessment.1: 'Atrial fibrillation (paroxysmal, in and out of a-fib with conversion pause)' is not of type 'object'
3. assessment.2: 'Reflux (well-controlled, no flare-ups in 5 months)' is not of type 'object'
4. diagnostic_tests.0: 'Event monitor (results: shows patient is in and out of atrial fibrillation with a conversion pause)' is not of type 'object'
5. diagnostic_tests.1: 'Autoimmune panel (results: normal / well controlled)' is not of type 'object'
6. medical_history.0: 'Rheumatoid arthritis' is not of type 'object'
7. medical_history.1: 'Atrial fibrillation' is not of type 'object'
8. medical_history.2: 'Reflux' is not of type 'object'
9. medical_history.3: 'A

In [ ]:
# --------------------------------------------------
# CASE 2 - SEMANTIC REVIEW
# D2N089-virtassist
# --------------------------------------------------

import pandas as pd

semantic_review_case2 = [

    # -------------------------
    # PATIENT
    # -------------------------
    {
        "field": "patient.age",
        "gold_value": 52,
        "model_value": 52,
        "classification": "correct_extraction",
        "notes": "Correctly extracted age"
    },
    {
        "field": "patient.sex",
        "gold_value": "female",
        "model_value": "female",
        "classification": "correct_extraction",
        "notes": "Correctly extracted sex"
    },

    # -------------------------
    # ENCOUNTER
    # -------------------------
    {
        "field": "encounter.visit_reason",
        "gold_value": "annual exam",
        "model_value": "Annual exam",
        "classification": "correct_extraction",
        "notes": "Correct encounter reason"
    },
    {
        "field": "encounter.chief_complaint",
        "gold_value": "annual exam",
        "model_value": "Palpitations and nasal congestion",
        "classification": "mapping_error",
        "notes": (
            "Palpitations and nasal congestion are supported symptoms, "
            "but they were incorrectly used as the chief complaint"
        )
    },

    # -------------------------
    # MEDICAL HISTORY
    # -------------------------
    {
        "field": "medical_history.rheumatoid_arthritis",
        "gold_value": "current",
        "model_value": "Rheumatoid arthritis",
        "classification": "correct_extraction",
        "notes": "Condition correctly identified"
    },
    {
        "field": "medical_history.atrial_fibrillation",
        "gold_value": "current",
        "model_value": "Atrial fibrillation",
        "classification": "correct_extraction",
        "notes": "Condition correctly identified"
    },
    {
        "field": "medical_history.reflux",
        "gold_value": "current",
        "model_value": "Reflux",
        "classification": "correct_extraction",
        "notes": "Condition correctly identified"
    },
    {
        "field": "medical_history.allergies",
        "gold_value": None,
        "model_value": "Allergies",
        "classification": "mapping_error",
        "notes": (
            "Allergy-related nasal congestion is mentioned, but allergies "
            "were not part of the explicitly stated past medical history"
        )
    },

    # -------------------------
    # SYMPTOMS
    # -------------------------
    {
        "field": "symptoms.joint_pain",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit denial of recent joint pain was omitted"
    },
    {
        "field": "symptoms.joint_stiffness",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit denial of joint stiffness was omitted"
    },
    {
        "field": "symptoms.palpitations",
        "gold_value": "present",
        "model_value": "Palpitations",
        "classification": "correct_extraction",
        "notes": "Correctly extracted palpitations"
    },
    {
        "field": "symptoms.nasal_congestion",
        "gold_value": "present",
        "model_value": "Nasal congestion",
        "classification": "correct_extraction",
        "notes": "Correctly extracted nasal congestion"
    },
    {
        "field": "symptoms.chest_pain",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit chest-pain denial omitted"
    },
    {
        "field": "symptoms.shortness_of_breath",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit shortness-of-breath denial omitted"
    },
    {
        "field": "symptoms.nausea",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit nausea denial omitted"
    },
    {
        "field": "symptoms.vomiting",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit vomiting denial omitted"
    },
    {
        "field": "symptoms.right_elbow_pain",
        "gold_value": None,
        "model_value": "Right elbow pain",
        "classification": "mapping_error",
        "notes": (
            "Pain is supported by the examination, but the model also "
            "placed the examination finding in the symptoms section"
        )
    },

    # -------------------------
    # MEDICATIONS
    # -------------------------
    {
        "field": "medications.methotrexate",
        "gold_value": "continue and refill 2.5 mg once weekly",
        "model_value": "Methotrexate 2.5 mg once weekly",
        "classification": "partial_extraction",
        "notes": (
            "Medication, dose and frequency are correct, "
            "but action was not represented in the medication item"
        )
    },
    {
        "field": "medications.protonix",
        "gold_value": "continue 40 mg once daily",
        "model_value": "Protonix 40 mg daily",
        "classification": "partial_extraction",
        "notes": (
            "Medication, dose and frequency are correct, "
            "but continue action was not represented"
        )
    },

    # -------------------------
    # PHYSICAL EXAM
    # -------------------------
    {
        "field": "physical_exam.cardiovascular",
        "gold_value": (
            "regular rate and rhythm with slight "
            "2/6 systolic ejection murmur"
        ),
        "model_value": (
            "Heart: Regular rate and rhythm, "
            "2/6 systolic ejection murmur"
        ),
        "classification": "correct_extraction",
        "notes": "Cardiovascular examination correctly extracted"
    },
    {
        "field": "physical_exam.respiratory",
        "gold_value": "lungs clear bilaterally",
        "model_value": "Lungs: Clear",
        "classification": "correct_extraction",
        "notes": "Respiratory finding correctly extracted"
    },
    {
        "field": "physical_exam.right_elbow_edema_erythema",
        "gold_value": "edema and erythema",
        "model_value": (
            "Right elbow: Edema, erythema, "
            "and pain to palpation"
        ),
        "classification": "correct_extraction",
        "notes": "Edema and erythema correctly extracted"
    },
    {
        "field": "physical_exam.right_elbow_pain",
        "gold_value": "pain to palpation",
        "model_value": (
            "Right elbow: Edema, erythema, "
            "and pain to palpation"
        ),
        "classification": "correct_extraction",
        "notes": "Pain to palpation correctly extracted"
    },
    {
        "field": "physical_exam.lower_extremity_edema",
        "gold_value": "no edema",
        "model_value": "Lower extremities: No edema",
        "classification": "correct_extraction",
        "notes": "Correctly preserved negative examination finding"
    },

    # -------------------------
    # DIAGNOSTIC TESTS
    # -------------------------
    {
        "field": "diagnostic_tests.event_monitor",
        "gold_value": (
            "intermittent atrial fibrillation "
            "with conversion pause"
        ),
        "model_value": (
            "Event monitor: In and out of atrial fibrillation "
            "with a conversion pause"
        ),
        "classification": "correct_extraction",
        "notes": "Event monitor result correctly extracted"
    },
    {
        "field": "diagnostic_tests.autoimmune_panel",
        "gold_value": (
            "normal; rheumatoid arthritis well controlled"
        ),
        "model_value": "Autoimmune panel: Looks good / stable",
        "classification": "correct_extraction",
        "notes": "Same clinical meaning despite wording difference"
    },

    # -------------------------
    # ASSESSMENT
    # -------------------------
    {
        "field": "assessment.rheumatoid_arthritis",
        "gold_value": "confirmed",
        "model_value": (
            "Rheumatoid arthritis: Stable and "
            "well controlled on Methotrexate"
        ),
        "classification": "partial_extraction",
        "notes": (
            "Condition correctly identified but required "
            "certainty field was not explicitly represented"
        )
    },
    {
        "field": "assessment.atrial_fibrillation",
        "gold_value": "confirmed",
        "model_value": (
            "Atrial fibrillation: Paroxysmal "
            "with conversion pause"
        ),
        "classification": "partial_extraction",
        "notes": (
            "Condition correctly identified but certainty "
            "was not explicitly represented"
        )
    },
    {
        "field": "assessment.reflux",
        "gold_value": "confirmed",
        "model_value": (
            "Reflux: Controlled, no flare-ups "
            "in over 5 months"
        ),
        "classification": "partial_extraction",
        "notes": (
            "Condition correctly identified but certainty "
            "was not explicitly represented"
        )
    },

    # -------------------------
    # PLAN
    # -------------------------
    {
        "field": "plan.methotrexate",
        "gold_value": (
            "continue and refill methotrexate "
            "2.5 mg once weekly"
        ),
        "model_value": (
            "Refill Methotrexate 2.5 mg once weekly"
        ),
        "classification": "partial_extraction",
        "notes": (
            "Refill, dose and frequency are correct; "
            "continue action was not explicitly stated"
        )
    },
    {
        "field": "plan.protonix",
        "gold_value": "continue Protonix 40 mg once daily",
        "model_value": "Continue Protonix 40 mg daily",
        "classification": "correct_extraction",
        "notes": "Correct medication plan"
    },
    {
        "field": "plan.cardiology_referral",
        "gold_value": (
            "cardiology referral for cardiac ablation"
        ),
        "model_value": (
            "Cardiology referral for cardiac ablation"
        ),
        "classification": "correct_extraction",
        "notes": "Referral correctly extracted"
    },
    {
        "field": "plan.reflux_instructions",
        "gold_value": (
            "continue dietary modifications and "
            "avoid dietary triggers"
        ),
        "model_value": (
            "Continue dietary modifications for reflux, "
            "avoiding soda, coffee, and spicy foods"
        ),
        "classification": "correct_extraction",
        "notes": (
            "Instruction is supported by the transcript; "
            "minor wording expansion does not change meaning"
        )
    },
    {
        "field": "plan.exercise_instruction",
        "gold_value": None,
        "model_value": (
            "Continue physical activities including "
            "walking, exercising, and yoga"
        ),
        "classification": "unsupported_inference",
        "notes": (
            "Patient described these activities and their benefit, "
            "but clinician did not explicitly prescribe continuing "
            "them in the final plan"
        )
    },

    # -------------------------
    # FOLLOW UP
    # -------------------------
    {
        "field": "follow_up",
        "gold_value": None,
        "model_value": None,
        "classification": "correct_extraction",
        "notes": "No follow-up interval was stated"
    }
]


semantic_review_df_case2 = pd.DataFrame(
    semantic_review_case2
)

print("=" * 80)
print("CASE 2 SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df_case2)

CASE 2 SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,52,52,correct_extraction,Correctly extracted age
1,patient.sex,female,female,correct_extraction,Correctly extracted sex
2,encounter.visit_reason,annual exam,Annual exam,correct_extraction,Correct encounter reason
3,encounter.chief_complaint,annual exam,Palpitations and nasal congestion,mapping_error,Palpitations and nasal congestion are supporte...
4,medical_history.rheumatoid_arthritis,current,Rheumatoid arthritis,correct_extraction,Condition correctly identified
5,medical_history.atrial_fibrillation,current,Atrial fibrillation,correct_extraction,Condition correctly identified
6,medical_history.reflux,current,Reflux,correct_extraction,Condition correctly identified
7,medical_history.allergies,None,Allergies,mapping_error,"Allergy-related nasal congestion is mentioned,..."
8,symptoms.joint_pain,denied,None,omission,Explicit denial of recent joint pain was omitted
9,symptoms.joint_stiffness,denied,None,omission,Explicit denial of joint stiffness was omitted


In [ ]:
# --------------------------------------------------
# CASE 2 - SEMANTIC COUNTS
# --------------------------------------------------

semantic_counts_case2 = (
    semantic_review_df_case2["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 2 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts_case2.items():
    print(f"{category}: {count}")

print("-" * 60)
print(
    "Total reviewed items:",
    len(semantic_review_df_case2)
)

CASE 2 SEMANTIC COUNTS
correct_extraction: 19
omission: 6
partial_extraction: 6
mapping_error: 3
unsupported_inference: 1
------------------------------------------------------------
Total reviewed items: 35


In [ ]:
# --------------------------------------------------
# CASE 2 - FINAL FROZEN BASELINE RESULT
# --------------------------------------------------

case2_final_result = {
    "case_id": "D2N089-virtassist",
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    # Structural metrics
    "json_valid": 1,
    "schema_valid": 0,
    "schema_error_count": 19,
    "required_field_errors": 0,
    "additional_property_errors": 0,
    "type_errors": 19,

    # Semantic metrics
    "semantic_items_reviewed": 35,
    "correct_extractions": 19,
    "partial_extractions": 6,
    "omissions": 6,
    "mapping_errors": 3,
    "status_errors": 0,
    "unsupported_inferences": 1
}

print("=" * 70)
print("CASE 2 FINAL BASELINE RESULT")
print("=" * 70)

print(
    json.dumps(
        case2_final_result,
        indent=2
    )
)

CASE 2 FINAL BASELINE RESULT
{
  "case_id": "D2N089-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 19,
  "required_field_errors": 0,
  "additional_property_errors": 0,
  "type_errors": 19,
  "semantic_items_reviewed": 35,
  "correct_extractions": 19,
  "partial_extractions": 6,
  "omissions": 6,
  "mapping_errors": 3,
  "status_errors": 0,
  "unsupported_inferences": 1
}


In [ ]:
# --------------------------------------------------
# SELECT CASE 3
# --------------------------------------------------

case_index = 2
current_case = research_cases[case_index]

print("=" * 80)
print("CASE 3")
print("=" * 80)

print("CASE ID:", current_case["case_id"])

print("\nTRANSCRIPT:")
print("-" * 80)
print(current_case["transcript"])

print("\nREFERENCE NOTE:")
print("-" * 80)
print(current_case["reference_note"])

CASE 3
CASE ID: D2N090-virtassist

TRANSCRIPT:
--------------------------------------------------------------------------------
[doctor] hi , albert . how are you ?
[patient] hey , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] so , albert is a 62-year-old male , with a past medical history significant for depression , type 2 diabetes , and kidney transplant , who is here today for emergency room follow-up .
[patient] mm-hmm .
[doctor] so , i got a notification that you were in the emergency room , but , but what were you there for ?
[patient] well , i , uh , i was n't really , uh , staying on top of my , uh , blood sugar readings , and i felt kinda woozy over the weekend . and i was little concerned , and my wife wanted to take me in and just have me checked out .
[doctor] okay . and , and was it , in fact , high ?
[patient] yeah , it was .
[doctor] okay . did you 

In [ ]:
# --------------------------------------------------
# CASE 3 - CREATE + FILL GOLD ANNOTATION
# D2N090-virtassist
# --------------------------------------------------

gold_annotation_case3 = {
    "case_id": current_case["case_id"],

    "gold_output": {

        "patient": {
            "age": 62,
            "sex": "male"
        },

        "encounter": {
            "visit_reason": "emergency room follow-up",
            "chief_complaint": "emergency room follow-up"
        },

        "medical_history": [
            {
                "condition": "depression",
                "status": "current"
            },
            {
                "condition": "type 2 diabetes",
                "status": "current"
            },
            {
                "condition": "kidney transplant",
                "status": "current"
            }
        ],

        "symptoms": [
            {
                "name": "wooziness",
                "status": "resolved",
                "body_site": None,
                "severity": None,
                "duration": "over the weekend"
            },
            {
                "name": "chest pain",
                "status": "denied",
                "body_site": "chest",
                "severity": None,
                "duration": None
            },
            {
                "name": "shortness of breath",
                "status": "denied",
                "body_site": None,
                "severity": None,
                "duration": None
            },
            {
                "name": "lightheadedness",
                "status": "denied",
                "body_site": None,
                "severity": None,
                "duration": None
            },
            {
                "name": "dizziness",
                "status": "denied",
                "body_site": None,
                "severity": None,
                "duration": None
            }
        ],

        "medications": [
            {
                "name": "Lantus",
                "dosage": "20 units",
                "frequency": "at night",
                "action": "increase"
            },
            {
                "name": "immunosuppression medications",
                "dosage": None,
                "frequency": None,
                "action": "continue under transplant specialist management"
            }
        ],

        "physical_exam": [
            {
                "system": "constitutional",
                "finding": "no apparent distress",
                "body_site": None
            },
            {
                "system": "neck",
                "finding": "no carotid bruits",
                "body_site": "neck"
            },
            {
                "system": "cardiovascular",
                "finding": "slight 2/6 systolic ejection murmur",
                "body_site": "heart"
            },
            {
                "system": "respiratory",
                "finding": "lungs clear bilaterally",
                "body_site": "lungs"
            },
            {
                "system": "musculoskeletal",
                "finding": "1+ edema",
                "body_site": "bilateral lower extremities"
            }
        ],

        "diagnostic_tests": [
            {
                "test_name": "blood glucose",
                "status": "reviewed",
                "result": "162"
            },
            {
                "test_name": "hemoglobin A1c",
                "status": "reviewed",
                "result": "8"
            },
            {
                "test_name": "hemoglobin A1c",
                "status": "ordered",
                "result": None
            }
        ],

        "assessment": [
            {
                "condition": "hyperglycemia",
                "certainty": "confirmed"
            },
            {
                "condition": "depression",
                "certainty": "confirmed"
            },
            {
                "condition": "status post kidney transplant",
                "certainty": "confirmed"
            }
        ],

        "plan": {

            "medication_changes": [
                "increase Lantus to 20 units at night"
            ],

            "tests_ordered": [
                "repeat hemoglobin A1c in a couple of months"
            ],

            "procedures": [],

            "referrals": [
                "follow up with Dr. Reyes for management of immunosuppression medications"
            ],

            "patient_instructions": [
                "continue monitoring blood glucose at home",
                "report blood glucose readings to clinician",
                "continue current depression management strategies including meditation",
                "contact clinician if additional help is needed"
            ]
        },

        "follow_up": {
            "value": None,
            "unit": None,
            "condition": None
        }
    },

    "evidence": {},

    "review_status": "manually_annotated"
}


print("=" * 80)
print("CASE 3 GOLD ANNOTATION")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case3["gold_output"],
        indent=2
    )
)

CASE 3 GOLD ANNOTATION
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "emergency room follow-up",
    "chief_complaint": "emergency room follow-up"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current"
    },
    {
      "condition": "kidney transplant",
      "status": "current"
    }
  ],
  "symptoms": [
    {
      "name": "wooziness",
      "status": "resolved",
      "body_site": null,
      "severity": null,
      "duration": "over the weekend"
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": null,
      "duration": null
    },
    {
      "name": "shortness of breath",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": null
    },
    {
      "name": "lightheadedness",
      "status": "denied",
      "body_site": nul

In [ ]:
# --------------------------------------------------
# CASE 3 - VALIDATE GOLD AGAINST SCHEMA
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator = Draft202012Validator(clinical_schema)

gold_errors_case3 = sorted(
    validator.iter_errors(
        gold_annotation_case3["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 3 GOLD SCHEMA VALIDATION")
print("=" * 80)

if not gold_errors_case3:
    print("✅ Gold annotation is valid against clinical_schema")
else:
    print(
        f"❌ Gold annotation has "
        f"{len(gold_errors_case3)} schema errors\n"
    )

    for i, error in enumerate(
        gold_errors_case3,
        start=1
    ):
        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. {path or '<root>'}: "
            f"{error.message}"
        )

CASE 3 GOLD SCHEMA VALIDATION
✅ Gold annotation is valid against clinical_schema


In [ ]:
# --------------------------------------------------
# CASE 3 - BUILD BASELINE PROMPT
# --------------------------------------------------

baseline_prompt_case3 = f"""
You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{{
  "patient": {{
    "age": null,
    "sex": null
  }},
  "encounter": {{
    "visit_reason": null,
    "chief_complaint": null
  }},
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {{
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  }},
  "follow_up": {{
    "value": null,
    "unit": null,
    "condition": null
  }}
}}

Extract information only from the transcript below.

TRANSCRIPT:
{current_case["transcript"]}
"""

print("=" * 80)
print("CASE 3 BASELINE PROMPT CREATED")
print("=" * 80)

print(baseline_prompt_case3[:2000])

CASE 3 BASELINE PROMPT CREATED

You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
[doctor] hi , albert . how are you ?
[patient] hey , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] so , albert is a 62-year-old male , with a past medical histor

In [ ]:
# --------------------------------------------------
# CASE 3 - RUN BASELINE GEMINI ONCE AND FREEZE OUTPUT
# --------------------------------------------------

if "baseline_outputs" not in globals():
    baseline_outputs = {}

baseline_response_case3 = gemini_client.interactions.create(
    model="gemini-3.5-flash",
    input=baseline_prompt_case3
)

# Save the exact raw response
baseline_raw_case3 = baseline_response_case3.output_text

# Freeze it using the case ID
baseline_outputs[current_case["case_id"]] = baseline_raw_case3

print("=" * 80)
print("CASE 3 BASELINE OUTPUT - FROZEN")
print("=" * 80)

print(baseline_raw_case3)

CASE 3 BASELINE OUTPUT - FROZEN
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "emergency room follow-up",
    "chief_complaint": "emergency room follow-up"
  },
  "medical_history": [
    "depression",
    "type 2 diabetes",
    "kidney transplant"
  ],
  "symptoms": [
    "woozy",
    "no chest pain",
    "no shortness of breath",
    "no lightheadedness",
    "no dizziness"
  ],
  "medications": [
    "immunotherapy meds",
    "Lantus"
  ],
  "physical_exam": [
    "no distress",
    "no carotid bruits",
    "slight 2/6 systolic ejection murmur",
    "lungs sound nice and clear",
    "1+ edema in lower extremities",
    "blood pressure and heart rate are right where they should be",
    "pulse ox is great"
  ],
  "diagnostic_tests": [
    "blood sugar: 162",
    "hemoglobin a1c: about 8",
    "kidney function: good"
  ],
  "assessment": [
    "hyperglycemia",
    "depression",
    "kidney transplant"
  ],
  "plan": {
    "medication_chang

In [ ]:
# --------------------------------------------------
# CASE 3 - PARSE + VALIDATE BASELINE OUTPUT
# --------------------------------------------------

import json
import re
from jsonschema import Draft202012Validator


# 1. Use the exact frozen Case 3 output
clean_case3 = baseline_raw_case3.strip()


# 2. Remove ```json code fences
clean_case3 = re.sub(
    r"^```(?:json)?\s*",
    "",
    clean_case3,
    flags=re.IGNORECASE
)

clean_case3 = re.sub(
    r"\s*```$",
    "",
    clean_case3
)


# 3. Parse JSON
try:
    baseline_output_case3 = json.loads(clean_case3)

    json_valid_case3 = 1

    print("✅ Case 3 JSON parsed successfully")

except json.JSONDecodeError as e:

    baseline_output_case3 = None
    json_valid_case3 = 0

    print("❌ Case 3 JSON parsing failed")
    print(e)


# 4. Validate against clinical_schema
schema_errors_case3 = []

if baseline_output_case3 is not None:

    validator = Draft202012Validator(
        clinical_schema
    )

    schema_errors_case3 = sorted(
        validator.iter_errors(
            baseline_output_case3
        ),
        key=lambda e: list(e.absolute_path)
    )


# 5. Count error categories
required_errors_case3 = sum(
    1
    for e in schema_errors_case3
    if e.validator == "required"
)

additional_errors_case3 = sum(
    1
    for e in schema_errors_case3
    if e.validator == "additionalProperties"
)

type_errors_case3 = sum(
    1
    for e in schema_errors_case3
    if e.validator == "type"
)


schema_valid_case3 = int(
    json_valid_case3 == 1
    and len(schema_errors_case3) == 0
)


# 6. Print summary
print("\n" + "=" * 80)
print("CASE 3 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", json_valid_case3)
print("Schema valid:", schema_valid_case3)
print(
    "Total schema errors:",
    len(schema_errors_case3)
)
print(
    "Required-field errors:",
    required_errors_case3
)
print(
    "Additional-property errors:",
    additional_errors_case3
)
print(
    "Type errors:",
    type_errors_case3
)


# 7. Print detailed errors
if schema_errors_case3:

    print("\n" + "=" * 80)
    print("SCHEMA ERROR DETAILS")
    print("=" * 80)

    for i, error in enumerate(
        schema_errors_case3,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

✅ Case 3 JSON parsed successfully

CASE 3 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 0
Total schema errors: 23
Required-field errors: 0
Additional-property errors: 0
Type errors: 23

SCHEMA ERROR DETAILS
1. assessment.0: 'hyperglycemia' is not of type 'object'
2. assessment.1: 'depression' is not of type 'object'
3. assessment.2: 'kidney transplant' is not of type 'object'
4. diagnostic_tests.0: 'blood sugar: 162' is not of type 'object'
5. diagnostic_tests.1: 'hemoglobin a1c: about 8' is not of type 'object'
6. diagnostic_tests.2: 'kidney function: good' is not of type 'object'
7. medical_history.0: 'depression' is not of type 'object'
8. medical_history.1: 'type 2 diabetes' is not of type 'object'
9. medical_history.2: 'kidney transplant' is not of type 'object'
10. medications.0: 'immunotherapy meds' is not of type 'object'
11. medications.1: 'Lantus' is not of type 'object'
12. physical_exam.0: 'no distress' is not of type 'object'
13. physical_exam.1: 'no carotid bruits' is

In [ ]:
# --------------------------------------------------
# CASE 3 - SEMANTIC REVIEW
# D2N090-virtassist
# --------------------------------------------------

import pandas as pd

semantic_review_case3 = [

    # -------------------------
    # PATIENT
    # -------------------------
    {
        "field": "patient.age",
        "gold_value": 62,
        "model_value": "62",
        "classification": "correct_extraction",
        "notes": (
            "Age is semantically correct. "
            "String-versus-integer is already counted as a schema type error."
        )
    },
    {
        "field": "patient.sex",
        "gold_value": "male",
        "model_value": "male",
        "classification": "correct_extraction",
        "notes": "Correctly extracted sex"
    },

    # -------------------------
    # ENCOUNTER
    # -------------------------
    {
        "field": "encounter.visit_reason",
        "gold_value": "emergency room follow-up",
        "model_value": "Emergency room follow-up",
        "classification": "correct_extraction",
        "notes": "Correct encounter reason"
    },
    {
        "field": "encounter.chief_complaint",
        "gold_value": "emergency room follow-up",
        "model_value": "Elevated blood sugar and feeling woozy",
        "classification": "mapping_error",
        "notes": (
            "Elevated blood sugar and wooziness are supported by the transcript, "
            "but the stated encounter reason/chief complaint is ER follow-up."
        )
    },

    # -------------------------
    # MEDICAL HISTORY
    # -------------------------
    {
        "field": "medical_history.depression",
        "gold_value": "current",
        "model_value": "Depression",
        "classification": "correct_extraction",
        "notes": "Correctly identified medical history"
    },
    {
        "field": "medical_history.type_2_diabetes",
        "gold_value": "current",
        "model_value": "Type 2 diabetes",
        "classification": "correct_extraction",
        "notes": "Correctly identified medical history"
    },
    {
        "field": "medical_history.kidney_transplant",
        "gold_value": "current",
        "model_value": "Kidney transplant",
        "classification": "correct_extraction",
        "notes": "Correctly identified transplant history"
    },
    {
        "field": "medical_history.systolic_ejection_murmur",
        "gold_value": None,
        "model_value": "Systolic ejection murmur",
        "classification": "mapping_error",
        "notes": (
            "The murmur is supported, but it is a physical examination finding "
            "and was incorrectly also placed in medical history."
        )
    },

    # -------------------------
    # SYMPTOMS
    # -------------------------
    {
        "field": "symptoms.wooziness",
        "gold_value": "resolved",
        "model_value": "Wooziness (resolved)",
        "classification": "correct_extraction",
        "notes": "Past wooziness and its resolution were correctly captured"
    },
    {
        "field": "symptoms.chest_pain",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit chest-pain denial was omitted"
    },
    {
        "field": "symptoms.shortness_of_breath",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit shortness-of-breath denial was omitted"
    },
    {
        "field": "symptoms.lightheadedness",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit current lightheadedness denial was omitted"
    },
    {
        "field": "symptoms.dizziness",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit current dizziness denial was omitted"
    },

    # -------------------------
    # MEDICATIONS
    # -------------------------
    {
        "field": "medications.lantus",
        "gold_value": "increase to 20 units at night",
        "model_value": "Lantus",
        "classification": "partial_extraction",
        "notes": (
            "Medication was identified, but dose, frequency and action "
            "were missing from the medication item."
        )
    },
    {
        "field": "medications.immunosuppression",
        "gold_value": (
            "continue under transplant specialist management"
        ),
        "model_value": "Immunosuppressive medications",
        "classification": "partial_extraction",
        "notes": (
            "Medication category was correctly identified, "
            "but management/action was not represented here."
        )
    },

    # -------------------------
    # PHYSICAL EXAM
    # -------------------------
    {
        "field": "physical_exam.general",
        "gold_value": "no apparent distress",
        "model_value": "General: No acute distress",
        "classification": "correct_extraction",
        "notes": "Equivalent clinical meaning"
    },
    {
        "field": "physical_exam.neck",
        "gold_value": "no carotid bruits",
        "model_value": "Neck: No carotid bruits appreciated",
        "classification": "correct_extraction",
        "notes": "Correct negative finding"
    },
    {
        "field": "physical_exam.cardiovascular",
        "gold_value": "slight 2/6 systolic ejection murmur",
        "model_value": "Cardiovascular: 2/6 systolic ejection murmur",
        "classification": "correct_extraction",
        "notes": "Correctly extracted murmur"
    },
    {
        "field": "physical_exam.respiratory",
        "gold_value": "lungs clear bilaterally",
        "model_value": "Lungs: Clear to auscultation",
        "classification": "correct_extraction",
        "notes": "Correct respiratory finding"
    },
    {
        "field": "physical_exam.lower_extremity_edema",
        "gold_value": "1+ edema",
        "model_value": "Extremities: 1+ lower extremity edema",
        "classification": "correct_extraction",
        "notes": "Correctly extracted lower-extremity edema"
    },

    # -------------------------
    # DIAGNOSTIC TESTS
    # -------------------------
    {
        "field": "diagnostic_tests.glucose",
        "gold_value": "162",
        "model_value": "Blood glucose: 162 mg/dL (fasting)",
        "classification": "correct_extraction",
        "notes": "Glucose value correctly extracted"
    },
    {
        "field": "diagnostic_tests.hemoglobin_a1c",
        "gold_value": "8",
        "model_value": "Hemoglobin A1c: 8%",
        "classification": "correct_extraction",
        "notes": "A1c value correctly extracted"
    },

    # -------------------------
    # ASSESSMENT
    # -------------------------
    {
        "field": "assessment.hyperglycemia",
        "gold_value": "confirmed",
        "model_value": "Hyperglycemia / uncontrolled Type 2 diabetes",
        "classification": "correct_extraction",
        "notes": (
            "Hyperglycemia is explicitly identified by the clinician "
            "as the first problem."
        )
    },
    {
        "field": "assessment.uncontrolled_type_2_diabetes",
        "gold_value": None,
        "model_value": "uncontrolled Type 2 diabetes",
        "classification": "unsupported_inference",
        "notes": (
            "Elevated glucose and A1c are supported, but the clinician "
            "did not explicitly label the diabetes as 'uncontrolled'."
        )
    },
    {
        "field": "assessment.depression",
        "gold_value": "confirmed",
        "model_value": (
            "Depression (stable with non-pharmacological management)"
        ),
        "classification": "correct_extraction",
        "notes": (
            "Depression is confirmed and the patient is managing it "
            "with meditation without medication or therapy."
        )
    },
    {
        "field": "assessment.kidney_transplant",
        "gold_value": "confirmed",
        "model_value": "Kidney transplant (stable)",
        "classification": "correct_extraction",
        "notes": (
            "Transplant history and stable kidney function are supported."
        )
    },

    # -------------------------
    # PLAN
    # -------------------------
    {
        "field": "plan.lantus",
        "gold_value": "increase Lantus to 20 units at night",
        "model_value": "Increase Lantus to 20 units at night",
        "classification": "correct_extraction",
        "notes": "Medication change correctly extracted"
    },
    {
        "field": "plan.repeat_a1c",
        "gold_value": (
            "repeat hemoglobin A1c in a couple of months"
        ),
        "model_value": (
            "Repeat Hemoglobin A1c in a couple of months"
        ),
        "classification": "correct_extraction",
        "notes": "Ordered test and timing correctly extracted"
    },
    {
        "field": "plan.transplant_specialist",
        "gold_value": (
            "follow up with Dr. Reyes for management "
            "of immunosuppression medications"
        ),
        "model_value": (
            "Follow-up with Dr. Reyes for immunosuppression "
            "medication management"
        ),
        "classification": "correct_extraction",
        "notes": "Specialist management correctly extracted"
    },
    {
        "field": "plan.monitor_blood_glucose",
        "gold_value": "continue monitoring blood glucose at home",
        "model_value": (
            "Continue monitoring blood sugar readings and report values "
            "for further medication adjustment"
        ),
        "classification": "correct_extraction",
        "notes": "Home glucose-monitoring instruction is explicit"
    },
    {
        "field": "plan.report_glucose_readings",
        "gold_value": "report blood glucose readings to clinician",
        "model_value": (
            "Continue monitoring blood sugar readings and report values "
            "for further medication adjustment"
        ),
        "classification": "correct_extraction",
        "notes": "Reporting instruction correctly captured"
    },
    {
        "field": "plan.depression_management",
        "gold_value": (
            "continue current depression management strategies "
            "including meditation"
        ),
        "model_value": (
            "Continue meditation practices for depression management"
        ),
        "classification": "correct_extraction",
        "notes": (
            "Supported by the discussion of continuing the patient's "
            "current non-pharmacologic strategy."
        )
    },
    {
        "field": "plan.contact_clinician",
        "gold_value": (
            "contact clinician if additional help is needed"
        ),
        "model_value": None,
        "classification": "omission",
        "notes": (
            "Clinician explicitly told the patient to call if he needed anything."
        )
    },
    {
        "field": "plan.diabetic_diet",
        "gold_value": None,
        "model_value": "Follow recommended diabetic diet closely",
        "classification": "unsupported_inference",
        "notes": (
            "Diet adherence was discussed, but this was not explicitly "
            "issued as an instruction in the stated assessment and plan."
        )
    },

    # -------------------------
    # FOLLOW-UP
    # -------------------------
    {
        "field": "follow_up",
        "gold_value": None,
        "model_value": None,
        "classification": "correct_extraction",
        "notes": (
            "No explicit follow-up visit interval was given. "
            "The repeat A1c timing is not treated as a visit follow-up."
        )
    }
]


semantic_review_df_case3 = pd.DataFrame(
    semantic_review_case3
)

print("=" * 80)
print("CASE 3 SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df_case3)

CASE 3 SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,62,62,correct_extraction,Age is semantically correct. String-versus-int...
1,patient.sex,male,male,correct_extraction,Correctly extracted sex
2,encounter.visit_reason,emergency room follow-up,Emergency room follow-up,correct_extraction,Correct encounter reason
3,encounter.chief_complaint,emergency room follow-up,Elevated blood sugar and feeling woozy,mapping_error,Elevated blood sugar and wooziness are support...
4,medical_history.depression,current,Depression,correct_extraction,Correctly identified medical history
5,medical_history.type_2_diabetes,current,Type 2 diabetes,correct_extraction,Correctly identified medical history
6,medical_history.kidney_transplant,current,Kidney transplant,correct_extraction,Correctly identified transplant history
7,medical_history.systolic_ejection_murmur,None,Systolic ejection murmur,mapping_error,"The murmur is supported, but it is a physical ..."
8,symptoms.wooziness,resolved,Wooziness (resolved),correct_extraction,Past wooziness and its resolution were correct...
9,symptoms.chest_pain,denied,None,omission,Explicit chest-pain denial was omitted


In [ ]:
# --------------------------------------------------
# CASE 3 - SEMANTIC COUNTS
# --------------------------------------------------

semantic_counts_case3 = (
    semantic_review_df_case3["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 3 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts_case3.items():
    print(f"{category}: {count}")

print("-" * 60)

print(
    "Total reviewed items:",
    len(semantic_review_df_case3)
)

CASE 3 SEMANTIC COUNTS
correct_extraction: 24
omission: 5
mapping_error: 2
partial_extraction: 2
unsupported_inference: 2
------------------------------------------------------------
Total reviewed items: 35


In [ ]:
# --------------------------------------------------
# CASE 3 - SEMANTIC COUNTS
# --------------------------------------------------

semantic_counts_case3 = (
    semantic_review_df_case3["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 3 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts_case3.items():
    print(f"{category}: {count}")

print("-" * 60)

print(
    "Total reviewed items:",
    len(semantic_review_df_case3)
)

CASE 3 SEMANTIC COUNTS
correct_extraction: 24
omission: 5
mapping_error: 2
partial_extraction: 2
unsupported_inference: 2
------------------------------------------------------------
Total reviewed items: 35


In [ ]:
# --------------------------------------------------
# CASE 3 - FINAL FROZEN BASELINE RESULT
# --------------------------------------------------

case3_final_result = {
    "case_id": "D2N090-virtassist",
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    # Structural metrics
    "json_valid": 1,
    "schema_valid": 0,
    "schema_error_count": 19,
    "required_field_errors": 0,
    "additional_property_errors": 0,
    "type_errors": 19,

    # Semantic metrics
    "semantic_items_reviewed": 35,
    "correct_extractions": 24,
    "partial_extractions": 2,
    "omissions": 5,
    "mapping_errors": 2,
    "status_errors": 0,
    "unsupported_inferences": 2
}

print("=" * 70)
print("CASE 3 FINAL BASELINE RESULT")
print("=" * 70)

print(
    json.dumps(
        case3_final_result,
        indent=2
    )
)

CASE 3 FINAL BASELINE RESULT
{
  "case_id": "D2N090-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 19,
  "required_field_errors": 0,
  "additional_property_errors": 0,
  "type_errors": 19,
  "semantic_items_reviewed": 35,
  "correct_extractions": 24,
  "partial_extractions": 2,
  "omissions": 5,
  "mapping_errors": 2,
  "status_errors": 0,
  "unsupported_inferences": 2
}


In [ ]:
# --------------------------------------------------
# CASE 4 - SELECT AND INSPECT
# --------------------------------------------------

case_index = 3
current_case = research_cases[case_index]

print("=" * 80)
print("CASE 4")
print("=" * 80)

print("CASE ID:", current_case["case_id"])

print("\nTRANSCRIPT:")
print("-" * 80)
print(current_case["transcript"])

print("\nREFERENCE NOTE:")
print("-" * 80)
print(current_case["reference_note"])

CASE 4
CASE ID: D2N091-virtassist

TRANSCRIPT:
--------------------------------------------------------------------------------
[doctor] hi jerry , how are you doing ?
[patient] hi , good to see you .
[doctor] good to see you as well . um , so i know that the nurse told you about dax . i'd like to tell dax about you .
[patient] sure .
[doctor] jerry is a 54 year old male with a past medical history , significant for osteoporosis and multiple sclerosis who presents for an annual exam . so jerry , what's been going on since the last time i saw you ?
[patient] uh , we have been traveling all over the country . it's been kind of a stressful summer . kinda adjusting to everything in the fall and so far it's been good , but ah , lack of sleep , it's been really getting to me .
[doctor] okay . all right . and have you taken anything for the insomnia . have you tried any strategies for it .
[patient] i've tried everything from melatonin to meditation to , uh , t- stretching out every morning w

In [ ]:
# --------------------------------------------------
# CASE 4 - CREATE + FILL GOLD ANNOTATION
# D2N091-virtassist
# --------------------------------------------------

gold_annotation_case4 = {
    "case_id": current_case["case_id"],

    "gold_output": {

        "patient": {
            "age": 54,
            "sex": "male"
        },

        "encounter": {
            "visit_reason": "annual exam",
            "chief_complaint": "annual exam"
        },

        "medical_history": [
            {
                "condition": "osteoporosis",
                "status": "current"
            },
            {
                "condition": "multiple sclerosis",
                "status": "current"
            }
        ],

        "symptoms": [
            {
                "name": "insomnia",
                "status": "present",
                "body_site": None,
                "severity": None,
                "duration": None
            },
            {
                "name": "chest pain",
                "status": "denied",
                "body_site": "chest",
                "severity": None,
                "duration": None
            },
            {
                "name": "shortness of breath",
                "status": "denied",
                "body_site": None,
                "severity": None,
                "duration": None
            }
        ],

        "medications": [
            {
                "name": "Fosamax",
                "dosage": "1 tablet",
                "frequency": "once weekly",
                "action": "continue and refill"
            },
            {
                "name": "multiple sclerosis medications",
                "dosage": None,
                "frequency": None,
                "action": "continue"
            }
        ],

        "physical_exam": [
            {
                "system": "respiratory",
                "finding": "lungs clear",
                "body_site": "lungs"
            },
            {
                "system": "cardiovascular",
                "finding": "heart sounds normal",
                "body_site": "heart"
            },
            {
                "system": "neurological",
                "finding": "strength 4/5",
                "body_site": "right lower extremity"
            },
            {
                "system": "neurological",
                "finding": "strength 3/5",
                "body_site": "left lower extremity"
            },
            {
                "system": "neurological",
                "finding": "reflexes good",
                "body_site": "ity"
            },
            {
                "system": "neurological",
                "finding": "reflexes good",
                "body_site": "lower extremities"
            },
            {
                "system": "musculoskeletal",
                "finding": "arthritic changes",
                "body_site": "right knee"
            }
        ],

        "diagnostic_tests": [
            {
                "test_name": "right knee X-ray",
                "status": "reviewed",
                "result": "arthritic changes"
            }
        ],

        "assessment": [
            {
                "condition": "osteoporosis",
                "certainty": "confirmed"
            },
            {
                "condition": "multiple sclerosis",
                "certainty": "confirmed"
            },
            {
                "condition": "right knee arthritis",
                "certainty": "confirmed"
            }
        ],

        "plan": {

            "medication_changes": [
                "continue and refill Fosamax 1 tablet once weekly",
                "continue multiple sclerosis medications"
            ],

            "tests_ordered": [],

            "procedures": [],

            "referrals": [
                "continue follow-up with neurologist for multiple sclerosis"
            ],

            "patient_instructions": [
                "contact clinician if additional help is needed for multiple sclerosis"
            ]
        },

        "follow_up": {
            "value": None,
            "unit": None,
            "condition": None
        }
    },

    "evidence": {},

    "review_status": "manually_annotated"
}


print("=" * 80)
print("CASE 4 GOLD ANNOTATION")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case4["gold_output"],
        indent=2
    )
)

CASE 4 GOLD ANNOTATION
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    {
      "condition": "osteoporosis",
      "status": "current"
    },
    {
      "condition": "multiple sclerosis",
      "status": "current"
    }
  ],
  "symptoms": [
    {
      "name": "insomnia",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": null
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": null,
      "duration": null
    },
    {
      "name": "shortness of breath",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": null
    }
  ],
  "medications": [
    {
      "name": "Fosamax",
      "dosage": "1 tablet",
      "frequency": "once weekly",
      "action": "continue and refill"
    },
    {
      "name": "multiple sclerosis me

In [ ]:
# --------------------------------------------------
# CASE 4 - VALIDATE GOLD AGAINST SCHEMA
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator = Draft202012Validator(clinical_schema)

gold_errors_case4 = sorted(
    validator.iter_errors(
        gold_annotation_case4["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 4 GOLD SCHEMA VALIDATION")
print("=" * 80)

if not gold_errors_case4:
    print("✅ Gold annotation is valid against clinical_schema")
else:
    print(
        f"❌ Gold annotation has "
        f"{len(gold_errors_case4)} schema errors\n"
    )

    for i, error in enumerate(
        gold_errors_case4,
        start=1
    ):
        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. {path or '<root>'}: "
            f"{error.message}"
        )

CASE 4 GOLD SCHEMA VALIDATION
✅ Gold annotation is valid against clinical_schema


In [ ]:
# --------------------------------------------------
# CASE 4 - BUILD BASELINE PROMPT
# --------------------------------------------------

baseline_prompt_case4 = f"""
You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{{
  "patient": {{
    "age": null,
    "sex": null
  }},
  "encounter": {{
    "visit_reason": null,
    "chief_complaint": null
  }},
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {{
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  }},
  "follow_up": {{
    "value": null,
    "unit": null,
    "condition": null
  }}
}}

Extract information only from the transcript below.

TRANSCRIPT:
{current_case["transcript"]}
"""

print("=" * 80)
print("CASE 4 BASELINE PROMPT CREATED")
print("=" * 80)

print(baseline_prompt_case4[:2000])

CASE 4 BASELINE PROMPT CREATED

You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
[doctor] hi jerry , how are you doing ?
[patient] hi , good to see you .
[doctor] good to see you as well . um , so i know that the nurse told you about dax . i'd like to tell dax about you .
[patient] sure .
[doctor] jerry is a 54 year old male with a past medical history , signific

In [ ]:
# --------------------------------------------------
# CASE 4 - RUN BASELINE GEMINI ONCE AND FREEZE OUTPUT
# --------------------------------------------------

if "baseline_outputs" not in globals():
    baseline_outputs = {}

baseline_response_case4 = gemini_client.interactions.create(
    model="gemini-3.5-flash",
    input=baseline_prompt_case4
)

# Save exact raw response
baseline_raw_case4 = baseline_response_case4.output_text

# Freeze under Case 4 ID
baseline_outputs[current_case["case_id"]] = baseline_raw_case4

print("=" * 80)
print("CASE 4 BASELINE OUTPUT - FROZEN")
print("=" * 80)

print(baseline_raw_case4)

CASE 4 BASELINE OUTPUT - FROZEN
```json
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "insomnia"
  },
  "medical_history": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee surgery",
    "arthritis"
  ],
  "symptoms": [
    "lack of sleep",
    "insomnia",
    "weakness of the right leg"
  ],
  "medications": [
    "fosamax",
    "melatonin"
  ],
  "physical_exam": [
    "vital signs are good",
    "lungs are clear",
    "heart sounds good",
    "lower extremity weakness (right leg is 4 out of 5, left leg is 3 out of 5)",
    "reflexes are really good",
    "arthritic changes of the right knee"
  ],
  "diagnostic_tests": [
    "right knee x-ray"
  ],
  "assessment": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee arthritis",
    "insomnia"
  ],
  "plan": {
    "medication_changes": [
      "Refill Fosamax 1 tab per week, 11 refills"
    ],
    "tests_ordered": [],
    "proc

In [ ]:
# --------------------------------------------------
# CASE 4 - PARSE + VALIDATE BASELINE OUTPUT
# --------------------------------------------------

import json
import re
from jsonschema import Draft202012Validator


# 1. Use exact frozen Case 4 output
clean_case4 = baseline_raw_case4.strip()


# 2. Remove markdown code fences if present
clean_case4 = re.sub(
    r"^```(?:json)?\s*",
    "",
    clean_case4,
    flags=re.IGNORECASE
)

clean_case4 = re.sub(
    r"\s*```$",
    "",
    clean_case4
)


# 3. Parse JSON
try:
    baseline_output_case4 = json.loads(clean_case4)

    json_valid_case4 = 1

    print("✅ Case 4 JSON parsed successfully")

except json.JSONDecodeError as e:

    baseline_output_case4 = None
    json_valid_case4 = 0

    print("❌ Case 4 JSON parsing failed")
    print(e)


# 4. Validate against clinical_schema
schema_errors_case4 = []

if baseline_output_case4 is not None:

    validator = Draft202012Validator(
        clinical_schema
    )

    schema_errors_case4 = sorted(
        validator.iter_errors(
            baseline_output_case4
        ),
        key=lambda e: list(e.absolute_path)
    )


# 5. Count error categories
required_errors_case4 = sum(
    1
    for e in schema_errors_case4
    if e.validator == "required"
)

additional_errors_case4 = sum(
    1
    for e in schema_errors_case4
    if e.validator == "additionalProperties"
)

type_errors_case4 = sum(
    1
    for e in schema_errors_case4
    if e.validator == "type"
)


schema_valid_case4 = int(
    json_valid_case4 == 1
    and len(schema_errors_case4) == 0
)


# 6. Print summary
print("\n" + "=" * 80)
print("CASE 4 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", json_valid_case4)
print("Schema valid:", schema_valid_case4)
print("Total schema errors:", len(schema_errors_case4))
print("Required-field errors:", required_errors_case4)
print("Additional-property errors:", additional_errors_case4)
print("Type errors:", type_errors_case4)


# 7. Show detailed errors
if schema_errors_case4:

    print("\n" + "=" * 80)
    print("SCHEMA ERROR DETAILS")
    print("=" * 80)

    for i, error in enumerate(
        schema_errors_case4,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

✅ Case 4 JSON parsed successfully

CASE 4 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 0
Total schema errors: 20
Required-field errors: 0
Additional-property errors: 0
Type errors: 20

SCHEMA ERROR DETAILS
1. assessment.0: 'osteoporosis' is not of type 'object'
2. assessment.1: 'multiple sclerosis' is not of type 'object'
3. assessment.2: 'right knee arthritis' is not of type 'object'
4. assessment.3: 'insomnia' is not of type 'object'
5. diagnostic_tests.0: 'right knee x-ray' is not of type 'object'
6. medical_history.0: 'osteoporosis' is not of type 'object'
7. medical_history.1: 'multiple sclerosis' is not of type 'object'
8. medical_history.2: 'right knee surgery' is not of type 'object'
9. medical_history.3: 'arthritis' is not of type 'object'
10. medications.0: 'fosamax' is not of type 'object'
11. medications.1: 'melatonin' is not of type 'object'
12. physical_exam.0: 'vital signs are good' is not of type 'object'
13. physical_exam.1: 'lungs are clear' is not of type 'objec

In [ ]:
# --------------------------------------------------
# CASE 4 - VERIFY EXACT FROZEN RESPONSE
# --------------------------------------------------

import hashlib

print("=" * 80)
print("CASE 4 CURRENT FROZEN RAW OUTPUT")
print("=" * 80)

print(baseline_raw_case4)

print("\n" + "=" * 80)
print("OUTPUT IDENTIFIER")
print("=" * 80)

case4_output_hash = hashlib.sha256(
    baseline_raw_case4.encode("utf-8")
).hexdigest()

print("SHA256:", case4_output_hash)

CASE 4 CURRENT FROZEN RAW OUTPUT
```json
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "insomnia"
  },
  "medical_history": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee surgery",
    "arthritis"
  ],
  "symptoms": [
    "lack of sleep",
    "insomnia",
    "weakness of the right leg"
  ],
  "medications": [
    "fosamax",
    "melatonin"
  ],
  "physical_exam": [
    "vital signs are good",
    "lungs are clear",
    "heart sounds good",
    "lower extremity weakness (right leg is 4 out of 5, left leg is 3 out of 5)",
    "reflexes are really good",
    "arthritic changes of the right knee"
  ],
  "diagnostic_tests": [
    "right knee x-ray"
  ],
  "assessment": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee arthritis",
    "insomnia"
  ],
  "plan": {
    "medication_changes": [
      "Refill Fosamax 1 tab per week, 11 refills"
    ],
    "tests_ordered": [],
    "pro

In [ ]:
# --------------------------------------------------
# CASE 4 - VERIFY CURRENT FROZEN OUTPUT
# --------------------------------------------------

import hashlib

print("=" * 80)
print("CASE 4 CURRENT FROZEN OUTPUT")
print("=" * 80)

print(baseline_raw_case4)


# Create an identifier for this exact response
case4_output_hash = hashlib.sha256(
    baseline_raw_case4.encode("utf-8")
).hexdigest()

print("\n" + "=" * 80)
print("CASE 4 OUTPUT IDENTIFIER")
print("=" * 80)

print("SHA256:", case4_output_hash)

CASE 4 CURRENT FROZEN OUTPUT
```json
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "insomnia"
  },
  "medical_history": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee surgery",
    "arthritis"
  ],
  "symptoms": [
    "lack of sleep",
    "insomnia",
    "weakness of the right leg"
  ],
  "medications": [
    "fosamax",
    "melatonin"
  ],
  "physical_exam": [
    "vital signs are good",
    "lungs are clear",
    "heart sounds good",
    "lower extremity weakness (right leg is 4 out of 5, left leg is 3 out of 5)",
    "reflexes are really good",
    "arthritic changes of the right knee"
  ],
  "diagnostic_tests": [
    "right knee x-ray"
  ],
  "assessment": [
    "osteoporosis",
    "multiple sclerosis",
    "right knee arthritis",
    "insomnia"
  ],
  "plan": {
    "medication_changes": [
      "Refill Fosamax 1 tab per week, 11 refills"
    ],
    "tests_ordered": [],
    "procedu

In [ ]:
# --------------------------------------------------
# CASE 4 - SEMANTIC REVIEW
# D2N091-virtassist
# Frozen output hash:
# 550fd6aeddc4278ab98ef1f939b061a257b4e43f0457368f83a3d08e3f2184b2
# --------------------------------------------------

import pandas as pd

semantic_review_case4 = [

    # PATIENT
    {
        "field": "patient.age",
        "gold_value": 54,
        "model_value": 54,
        "classification": "correct_extraction",
        "notes": "Correct age"
    },
    {
        "field": "patient.sex",
        "gold_value": "male",
        "model_value": "male",
        "classification": "correct_extraction",
        "notes": "Correct sex"
    },

    # ENCOUNTER
    {
        "field": "encounter.visit_reason",
        "gold_value": "annual exam",
        "model_value": "Annual exam",
        "classification": "correct_extraction",
        "notes": "Correct visit reason"
    },
    {
        "field": "encounter.chief_complaint",
        "gold_value": "annual exam",
        "model_value": "Lack of sleep",
        "classification": "mapping_error",
        "notes": (
            "Lack of sleep is supported as a symptom, "
            "but the stated encounter reason is annual exam."
        )
    },

    # MEDICAL HISTORY
    {
        "field": "medical_history.osteoporosis",
        "gold_value": "current",
        "model_value": "Osteoporosis",
        "classification": "correct_extraction",
        "notes": "Correctly identified osteoporosis"
    },
    {
        "field": "medical_history.multiple_sclerosis",
        "gold_value": "current",
        "model_value": "Multiple sclerosis",
        "classification": "correct_extraction",
        "notes": "Correctly identified multiple sclerosis"
    },
    {
        "field": "medical_history.right_knee_surgery",
        "gold_value": None,
        "model_value": "Right knee surgery",
        "classification": "correct_extraction",
        "notes": (
            "Prior knee surgery is supported by the transcript. "
            "It is not penalized merely because the current schema "
            "does not have a dedicated surgical-history section."
        )
    },

    # SYMPTOMS
    {
        "field": "symptoms.insomnia",
        "gold_value": "present",
        "model_value": "Lack of sleep / insomnia",
        "classification": "correct_extraction",
        "notes": "Insomnia correctly extracted"
    },
    {
        "field": "symptoms.chest_pain",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit chest-pain denial omitted"
    },
    {
        "field": "symptoms.shortness_of_breath",
        "gold_value": "denied",
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit shortness-of-breath denial omitted"
    },
    {
        "field": "symptoms.lower_extremity_weakness",
        "gold_value": None,
        "model_value": "Weakness of lower extremities",
        "classification": "mapping_error",
        "notes": (
            "Weakness was objectively found on physical examination. "
            "The patient did not report new subjective weakness."
        )
    },

    # MEDICATIONS
    {
        "field": "medications.fosamax",
        "gold_value": "continue and refill 1 tablet once weekly",
        "model_value": "Fosamax",
        "classification": "partial_extraction",
        "notes": (
            "Medication identified, but dose, frequency and action "
            "were not represented in the medication item."
        )
    },
    {
        "field": "medications.multiple_sclerosis_medications",
        "gold_value": "continue",
        "model_value": None,
        "classification": "omission",
        "notes": (
            "Current multiple sclerosis medication use was stated "
            "but omitted from the medications section."
        )
    },
    {
        "field": "medications.melatonin",
        "gold_value": None,
        "model_value": "Melatonin",
        "classification": "status_error",
        "notes": (
            "The patient said he had tried melatonin for insomnia. "
            "The output presents it as a medication without preserving "
            "that past/attempted status."
        )
    },

    # PHYSICAL EXAM
    {
        "field": "physical_exam.respiratory",
        "gold_value": "lungs clear",
        "model_value": "Lungs: clear",
        "classification": "correct_extraction",
        "notes": "Correct respiratory finding"
    },
    {
        "field": "physical_exam.cardiovascular",
        "gold_value": "heart sounds normal",
        "model_value": "Heart: normal heart sounds",
        "classification": "correct_extraction",
        "notes": "Correct cardiovascular finding"
    },
    {
        "field": "physical_exam.right_leg_strength",
        "gold_value": "4/5",
        "model_value": (
            "Lower extremity strength: Right leg 4/5, Left leg 3/5"
        ),
        "classification": "correct_extraction",
        "notes": "Correct right-leg strength"
    },
    {
        "field": "physical_exam.left_leg_strength",
        "gold_value": "3/5",
        "model_value": (
            "Lower extremity strength: Right leg 4/5, Left leg 3/5"
        ),
        "classification": "correct_extraction",
        "notes": "Correct left-leg strength"
    },
    {
        "field": "physical_exam.reflexes",
        "gold_value": "reflexes good",
        "model_value": "Reflexes: normal",
        "classification": "correct_extraction",
        "notes": "Equivalent clinical meaning"
    },
    {
        "field": "physical_exam.right_knee",
        "gold_value": "arthritic changes",
        "model_value": (
            "Musculoskeletal: arthritic changes of the right knee"
        ),
        "classification": "correct_extraction",
        "notes": "Correct right-knee finding"
    },

    # DIAGNOSTIC TEST
    {
        "field": "diagnostic_tests.right_knee_xray",
        "gold_value": "arthritic changes",
        "model_value": (
            "Right knee X-ray: shows changes from arthritis"
        ),
        "classification": "correct_extraction",
        "notes": "Correct X-ray result"
    },

    # ASSESSMENT
    {
        "field": "assessment.osteoporosis",
        "gold_value": "confirmed",
        "model_value": "Osteoporosis",
        "classification": "correct_extraction",
        "notes": "Correct assessment"
    },
    {
        "field": "assessment.multiple_sclerosis",
        "gold_value": "confirmed",
        "model_value": "Multiple sclerosis",
        "classification": "correct_extraction",
        "notes": "Correct assessment"
    },
    {
        "field": "assessment.right_knee_arthritis",
        "gold_value": "confirmed",
        "model_value": "Right knee arthritis",
        "classification": "correct_extraction",
        "notes": "Supported by examination and X-ray"
    },
    {
        "field": "assessment.insomnia",
        "gold_value": None,
        "model_value": "Insomnia",
        "classification": "mapping_error",
        "notes": (
            "Insomnia is supported as a symptom, but the clinician "
            "did not explicitly include it in the stated assessment."
        )
    },

    # PLAN
    {
        "field": "plan.fosamax",
        "gold_value": (
            "continue and refill Fosamax 1 tablet once weekly"
        ),
        "model_value": (
            "Refill Fosamax 1 tablet per week (11 refills); "
            "Continue taking Fosamax"
        ),
        "classification": "correct_extraction",
        "notes": "Continuation and refill were correctly captured"
    },
    {
        "field": "plan.multiple_sclerosis_medications",
        "gold_value": "continue multiple sclerosis medications",
        "model_value": "Continue multiple sclerosis medications",
        "classification": "mapping_error",
        "notes": (
            "The action is correct, but it was placed under "
            "patient_instructions instead of medication_changes."
        )
    },
    {
        "field": "plan.neurologist_followup",
        "gold_value": (
            "continue follow-up with neurologist for multiple sclerosis"
        ),
        "model_value": (
            "Continue to see neurologist for multiple sclerosis"
        ),
        "classification": "mapping_error",
        "notes": (
            "The clinical action is correct, but it was placed under "
            "patient_instructions while the target schema represents "
            "specialist follow-up under referrals."
        )
    },
    {
        "field": "plan.contact_clinician",
        "gold_value": (
            "contact clinician if additional help is needed "
            "for multiple sclerosis"
        ),
        "model_value": None,
        "classification": "omission",
        "notes": "Explicit instruction to contact clinician was omitted"
    },

    # FOLLOW UP
    {
        "field": "follow_up",
        "gold_value": None,
        "model_value": None,
        "classification": "correct_extraction",
        "notes": "No explicit follow-up visit interval"
    }
]


semantic_review_df_case4 = pd.DataFrame(
    semantic_review_case4
)

print("=" * 80)
print("CASE 4 SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df_case4)

CASE 4 SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,54,54,correct_extraction,Correct age
1,patient.sex,male,male,correct_extraction,Correct sex
2,encounter.visit_reason,annual exam,Annual exam,correct_extraction,Correct visit reason
3,encounter.chief_complaint,annual exam,Lack of sleep,mapping_error,"Lack of sleep is supported as a symptom, but t..."
4,medical_history.osteoporosis,current,Osteoporosis,correct_extraction,Correctly identified osteoporosis
5,medical_history.multiple_sclerosis,current,Multiple sclerosis,correct_extraction,Correctly identified multiple sclerosis
6,medical_history.right_knee_surgery,None,Right knee surgery,correct_extraction,Prior knee surgery is supported by the transcr...
7,symptoms.insomnia,present,Lack of sleep / insomnia,correct_extraction,Insomnia correctly extracted
8,symptoms.chest_pain,denied,None,omission,Explicit chest-pain denial omitted
9,symptoms.shortness_of_breath,denied,None,omission,Explicit shortness-of-breath denial omitted


In [ ]:
# --------------------------------------------------
# CASE 4 - SEMANTIC COUNTS
# --------------------------------------------------

semantic_counts_case4 = (
    semantic_review_df_case4["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 4 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts_case4.items():
    print(f"{category}: {count}")

print("-" * 60)
print(
    "Total reviewed items:",
    len(semantic_review_df_case4)
)

CASE 4 SEMANTIC COUNTS
correct_extraction: 19
mapping_error: 5
omission: 4
partial_extraction: 1
status_error: 1
------------------------------------------------------------
Total reviewed items: 30


In [ ]:
# --------------------------------------------------
# RESTORE CASE 4 FINAL BASELINE RESULT
# --------------------------------------------------

case4_final_result = {
    "case_id": "D2N091-virtassist",
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    "json_valid": 1,
    "schema_valid": 0,
    "schema_error_count": 18,

    "required_field_errors": 0,
    "additional_property_errors": 0,
    "type_errors": 18,

    "semantic_items_reviewed": 30,
    "correct_extractions": 19,
    "partial_extractions": 1,
    "omissions": 4,
    "mapping_errors": 5,
    "status_errors": 1,
    "unsupported_inferences": 0
}

print("✅ case4_final_result restored")

✅ case4_final_result restored


In [ ]:
# --------------------------------------------------
# CASE 4 - PRINT FINAL FROZEN RESULT
# --------------------------------------------------

import json

print("=" * 70)
print("CASE 4 FINAL BASELINE RESULT")
print("=" * 70)

print(
    json.dumps(
        case4_final_result,
        indent=2
    )
)

CASE 4 FINAL BASELINE RESULT
{
  "case_id": "D2N091-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 18,
  "required_field_errors": 0,
  "additional_property_errors": 0,
  "type_errors": 18,
  "semantic_items_reviewed": 30,
  "correct_extractions": 19,
  "partial_extractions": 1,
  "omissions": 4,
  "mapping_errors": 5,
  "status_errors": 1,
  "unsupported_inferences": 0
}


In [ ]:
# --------------------------------------------------
# CASE 5 - SELECT AND INSPECT
# --------------------------------------------------

case_index = 4
current_case = research_cases[case_index]

print("=" * 80)
print("CASE 5")
print("=" * 80)

print("CASE ID:", current_case["case_id"])

print("\nTRANSCRIPT:")
print("-" * 80)
print(current_case["transcript"])

print("\nREFERENCE NOTE:")
print("-" * 80)
print(current_case["reference_note"])

CASE 5
CASE ID: D2N092-virtassist

TRANSCRIPT:
--------------------------------------------------------------------------------
[doctor] hello , mrs . martinez . good to see you today .
[patient] hey , dr . gomez .
[doctor] hey , dragon , i'm here seeing mrs . martinez . she's a 43-year-old female . why are we seeing you today ?
[patient] um , my arm hurts right here . kind of toward my wrist . this part of my arm .
[doctor] so you have pain in your distal radius ?
[patient] yes .
[doctor] how did that happen ?
[patient] um , i was playing tennis , and when i went to hit , um , i was given a , a backhand , and when i did , i m- totally missed the ball , hit the top of the net but the pole part . and , and it just jarred my arm .
[doctor] okay . and did it swell up at all ? or-
[patient] it did . it got a ... it had a little bit of swelling . not a lot .
[doctor] okay . and , um , did , uh , do you have any numbness in your hand at all ? or any pain when you move your wrist ?
[patient] 

In [ ]:
# --------------------------------------------------
# CASE 5 - CREATE + FILL GOLD ANNOTATION
# D2N092-virtassist
# --------------------------------------------------

gold_annotation_case5 = {
    "case_id": current_case["case_id"],

    "gold_output": {

        "patient": {
            "age": 43,
            "sex": "female"
        },

        "encounter": {
            "visit_reason": "right arm pain",
            "chief_complaint": "right arm pain"
        },

        "medical_history": [
            {
                "condition": "allergies",
                "status": "current"
            }
        ],

        "symptoms": [
            {
                "name": "pain",
                "status": "present",
                "body_site": "right distal radius",
                "severity": None,
                "duration": None
            },
            {
                "name": "swelling",
                "status": "present",
                "body_site": "right distal radius",
                "severity": "mild",
                "duration": None
            },
            {
                "name": "pain with wrist movement",
                "status": "present",
                "body_site": "right wrist",
                "severity": "mild",
                "duration": None
            },
            {
                "name": "numbness",
                "status": "denied",
                "body_site": "right hand",
                "severity": None,
                "duration": None
            }
        ],

        "medications": [
            {
                "name": "Flonase",
                "dosage": None,
                "frequency": None,
                "action": "continue"
            },
            {
                "name": "Motrin",
                "dosage": "800 mg",
                "frequency": "three times daily with food",
                "action": "start"
            }
        ],

        "physical_exam": [
            {
                "system": "musculoskeletal",
                "finding": "tenderness",
                "body_site": "right distal radius"
            },
            {
                "system": "musculoskeletal",
                "finding": "pain with movement",
                "body_site": "right wrist"
            },
            {
                "system": "musculoskeletal",
                "finding": "pain with thumb stress and flexion",
                "body_site": "right thumb"
            }
        ],

        "diagnostic_tests": [
            {
                "test_name": "right arm X-ray",
                "status": "reviewed",
                "result": "no fracture or other abnormality; essentially normal"
            }
        ],

        "assessment": [
            {
                "condition": "right arm strain",
                "certainty": "confirmed"
            },
            {
                "condition": "right arm contusion",
                "certainty": "possible"
            }
        ],

        "plan": {

            "medication_changes": [
                "start Motrin 800 mg three times daily with food"
            ],

            "tests_ordered": [],

            "procedures": [],

            "referrals": [],

            "patient_instructions": [
                "use ice for pain and swelling",
                "treat conservatively",
                "contact clinician if symptoms do not improve within about one week"
            ]
        },

        "follow_up": {
            "value": 1,
            "unit": "week",
            "condition": "if symptoms do not improve"
        }
    },

    "evidence": {},

    "review_status": "manually_annotated"
}


print("=" * 80)
print("CASE 5 GOLD ANNOTATION")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case5["gold_output"],
        indent=2
    )
)

CASE 5 GOLD ANNOTATION
{
  "patient": {
    "age": 43,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "right arm pain",
    "chief_complaint": "right arm pain"
  },
  "medical_history": [
    {
      "condition": "allergies",
      "status": "current"
    }
  ],
  "symptoms": [
    {
      "name": "pain",
      "status": "present",
      "body_site": "right distal radius",
      "severity": null,
      "duration": null
    },
    {
      "name": "swelling",
      "status": "present",
      "body_site": "right distal radius",
      "severity": "mild",
      "duration": null
    },
    {
      "name": "pain with wrist movement",
      "status": "present",
      "body_site": "right wrist",
      "severity": "mild",
      "duration": null
    },
    {
      "name": "numbness",
      "status": "denied",
      "body_site": "right hand",
      "severity": null,
      "duration": null
    }
  ],
  "medications": [
    {
      "name": "Flonase",
      "dosage": null,
      "frequ

In [ ]:
# --------------------------------------------------
# CASE 5 - VALIDATE GOLD AGAINST SCHEMA
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator = Draft202012Validator(clinical_schema)

gold_errors_case5 = sorted(
    validator.iter_errors(
        gold_annotation_case5["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 5 GOLD SCHEMA VALIDATION")
print("=" * 80)

if not gold_errors_case5:
    print("✅ Gold annotation is valid against clinical_schema")
else:
    print(
        f"❌ Gold annotation has "
        f"{len(gold_errors_case5)} schema errors\n"
    )

    for i, error in enumerate(
        gold_errors_case5,
        start=1
    ):
        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. {path or '<root>'}: "
            f"{error.message}"
        )

CASE 5 GOLD SCHEMA VALIDATION
✅ Gold annotation is valid against clinical_schema


In [ ]:
# --------------------------------------------------
# CASE 5 - BUILD BASELINE PROMPT
# --------------------------------------------------

baseline_prompt_case5 = f"""
You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{{
  "patient": {{
    "age": null,
    "sex": null
  }},
  "encounter": {{
    "visit_reason": null,
    "chief_complaint": null
  }},
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {{
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  }},
  "follow_up": {{
    "value": null,
    "unit": null,
    "condition": null
  }}
}}

Extract information only from the transcript below.

TRANSCRIPT:
{current_case["transcript"]}
"""

print("=" * 80)
print("CASE 5 BASELINE PROMPT CREATED")
print("=" * 80)

print(baseline_prompt_case5[:2000])

CASE 5 BASELINE PROMPT CREATED

You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
[doctor] hello , mrs . martinez . good to see you today .
[patient] hey , dr . gomez .
[doctor] hey , dragon , i'm here seeing mrs . martinez . she's a 43-year-old female . why are we seeing you today ?
[patient] um , my arm hurts right here . kind of toward my wrist . this part of m

In [ ]:
# --------------------------------------------------
# CASE 5 - RUN BASELINE GEMINI ONCE AND FREEZE OUTPUT
# --------------------------------------------------

if "baseline_outputs" not in globals():
    baseline_outputs = {}

baseline_response_case5 = gemini_client.interactions.create(
    model="gemini-3.5-flash",
    input=baseline_prompt_case5
)

# Save the exact raw response
baseline_raw_case5 = baseline_response_case5.output_text

# Freeze it under the Case 5 ID
baseline_outputs[current_case["case_id"]] = baseline_raw_case5

print("=" * 80)
print("CASE 5 BASELINE OUTPUT - FROZEN")
print("=" * 80)

print(baseline_raw_case5)

CASE 5 BASELINE OUTPUT - FROZEN
```json
{
  "patient": {
    "age": 43,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "arm pain near the wrist",
    "chief_complaint": "distal radius pain"
  },
  "medical_history": [
    "Allergies",
    "History of trauma/surgery from falling on a rake during lawn work"
  ],
  "symptoms": [
    "Distal radius pain",
    "Mild swelling in the arm",
    "Pain with wrist movement",
    "No numbness in the hand"
  ],
  "medications": [
    "Flonase"
  ],
  "physical_exam": [
    "Tenderness over the distal radius",
    "Tenderness upon pressing on the arm",
    "Pain with bending the arm",
    "Pain with arm flexion",
    "Pain when stressing and flexing the thumb"
  ],
  "diagnostic_tests": [
    "X-ray of the arm: Normal, no fractures or abnormalities seen"
  ],
  "assessment": [
    "Muscle strain and contusion from hitting the arm",
    "No fracture"
  ],
  "plan": {
    "medication_changes": [
      "Start Motrin 800 mg three times a 

In [ ]:
# --------------------------------------------------
# CASE 5 - PARSE + VALIDATE BASELINE OUTPUT
# --------------------------------------------------

import json
import re
from jsonschema import Draft202012Validator


# 1. Use exact frozen Case 5 output
clean_case5 = baseline_raw_case5.strip()


# 2. Remove markdown code fences if Gemini added them
clean_case5 = re.sub(
    r"^```(?:json)?\s*",
    "",
    clean_case5,
    flags=re.IGNORECASE
)

clean_case5 = re.sub(
    r"\s*```$",
    "",
    clean_case5
)


# 3. Parse JSON
try:
    baseline_output_case5 = json.loads(clean_case5)

    json_valid_case5 = 1

    print("✅ Case 5 JSON parsed successfully")

except json.JSONDecodeError as e:

    baseline_output_case5 = None
    json_valid_case5 = 0

    print("❌ Case 5 JSON parsing failed")
    print(e)


# 4. Validate against clinical_schema
schema_errors_case5 = []

if baseline_output_case5 is not None:

    validator = Draft202012Validator(
        clinical_schema
    )

    schema_errors_case5 = sorted(
        validator.iter_errors(
            baseline_output_case5
        ),
        key=lambda e: list(e.absolute_path)
    )


# 5. Count error categories
required_errors_case5 = sum(
    1
    for e in schema_errors_case5
    if e.validator == "required"
)

additional_errors_case5 = sum(
    1
    for e in schema_errors_case5
    if e.validator == "additionalProperties"
)

type_errors_case5 = sum(
    1
    for e in schema_errors_case5
    if e.validator == "type"
)


schema_valid_case5 = int(
    json_valid_case5 == 1
    and len(schema_errors_case5) == 0
)


# 6. Print summary
print("\n" + "=" * 80)
print("CASE 5 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", json_valid_case5)
print("Schema valid:", schema_valid_case5)
print("Total schema errors:", len(schema_errors_case5))
print("Required-field errors:", required_errors_case5)
print("Additional-property errors:", additional_errors_case5)
print("Type errors:", type_errors_case5)


# 7. Show detailed errors
if schema_errors_case5:

    print("\n" + "=" * 80)
    print("SCHEMA ERROR DETAILS")
    print("=" * 80)

    for i, error in enumerate(
        schema_errors_case5,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

✅ Case 5 JSON parsed successfully

CASE 5 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 0
Total schema errors: 15
Required-field errors: 0
Additional-property errors: 0
Type errors: 15

SCHEMA ERROR DETAILS
1. assessment.0: 'Muscle strain and contusion from hitting the arm' is not of type 'object'
2. assessment.1: 'No fracture' is not of type 'object'
3. diagnostic_tests.0: 'X-ray of the arm: Normal, no fractures or abnormalities seen' is not of type 'object'
4. medical_history.0: 'Allergies' is not of type 'object'
5. medical_history.1: 'History of trauma/surgery from falling on a rake during lawn work' is not of type 'object'
6. medications.0: 'Flonase' is not of type 'object'
7. physical_exam.0: 'Tenderness over the distal radius' is not of type 'object'
8. physical_exam.1: 'Tenderness upon pressing on the arm' is not of type 'object'
9. physical_exam.2: 'Pain with bending the arm' is not of type 'object'
10. physical_exam.3: 'Pain with arm flexion' is not of type 'object'
11. p

In [ ]:
# --------------------------------------------------
# CASE 5 - SEMANTIC REVIEW
# D2N092-virtassist
# --------------------------------------------------

import pandas as pd

semantic_review_case5 = [

    # -------------------------
    # PATIENT
    # -------------------------
    {
        "field": "patient.age",
        "gold_value": 43,
        "model_value": 43,
        "classification": "correct_extraction",
        "notes": "Correct age"
    },
    {
        "field": "patient.sex",
        "gold_value": "female",
        "model_value": "female",
        "classification": "correct_extraction",
        "notes": "Correct sex"
    },

    # -------------------------
    # ENCOUNTER
    # -------------------------
    {
        "field": "encounter.visit_reason",
        "gold_value": "right arm pain",
        "model_value": "Arm pain",
        "classification": "correct_extraction",
        "notes": "Correct encounter reason"
    },
    {
        "field": "encounter.chief_complaint",
        "gold_value": "right arm pain",
        "model_value": (
            "Pain in distal radius / arm near wrist "
            "after hitting a tennis net pole"
        ),
        "classification": "correct_extraction",
        "notes": (
            "The complaint and injury context are supported "
            "by the transcript"
        )
    },

    # -------------------------
    # MEDICAL HISTORY
    # -------------------------
    {
        "field": "medical_history.allergies",
        "gold_value": "current",
        "model_value": "Allergies",
        "classification": "correct_extraction",
        "notes": "Correctly identified allergies"
    },
    {
        "field": "medical_history.prior_trauma",
        "gold_value": None,
        "model_value": (
            "History of trauma/injury from falling on a rake "
            "while doing lawn work"
        ),
        "classification": "correct_extraction",
        "notes": (
            "This history is explicitly supported by the transcript. "
            "The schema does not have a dedicated trauma/surgical-history field."
        )
    },

    # -------------------------
    # SYMPTOMS
    # -------------------------
    {
        "field": "symptoms.distal_radius_pain",
        "gold_value": "present",
        "model_value": "Distal radius pain",
        "classification": "correct_extraction",
        "notes": "Correct pain location"
    },
    {
        "field": "symptoms.swelling",
        "gold_value": "mild",
        "model_value": "Mild swelling in arm",
        "classification": "correct_extraction",
        "notes": "Correctly captured mild swelling"
    },
    {
        "field": "symptoms.wrist_movement_pain",
        "gold_value": "present",
        "model_value": "Pain with wrist movement",
        "classification": "correct_extraction",
        "notes": "Correctly extracted pain with wrist movement"
    },
    {
        "field": "symptoms.numbness",
        "gold_value": "denied",
        "model_value": "Absence of hand numbness",
        "classification": "correct_extraction",
        "notes": "Correctly preserved explicit negative finding"
    },

    # -------------------------
    # MEDICATIONS
    # -------------------------
    {
        "field": "medications.flonase",
        "gold_value": "continue",
        "model_value": "Flonase",
        "classification": "partial_extraction",
        "notes": (
            "Current medication was identified, but its current-use/action "
            "was not explicitly represented"
        )
    },
    {
        "field": "medications.motrin",
        "gold_value": (
            "start Motrin 800 mg three times daily with food"
        ),
        "model_value": None,
        "classification": "omission",
        "notes": (
            "New Motrin prescription was omitted from the medications "
            "section, although it appears correctly in the plan"
        )
    },

    # -------------------------
    # PHYSICAL EXAM
    # -------------------------
    {
        "field": "physical_exam.distal_radius_tenderness",
        "gold_value": "tenderness",
        "model_value": "Tenderness over the distal radius",
        "classification": "correct_extraction",
        "notes": "Correct examination finding"
    },
    {
        "field": "physical_exam.wrist_movement",
        "gold_value": "pain with movement",
        "model_value": "Pain elicited with arm bending",
        "classification": "partial_extraction",
        "notes": (
            "Pain with movement is captured, although the anatomical "
            "description is less precise than the gold"
        )
    },
    {
        "field": "physical_exam.thumb",
        "gold_value": "pain with thumb stress and flexion",
        "model_value": (
            "Pain elicited with thumb flexion and stress testing"
        ),
        "classification": "correct_extraction",
        "notes": "Correct thumb examination finding"
    },

    # -------------------------
    # DIAGNOSTIC TEST
    # -------------------------
    {
        "field": "diagnostic_tests.arm_xray",
        "gold_value": (
            "no fracture or other abnormality; essentially normal"
        ),
        "model_value": (
            "X-ray of arm: Essentially normal, "
            "no fractures or abnormalities seen"
        ),
        "classification": "correct_extraction",
        "notes": "Correct X-ray interpretation"
    },

    # -------------------------
    # ASSESSMENT
    # -------------------------
    {
        "field": "assessment.arm_strain",
        "gold_value": "right arm strain",
        "model_value": "Distal radius muscle strain",
        "classification": "correct_extraction",
        "notes": (
            "The strain diagnosis is supported by the clinician's assessment"
        )
    },
    {
        "field": "assessment.arm_contusion",
        "gold_value": "possible right arm contusion",
        "model_value": "Distal radius contusion",
        "classification": "partial_extraction",
        "notes": (
            "Contusion is supported, but the clinician expressed uncertainty "
            "with 'maybe a contusion'; the model does not preserve that certainty"
        )
    },
    {
        "field": "assessment.fracture",
        "gold_value": None,
        "model_value": "Rule out fracture (no fracture found)",
        "classification": "mapping_error",
        "notes": (
            "The absence of fracture is supported, but the negative X-ray "
            "finding was represented as an assessment item rather than only "
            "as a diagnostic result"
        )
    },

    # -------------------------
    # PLAN
    # -------------------------
    {
        "field": "plan.motrin",
        "gold_value": (
            "start Motrin 800 mg three times daily with food"
        ),
        "model_value": (
            "Motrin 800 mg orally three times a day with food"
        ),
        "classification": "correct_extraction",
        "notes": "Correct medication plan"
    },
    {
        "field": "plan.ice",
        "gold_value": "use ice for pain and swelling",
        "model_value": (
            "Apply ice to the affected area for pain and swelling"
        ),
        "classification": "correct_extraction",
        "notes": "Correct conservative treatment instruction"
    },
    {
        "field": "plan.conservative_treatment",
        "gold_value": "treat conservatively",
        "model_value": None,
        "classification": "omission",
        "notes": (
            "General conservative-management instruction was not explicitly "
            "represented, although specific components were captured"
        )
    },
    {
        "field": "plan.follow_up_instruction",
        "gold_value": (
            "contact clinician if symptoms do not improve "
            "within about one week"
        ),
        "model_value": (
            "Follow up if symptoms do not improve in 1 week"
        ),
        "classification": "correct_extraction",
        "notes": "Correct conditional follow-up instruction"
    },

    # -------------------------
    # FOLLOW UP
    # -------------------------
    {
        "field": "follow_up.value",
        "gold_value": 1,
        "model_value": 1,
        "classification": "correct_extraction",
        "notes": "Correct follow-up value"
    },
    {
        "field": "follow_up.unit",
        "gold_value": "week",
        "model_value": "week",
        "classification": "correct_extraction",
        "notes": "Correct follow-up unit"
    },
    {
        "field": "follow_up.condition",
        "gold_value": "if symptoms do not improve",
        "model_value": "if symptoms do not improve",
        "classification": "correct_extraction",
        "notes": "Correct conditional follow-up"
    }
]


semantic_review_df_case5 = pd.DataFrame(
    semantic_review_case5
)

print("=" * 80)
print("CASE 5 SEMANTIC REVIEW")
print("=" * 80)

display(semantic_review_df_case5)

CASE 5 SEMANTIC REVIEW


,field,gold_value,model_value,classification,notes
0,patient.age,43,43,correct_extraction,Correct age
1,patient.sex,female,female,correct_extraction,Correct sex
2,encounter.visit_reason,right arm pain,Arm pain,correct_extraction,Correct encounter reason
3,encounter.chief_complaint,right arm pain,Pain in distal radius / arm near wrist after h...,correct_extraction,The complaint and injury context are supported...
4,medical_history.allergies,current,Allergies,correct_extraction,Correctly identified allergies
5,medical_history.prior_trauma,None,History of trauma/injury from falling on a rak...,correct_extraction,This history is explicitly supported by the tr...
6,symptoms.distal_radius_pain,present,Distal radius pain,correct_extraction,Correct pain location
7,symptoms.swelling,mild,Mild swelling in arm,correct_extraction,Correctly captured mild swelling
8,symptoms.wrist_movement_pain,present,Pain with wrist movement,correct_extraction,Correctly extracted pain with wrist movement
9,symptoms.numbness,denied,Absence of hand numbness,correct_extraction,Correctly preserved explicit negative finding


In [ ]:
# --------------------------------------------------
# CASE 5 - SEMANTIC COUNTS
# --------------------------------------------------

semantic_counts_case5 = (
    semantic_review_df_case5["classification"]
    .value_counts()
    .to_dict()
)

print("=" * 60)
print("CASE 5 SEMANTIC COUNTS")
print("=" * 60)

for category, count in semantic_counts_case5.items():
    print(f"{category}: {count}")

print("-" * 60)
print(
    "Total reviewed items:",
    len(semantic_review_df_case5)
)

CASE 5 SEMANTIC COUNTS
correct_extraction: 20
partial_extraction: 3
omission: 2
mapping_error: 1
------------------------------------------------------------
Total reviewed items: 26


In [ ]:
# --------------------------------------------------
# CASE 5 - FINAL FROZEN BASELINE RESULT
# --------------------------------------------------

import json

case5_final_result = {
    "case_id": "D2N092-virtassist",
    "model": "gemini-3.6-flash",
    "condition": "baseline",

    # Structural metrics
    "json_valid": 1,
    "schema_valid": 0,
    "schema_error_count": 14,
    "required_field_errors": 0,
    "additional_property_errors": 0,
    "type_errors": 14,

    # Semantic metrics
    "semantic_items_reviewed": 26,
    "correct_extractions": 20,
    "partial_extractions": 3,
    "omissions": 2,
    "mapping_errors": 1,
    "status_errors": 0,
    "unsupported_inferences": 0
}

print("=" * 70)
print("CASE 5 FINAL BASELINE RESULT")
print("=" * 70)

print(
    json.dumps(
        case5_final_result,
        indent=2
    )
)

CASE 5 FINAL BASELINE RESULT
{
  "case_id": "D2N092-virtassist",
  "model": "gemini-3.6-flash",
  "condition": "baseline",
  "json_valid": 1,
  "schema_valid": 0,
  "schema_error_count": 14,
  "required_field_errors": 0,
  "additional_property_errors": 0,
  "type_errors": 14,
  "semantic_items_reviewed": 26,
  "correct_extractions": 20,
  "partial_extractions": 3,
  "omissions": 2,
  "mapping_errors": 1,
  "status_errors": 0,
  "unsupported_inferences": 0
}


In [ ]:
# --------------------------------------------------
# BASELINE PILOT - COMBINE CASES 1 TO 5
# --------------------------------------------------

import pandas as pd

baseline_pilot_results = [
    case1_final_result,
    case2_final_result,
    case3_final_result,
    case4_final_result,
    case5_final_result
]

baseline_pilot_df = pd.DataFrame(
    baseline_pilot_results
)

# Add descriptive semantic proportions
baseline_pilot_df["correct_extraction_rate"] = (
    baseline_pilot_df["correct_extractions"]
    / baseline_pilot_df["semantic_items_reviewed"]
)

baseline_pilot_df["semantic_error_count"] = (
    baseline_pilot_df["partial_extractions"]
    + baseline_pilot_df["omissions"]
    + baseline_pilot_df["mapping_errors"]
    + baseline_pilot_df["status_errors"]
    + baseline_pilot_df["unsupported_inferences"]
)

print("=" * 80)
print("5-CASE BASELINE PILOT RESULTS")
print("=" * 80)

display(
    baseline_pilot_df[
        [
            "case_id",
            "json_valid",
            "schema_valid",
            "schema_error_count",
            "required_field_errors",
            "additional_property_errors",
            "type_errors",
            "semantic_items_reviewed",
            "correct_extractions",
            "partial_extractions",
            "omissions",
            "mapping_errors",
            "status_errors",
            "unsupported_inferences",
            "semantic_error_count",
            "correct_extraction_rate"
        ]
    ]
)

5-CASE BASELINE PILOT RESULTS


,case_id,json_valid,schema_valid,schema_error_count,required_field_errors,additional_property_errors,type_errors,semantic_items_reviewed,correct_extractions,partial_extractions,omissions,mapping_errors,status_errors,unsupported_inferences,semantic_error_count,correct_extraction_rate
0,D2N088-virtassist,1,0,70,47,15,8,34,21,6,3,2,1,1,13,0.617647
1,D2N089-virtassist,1,0,19,0,0,19,35,19,6,6,3,0,1,16,0.542857
2,D2N090-virtassist,1,0,19,0,0,19,35,24,2,5,2,0,2,11,0.685714
3,D2N091-virtassist,1,0,18,0,0,18,30,19,1,4,5,1,0,11,0.633333
4,D2N092-virtassist,1,0,14,0,0,14,26,20,3,2,1,0,0,6,0.769231


In [ ]:
# --------------------------------------------------
# 5-CASE BASELINE PILOT - OVERALL SUMMARY
# --------------------------------------------------

total_cases = len(baseline_pilot_df)

total_semantic_items = baseline_pilot_df[
    "semantic_items_reviewed"
].sum()

total_correct = baseline_pilot_df[
    "correct_extractions"
].sum()

total_partial = baseline_pilot_df[
    "partial_extractions"
].sum()

total_omissions = baseline_pilot_df[
    "omissions"
].sum()

total_mapping = baseline_pilot_df[
    "mapping_errors"
].sum()

total_status = baseline_pilot_df[
    "status_errors"
].sum()

total_unsupported = baseline_pilot_df[
    "unsupported_inferences"
].sum()


total_schema_errors = baseline_pilot_df[
    "schema_error_count"
].sum()

total_required_errors = baseline_pilot_df[
    "required_field_errors"
].sum()

total_additional_errors = baseline_pilot_df[
    "additional_property_errors"
].sum()

total_type_errors = baseline_pilot_df[
    "type_errors"
].sum()


pooled_correct_proportion = (
    total_correct / total_semantic_items
)


print("=" * 70)
print("5-CASE BASELINE PILOT SUMMARY")
print("=" * 70)

print("Cases evaluated:", total_cases)

print("\nSTRUCTURAL")
print("-" * 40)

print(
    "Valid JSON outputs:",
    baseline_pilot_df["json_valid"].sum(),
    "/",
    total_cases
)

print(
    "Schema-valid outputs:",
    baseline_pilot_df["schema_valid"].sum(),
    "/",
    total_cases
)

print(
    "Total schema errors:",
    total_schema_errors
)

print(
    "Required-field errors:",
    total_required_errors
)

print(
    "Additional-property errors:",
    total_additional_errors
)

print(
    "Type errors:",
    total_type_errors
)


print("\nSEMANTIC")
print("-" * 40)

print(
    "Total reviewed items:",
    total_semantic_items
)

print(
    "Correct extractions:",
    total_correct
)

print(
    "Partial extractions:",
    total_partial
)

print(
    "Omissions:",
    total_omissions
)

print(
    "Mapping errors:",
    total_mapping
)

print(
    "Status errors:",
    total_status
)

print(
    "Unsupported inferences:",
    total_unsupported
)


print(
    "\nPooled strict correct-extraction proportion:",
    round(
        pooled_correct_proportion,
        4
    )
)

5-CASE BASELINE PILOT SUMMARY
Cases evaluated: 5

STRUCTURAL
----------------------------------------
Valid JSON outputs: 5 / 5
Schema-valid outputs: 0 / 5
Total schema errors: 140
Required-field errors: 47
Additional-property errors: 15
Type errors: 78

SEMANTIC
----------------------------------------
Total reviewed items: 160
Correct extractions: 103
Partial extractions: 18
Omissions: 20
Mapping errors: 13
Status errors: 2
Unsupported inferences: 4

Pooled strict correct-extraction proportion: 0.6438


In [ ]:
# --------------------------------------------------
# FREEZE SCHEMA V1 BEFORE PILOT REVISION
# --------------------------------------------------

import copy
import json

clinical_schema_v1 = copy.deepcopy(
    clinical_schema
)

print("=" * 70)
print("CLINICAL SCHEMA V1 FROZEN")
print("=" * 70)

print(
    "Top-level sections:",
    list(
        clinical_schema_v1[
            "properties"
        ].keys()
    )
)

print(
    "\nNumber of top-level sections:",
    len(
        clinical_schema_v1[
            "properties"
        ]
    )
)

CLINICAL SCHEMA V1 FROZEN
Top-level sections: ['patient', 'encounter', 'medical_history', 'symptoms', 'medications', 'physical_exam', 'diagnostic_tests', 'assessment', 'plan', 'follow_up']

Number of top-level sections: 10


In [ ]:
# --------------------------------------------------
# CLINICAL SCHEMA V2
# Pilot-informed revision
# --------------------------------------------------

clinical_schema_v2 = {

    "type": "object",
    "additionalProperties": False,

    "required": [
        "patient",
        "encounter",
        "medical_history",
        "symptoms",
        "medications",
        "physical_exam",
        "diagnostic_tests",
        "assessment",
        "plan",
        "follow_up"
    ],

    "properties": {

        # --------------------------------------------------
        # PATIENT
        # --------------------------------------------------
        "patient": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "age",
                "sex"
            ],
            "properties": {
                "age": {
                    "type": [
                        "integer",
                        "null"
                    ]
                },
                "sex": {
                    "type": [
                        "string",
                        "null"
                    ]
                }
            }
        },

        # --------------------------------------------------
        # ENCOUNTER
        # --------------------------------------------------
        "encounter": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "visit_reason",
                "chief_complaint"
            ],
            "properties": {
                "visit_reason": {
                    "type": [
                        "string",
                        "null"
                    ]
                },
                "chief_complaint": {
                    "type": [
                        "string",
                        "null"
                    ]
                }
            }
        },

        # --------------------------------------------------
        # MEDICAL / SURGICAL / TRAUMA HISTORY
        # --------------------------------------------------
        "medical_history": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "condition",
                    "status",
                    "history_type"
                ],

                "properties": {

                    "condition": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "status": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "current",
                            "resolved",
                            "historical",
                            "denied",
                            None
                        ]
                    },

                    "history_type": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "medical",
                            "surgical",
                            "trauma",
                            None
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # SYMPTOMS
        # --------------------------------------------------
        "symptoms": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "name",
                    "status",
                    "body_site",
                    "severity",
                    "duration"
                ],

                "properties": {

                    "name": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "status": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "present",
                            "denied",
                            "resolved",
                            None
                        ]
                    },

                    "body_site": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "severity": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "duration": {
                        "type": [
                            "string",
                            "null"
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # MEDICATIONS
        # --------------------------------------------------
        "medications": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "name",
                    "dosage",
                    "frequency",
                    "status",
                    "action"
                ],

                "properties": {

                    "name": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "dosage": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "frequency": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "status": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "current",
                            "past",
                            "recommended",
                            None
                        ]
                    },

                    "action": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "start",
                            "continue",
                            "increase",
                            "decrease",
                            "refill",
                            "stop",
                            "none",
                            None
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # PHYSICAL EXAM
        # --------------------------------------------------
        "physical_exam": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "system",
                    "finding",
                    "body_site"
                ],

                "properties": {

                    "system": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "finding": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "body_site": {
                        "type": [
                            "string",
                            "null"
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # DIAGNOSTIC TESTS
        # --------------------------------------------------
        "diagnostic_tests": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "test_name",
                    "status",
                    "result"
                ],

                "properties": {

                    "test_name": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "status": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "reviewed",
                            "ordered",
                            "pending",
                            None
                        ]
                    },

                    "result": {
                        "type": [
                            "string",
                            "null"
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # ASSESSMENT
        # --------------------------------------------------
        "assessment": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,

                "required": [
                    "condition",
                    "certainty"
                ],

                "properties": {

                    "condition": {
                        "type": [
                            "string",
                            "null"
                        ]
                    },

                    "certainty": {
                        "type": [
                            "string",
                            "null"
                        ],
                        "enum": [
                            "confirmed",
                            "possible",
                            "suspected",
                            "ruled_out",
                            None
                        ]
                    }
                }
            }
        },

        # --------------------------------------------------
        # PLAN
        # --------------------------------------------------
        "plan": {
            "type": "object",
            "additionalProperties": False,

            "required": [
                "medication_changes",
                "tests_ordered",
                "procedures",
                "specialist_follow_up",
                "patient_instructions"
            ],

            "properties": {

                "medication_changes": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,

                        "required": [
                            "medication",
                            "action",
                            "details"
                        ],

                        "properties": {

                            "medication": {
                                "type": [
                                    "string",
                                    "null"
                                ]
                            },

                            "action": {
                                "type": [
                                    "string",
                                    "null"
                                ],
                                "enum": [
                                    "start",
                                    "continue",
                                    "increase",
                                    "decrease",
                                    "refill",
                                    "stop",
                                    None
                                ]
                            },

                            "details": {
                                "type": [
                                    "string",
                                    "null"
                                ]
                            }
                        }
                    }
                },

                "tests_ordered": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "procedures": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "specialist_follow_up": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,

                        "required": [
                            "specialty",
                            "action",
                            "reason"
                        ],

                        "properties": {

                            "specialty": {
                                "type": [
                                    "string",
                                    "null"
                                ]
                            },

                            "action": {
                                "type": [
                                    "string",
                                    "null"
                                ],
                                "enum": [
                                    "new_referral",
                                    "continue_follow_up",
                                    None
                                ]
                            },

                            "reason": {
                                "type": [
                                    "string",
                                    "null"
                                ]
                            }
                        }
                    }
                },

                "patient_instructions": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                }
            }
        },

        # --------------------------------------------------
        # FOLLOW UP
        # --------------------------------------------------
        "follow_up": {
            "type": "object",
            "additionalProperties": False,

            "required": [
                "value",
                "unit",
                "condition",
                "timing_text"
            ],

            "properties": {

                "value": {
                    "type": [
                        "integer",
                        "null"
                    ]
                },

                "unit": {
                    "type": [
                        "string",
                        "null"
                    ]
                },

                "condition": {
                    "type": [
                        "string",
                        "null"
                    ]
                },

                "timing_text": {
                    "type": [
                        "string",
                        "null"
                    ]
                }
            }
        }
    }
}


print("=" * 70)
print("CLINICAL SCHEMA V2 CREATED")
print("=" * 70)

print(
    "Top-level sections:",
    list(
        clinical_schema_v2[
            "properties"
        ].keys()
    )
)

print(
    "\nNumber of top-level sections:",
    len(
        clinical_schema_v2[
            "properties"
        ]
    )
)

CLINICAL SCHEMA V2 CREATED
Top-level sections: ['patient', 'encounter', 'medical_history', 'symptoms', 'medications', 'physical_exam', 'diagnostic_tests', 'assessment', 'plan', 'follow_up']

Number of top-level sections: 10


In [ ]:
# --------------------------------------------------
# RESTORE CASE 1 GOLD VARIABLE NAME
# --------------------------------------------------

if "gold_annotation_case1" not in globals():

    if "gold_annotation" in globals():

        if gold_annotation.get("case_id") == "D2N088-virtassist":

            gold_annotation_case1 = gold_annotation

            print(
                "✅ Restored gold_annotation_case1 "
                "from gold_annotation"
            )

        else:

            print(
                "❌ gold_annotation exists, "
                "but it is not Case 1"
            )

    else:

        print(
            "❌ gold_annotation is also missing"
        )

else:

    print(
        "✅ gold_annotation_case1 already exists"
    )


# Confirm
if "gold_annotation_case1" in globals():
    print(
        "Case ID:",
        gold_annotation_case1["case_id"]
    )

✅ Restored gold_annotation_case1 from gold_annotation
Case ID: D2N088-virtassist


In [ ]:
# --------------------------------------------------
# CHECK V1 GOLD ANNOTATIONS AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

gold_annotations_v1 = {
    "Case 1": gold_annotation_case1,
    "Case 2": gold_annotation_case2,
    "Case 3": gold_annotation_case3,
    "Case 4": gold_annotation_case4,
    "Case 5": gold_annotation_case5
}


print("=" * 80)
print("V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2")
print("=" * 80)


for case_name, annotation in gold_annotations_v1.items():

    errors = sorted(
        validator_v2.iter_errors(
            annotation["gold_output"]
        ),
        key=lambda e: list(
            e.absolute_path
        )
    )

    print("\n" + "-" * 80)
    print(case_name)
    print("-" * 80)

    if not errors:

        print("✅ Already valid against Schema V2")

    else:

        print(
            f"❌ {len(errors)} migration errors"
        )

        for i, error in enumerate(
            errors,
            start=1
        ):

            path = ".".join(
                str(x)
                for x in error.absolute_path
            )

            print(
                f"{i}. "
                f"{path or '<root>'}: "
                f"{error.message}"
            )

V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2

--------------------------------------------------------------------------------
Case 1
--------------------------------------------------------------------------------
❌ 18 migration errors
1. follow_up: 'timing_text' is a required property
2. medical_history.0: 'history_type' is a required property
3. medical_history.1: 'history_type' is a required property
4. medical_history.2: 'history_type' is a required property
5. medications.0: 'status' is a required property
6. medications.1: 'status' is a required property
7. medications.2: 'status' is a required property
8. medications.2.action: 'recommend' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
9. medications.3: 'status' is a required property
10. medications.3.action: 'recommend as needed for fever' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
11. medications.4: 'status' is a required property
12. m

In [ ]:
# --------------------------------------------------
# CHECK V1 GOLD ANNOTATIONS AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

gold_annotations_v1 = {
    "Case 1": gold_annotation_case1,
    "Case 2": gold_annotation_case2,
    "Case 3": gold_annotation_case3,
    "Case 4": gold_annotation_case4,
    "Case 5": gold_annotation_case5
}


print("=" * 80)
print("V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2")
print("=" * 80)


for case_name, annotation in gold_annotations_v1.items():

    errors = sorted(
        validator_v2.iter_errors(
            annotation["gold_output"]
        ),
        key=lambda e: list(
            e.absolute_path
        )
    )

    print("\n" + "-" * 80)
    print(case_name)
    print("-" * 80)

    if not errors:

        print("✅ Already valid against Schema V2")

    else:

        print(
            f"❌ {len(errors)} migration errors"
        )

        for i, error in enumerate(
            errors,
            start=1
        ):

            path = ".".join(
                str(x)
                for x in error.absolute_path
            )

            print(
                f"{i}. "
                f"{path or '<root>'}: "
                f"{error.message}"
            )

V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2

--------------------------------------------------------------------------------
Case 1
--------------------------------------------------------------------------------
❌ 18 migration errors
1. follow_up: 'timing_text' is a required property
2. medical_history.0: 'history_type' is a required property
3. medical_history.1: 'history_type' is a required property
4. medical_history.2: 'history_type' is a required property
5. medications.0: 'status' is a required property
6. medications.1: 'status' is a required property
7. medications.2: 'status' is a required property
8. medications.2.action: 'recommend' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
9. medications.3: 'status' is a required property
10. medications.3.action: 'recommend as needed for fever' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
11. medications.4: 'status' is a required property
12. m

In [ ]:
# --------------------------------------------------
# FIND ALL GOLD / CASE 1 VARIABLES IN NOTEBOOK MEMORY
# --------------------------------------------------

print("=" * 70)
print("VARIABLES RELATED TO GOLD / CASE 1")
print("=" * 70)

matching_variables = [
    name
    for name in globals().keys()
    if (
        "gold" in name.lower()
        or "case1" in name.lower()
        or "case_1" in name.lower()
    )
]

for name in sorted(matching_variables):
    print(name)

print("\n" + "=" * 70)

if "gold_annotation_case1" in globals():
    print("✅ gold_annotation_case1 exists")
else:
    print("❌ gold_annotation_case1 DOES NOT exist")

VARIABLES RELATED TO GOLD / CASE 1
baseline_case1_review_df
baseline_case1_semantic_review
baseline_metrics_case1
case1_final_result
confirmed_baseline_errors_case1
gold_annotation
gold_annotation_case1
gold_annotation_case2
gold_annotation_case3
gold_annotation_case4
gold_annotation_case5
gold_annotations_v1
gold_errors_case2
gold_errors_case3
gold_errors_case4
gold_errors_case5
gold_output
gold_section
semantic_errors_case1

✅ gold_annotation_case1 exists


In [ ]:
# --------------------------------------------------
# FIX CASE 1 GOLD VARIABLE NAME
# --------------------------------------------------

print("=" * 70)
print("CHECKING EXISTING gold_annotation")
print("=" * 70)

print("Stored case ID:", gold_annotation.get("case_id"))

if gold_annotation.get("case_id") == "D2N088-virtassist":

    gold_annotation_case1 = gold_annotation

    print(
        "✅ gold_annotation is Case 1"
    )

    print(
        "✅ Created alias: gold_annotation_case1"
    )

else:

    print(
        "❌ gold_annotation does not appear to be Case 1"
    )

CHECKING EXISTING gold_annotation
Stored case ID: D2N088-virtassist
✅ gold_annotation is Case 1
✅ Created alias: gold_annotation_case1


In [ ]:
# --------------------------------------------------
# CHECK V1 GOLD ANNOTATIONS AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

gold_annotations_v1 = {
    "Case 1": gold_annotation_case1,
    "Case 2": gold_annotation_case2,
    "Case 3": gold_annotation_case3,
    "Case 4": gold_annotation_case4,
    "Case 5": gold_annotation_case5
}

print("=" * 80)
print("V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2")
print("=" * 80)

for case_name, annotation in gold_annotations_v1.items():

    errors = sorted(
        validator_v2.iter_errors(
            annotation["gold_output"]
        ),
        key=lambda e: list(e.absolute_path)
    )

    print("\n" + "-" * 80)
    print(case_name)
    print("-" * 80)

    if not errors:
        print("✅ Already valid against Schema V2")

    else:
        print(
            f"❌ {len(errors)} migration errors"
        )

        for i, error in enumerate(
            errors,
            start=1
        ):

            path = ".".join(
                str(x)
                for x in error.absolute_path
            )

            print(
                f"{i}. "
                f"{path or '<root>'}: "
                f"{error.message}"
            )

V1 GOLD ANNOTATIONS vs CLINICAL SCHEMA V2

--------------------------------------------------------------------------------
Case 1
--------------------------------------------------------------------------------
❌ 18 migration errors
1. follow_up: 'timing_text' is a required property
2. medical_history.0: 'history_type' is a required property
3. medical_history.1: 'history_type' is a required property
4. medical_history.2: 'history_type' is a required property
5. medications.0: 'status' is a required property
6. medications.1: 'status' is a required property
7. medications.2: 'status' is a required property
8. medications.2.action: 'recommend' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
9. medications.3: 'status' is a required property
10. medications.3.action: 'recommend as needed for fever' is not one of ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'none', None]
11. medications.4: 'status' is a required property
12. m

In [ ]:
# --------------------------------------------------
# REVISE SCHEMA V2 MEDICATION ACTION MODEL
# --------------------------------------------------

import copy
from jsonschema import Draft202012Validator

clinical_schema_v2 = copy.deepcopy(clinical_schema_v2)

# Allowed medication actions
medication_action_values = [
    "start",
    "continue",
    "increase",
    "decrease",
    "refill",
    "stop",
    "recommend"
]


# --------------------------------------------------
# 1. MEDICATIONS
# Allow more than one action:
# ["continue", "refill"]
# --------------------------------------------------

clinical_schema_v2[
    "properties"
][
    "medications"
][
    "items"
][
    "properties"
][
    "action"
] = {
    "type": "array",
    "items": {
        "type": "string",
        "enum": medication_action_values
    },
    "uniqueItems": True
}


# --------------------------------------------------
# 2. PLAN -> MEDICATION CHANGES
# Allow more than one action here too
# --------------------------------------------------

clinical_schema_v2[
    "properties"
][
    "plan"
][
    "properties"
][
    "medication_changes"
][
    "items"
][
    "properties"
][
    "action"
] = {
    "type": "array",
    "items": {
        "type": "string",
        "enum": medication_action_values
    },
    "uniqueItems": True
}


# --------------------------------------------------
# 3. CHECK THAT SCHEMA V2 ITSELF IS VALID
# --------------------------------------------------

Draft202012Validator.check_schema(
    clinical_schema_v2
)

print("=" * 70)
print("CLINICAL SCHEMA V2 MEDICATION MODEL UPDATED")
print("=" * 70)

print(
    "Medication action schema:",
    clinical_schema_v2[
        "properties"
    ][
        "medications"
    ][
        "items"
    ][
        "properties"
    ][
        "action"
    ]
)

print(
    "\n✅ clinical_schema_v2 is a valid JSON Schema"
)

CLINICAL SCHEMA V2 MEDICATION MODEL UPDATED
Medication action schema: {'type': 'array', 'items': {'type': 'string', 'enum': ['start', 'continue', 'increase', 'decrease', 'refill', 'stop', 'recommend']}, 'uniqueItems': True}

✅ clinical_schema_v2 is a valid JSON Schema


In [ ]:
# --------------------------------------------------
# CASE 1 - MIGRATE GOLD FROM SCHEMA V1 TO V2
# --------------------------------------------------

import copy
import json

gold_annotation_case1_v2 = copy.deepcopy(
    gold_annotation_case1
)

gold1 = gold_annotation_case1_v2["gold_output"]


# --------------------------------------------------
# 1. MEDICAL HISTORY
# Add history_type
# --------------------------------------------------

for item in gold1["medical_history"]:
    item["history_type"] = "medical"


# --------------------------------------------------
# 2. MEDICATIONS
# Add status and convert action -> list
# --------------------------------------------------

for med in gold1["medications"]:

    old_action = med["action"]

    # Metformin increase
    if old_action == "increase":
        med["status"] = "current"
        med["action"] = ["increase"]

    # Lisinopril continuation
    elif old_action == "continue":
        med["status"] = "current"
        med["action"] = ["continue"]

    # Recommended medications
    elif old_action == "recommend":
        med["status"] = "recommended"
        med["action"] = ["recommend"]

    elif old_action == "recommend as needed for fever":
        med["status"] = "recommended"
        med["action"] = ["recommend"]

    else:
        print(
            "⚠️ Unrecognized medication action:",
            med["name"],
            old_action
        )


# --------------------------------------------------
# 3. PLAN - MEDICATION CHANGES
# Convert strings -> structured objects
# --------------------------------------------------

gold1["plan"]["medication_changes"] = [

    {
        "medication": "metformin",
        "action": ["increase"],
        "details": "increase to 1000 mg twice daily"
    },

    {
        "medication": "lisinopril",
        "action": ["continue"],
        "details": "continue 20 mg once daily"
    },

    {
        "medication": "Robitussin",
        "action": ["recommend"],
        "details": None
    },

    {
        "medication": "ibuprofen or Tylenol",
        "action": ["recommend"],
        "details": "use as needed if fever develops"
    }
]


# --------------------------------------------------
# 4. PLAN - SPECIALIST FOLLOW-UP
# Case 1 has no specialist referral/follow-up
# --------------------------------------------------

gold1["plan"].pop(
    "referrals",
    None
)

gold1["plan"]["specialist_follow_up"] = []


# --------------------------------------------------
# 5. FOLLOW-UP
# Add timing_text
# --------------------------------------------------

gold1["follow_up"]["timing_text"] = "about 4 months"


# --------------------------------------------------
# 6. MARK AS V2
# --------------------------------------------------

gold_annotation_case1_v2[
    "schema_version"
] = "v2"

gold_annotation_case1_v2[
    "review_status"
] = "manually_migrated_to_v2"


print("=" * 80)
print("CASE 1 GOLD - SCHEMA V2")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case1_v2["gold_output"],
        indent=2
    )
)

CASE 1 GOLD - SCHEMA V2
{
  "patient": {
    "age": 59,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "upper respiratory infection",
    "chief_complaint": "upper respiratory infection"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "hypertension",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "fatigue",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "about one week"
    },
    {
      "name": "shortness of breath",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "about one week"
    },
    {
      "name": "elbow pain",
      "status": "present",
      "body_site": "bilateral elbows",
      "severity": nu

In [ ]:
# --------------------------------------------------
# CASE 1 - FINALIZE + VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. ADD MISSING EXPLICIT FEVER DENIAL
# --------------------------------------------------

fever_already_present = any(
    item.get("name", "").lower() == "fever"
    for item in gold_annotation_case1_v2[
        "gold_output"
    ]["symptoms"]
)

if not fever_already_present:

    gold_annotation_case1_v2[
        "gold_output"
    ]["symptoms"].append(
        {
            "name": "fever",
            "status": "denied",
            "body_site": None,
            "severity": None,
            "duration": None
        }
    )

    print("✅ Added explicit fever denial to Case 1 gold")

else:
    print("ℹ️ Fever already exists in Case 1 gold")


# --------------------------------------------------
# 2. VALIDATE AGAINST SCHEMA V2
# --------------------------------------------------

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case1_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case1_v2[
            "gold_output"
        ]
    ),
    key=lambda e: list(e.absolute_path)
)


print("\n" + "=" * 80)
print("CASE 1 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)


if not case1_v2_errors:

    print(
        "✅ Case 1 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 1 gold has "
        f"{len(case1_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case1_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

✅ Added explicit fever denial to Case 1 gold

CASE 1 GOLD V2 SCHEMA VALIDATION
✅ Case 1 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# CASE 1 - VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case1_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case1_v2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 1 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)

if not case1_v2_errors:

    print(
        "✅ Case 1 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 1 gold has "
        f"{len(case1_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case1_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

CASE 1 GOLD V2 SCHEMA VALIDATION
✅ Case 1 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# CASE 2 - MIGRATE GOLD FROM SCHEMA V1 TO V2
# --------------------------------------------------

import copy
import json

gold_annotation_case2_v2 = copy.deepcopy(
    gold_annotation_case2
)

gold2 = gold_annotation_case2_v2["gold_output"]


# --------------------------------------------------
# 1. MEDICAL HISTORY
# --------------------------------------------------

for item in gold2["medical_history"]:
    item["history_type"] = "medical"


# --------------------------------------------------
# 2. MEDICATIONS
# Add status + convert action to action list
# --------------------------------------------------

gold2["medications"] = [

    {
        "name": "methotrexate",
        "dosage": "2.5 mg",
        "frequency": "once weekly",
        "status": "current",
        "action": [
            "continue",
            "refill"
        ]
    },

    {
        "name": "Protonix",
        "dosage": "40 mg",
        "frequency": "once daily",
        "status": "current",
        "action": [
            "continue"
        ]
    }
]


# --------------------------------------------------
# 3. PLAN - STRUCTURED MEDICATION CHANGES
# --------------------------------------------------

gold2["plan"]["medication_changes"] = [

    {
        "medication": "methotrexate",
        "action": [
            "continue",
            "refill"
        ],
        "details": "2.5 mg once weekly"
    },

    {
        "medication": "Protonix",
        "action": [
            "continue"
        ],
        "details": "40 mg once daily"
    }
]


# --------------------------------------------------
# 4. PLAN - SPECIALIST FOLLOW-UP
#
# This is a NEW cardiology referral,
# not continuing care with an existing specialist.
# --------------------------------------------------

gold2["plan"].pop(
    "referrals",
    None
)

gold2["plan"]["specialist_follow_up"] = [

    {
        "specialty": "cardiology",
        "action": "new_referral",
        "reason": "evaluation for cardiac ablation"
    }
]


# --------------------------------------------------
# 5. FOLLOW-UP
#
# No explicit follow-up visit timing was stated.
# --------------------------------------------------

gold2["follow_up"]["timing_text"] = None


# --------------------------------------------------
# 6. MARK AS SCHEMA V2
# --------------------------------------------------

gold_annotation_case2_v2[
    "schema_version"
] = "v2"

gold_annotation_case2_v2[
    "review_status"
] = "manually_migrated_to_v2"


print("=" * 80)
print("CASE 2 GOLD - SCHEMA V2")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case2_v2["gold_output"],
        indent=2
    )
)


CASE 2 GOLD - SCHEMA V2
{
  "patient": {
    "age": 52,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    {
      "condition": "rheumatoid arthritis",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "atrial fibrillation",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "reflux",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "joint pain",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": "over the last year"
    },
    {
      "name": "joint stiffness",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": "over the last year"
    },
    {
      "name": "palpitations",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "last e

In [ ]:
# --------------------------------------------------
# CASE 2 - VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case2_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case2_v2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 2 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)

if not case2_v2_errors:

    print(
        "✅ Case 2 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 2 gold has "
        f"{len(case2_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case2_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

CASE 2 GOLD V2 SCHEMA VALIDATION
✅ Case 2 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# CASE 3 - MIGRATE GOLD FROM SCHEMA V1 TO V2
# --------------------------------------------------

import copy
import json

gold_annotation_case3_v2 = copy.deepcopy(
    gold_annotation_case3
)

gold3 = gold_annotation_case3_v2["gold_output"]


# --------------------------------------------------
# 1. MEDICAL HISTORY
# Add history_type
# --------------------------------------------------

for item in gold3["medical_history"]:
    item["history_type"] = "medical"


# --------------------------------------------------
# 2. MEDICATIONS
# Add status + convert action to action list
# --------------------------------------------------

gold3["medications"] = [

    {
        "name": "Lantus",
        "dosage": "20 units",
        "frequency": "at night",
        "status": "current",
        "action": [
            "increase"
        ]
    },

    {
        "name": "immunosuppression medications",
        "dosage": None,
        "frequency": None,
        "status": "current",
        "action": [
            "continue"
        ]
    }
]


# --------------------------------------------------
# 3. PLAN - STRUCTURED MEDICATION CHANGES
# --------------------------------------------------

gold3["plan"]["medication_changes"] = [

    {
        "medication": "Lantus",
        "action": [
            "increase"
        ],
        "details": "increase to 20 units at night"
    }
]


# --------------------------------------------------
# 4. PLAN - SPECIALIST FOLLOW-UP
#
# Dr. Reyes is already managing the transplant,
# so this is continuing specialist follow-up,
# not a new referral.
# --------------------------------------------------

gold3["plan"].pop(
    "referrals",
    None
)

gold3["plan"]["specialist_follow_up"] = [

    {
        "specialty": "transplant specialist",
        "action": "continue_follow_up",
        "reason": "management of immunosuppression medications"
    }
]


# --------------------------------------------------
# 5. FOLLOW-UP
#
# No explicit office follow-up interval was stated.
# "A1c in a couple months" is test timing,
# not a scheduled visit.
# --------------------------------------------------

gold3["follow_up"]["timing_text"] = None


# --------------------------------------------------
# 6. MARK AS SCHEMA V2
# --------------------------------------------------

gold_annotation_case3_v2[
    "schema_version"
] = "v2"

gold_annotation_case3_v2[
    "review_status"
] = "manually_migrated_to_v2"


print("=" * 80)
print("CASE 3 GOLD - SCHEMA V2")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case3_v2["gold_output"],
        indent=2
    )
)

CASE 3 GOLD - SCHEMA V2
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "emergency room follow-up",
    "chief_complaint": "emergency room follow-up"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "kidney transplant",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "wooziness",
      "status": "resolved",
      "body_site": null,
      "severity": null,
      "duration": "over the weekend"
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": null,
      "duration": null
    },
    {
      "name": "shortness of breath",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration":

In [ ]:
# --------------------------------------------------
# CASE 3 - VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case3_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case3_v2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 3 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)

if not case3_v2_errors:

    print(
        "✅ Case 3 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 3 gold has "
        f"{len(case3_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case3_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

CASE 3 GOLD V2 SCHEMA VALIDATION
✅ Case 3 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# CASE 4 - MIGRATE GOLD FROM SCHEMA V1 TO V2
# --------------------------------------------------

import copy
import json

gold_annotation_case4_v2 = copy.deepcopy(
    gold_annotation_case4
)

gold4 = gold_annotation_case4_v2["gold_output"]


# --------------------------------------------------
# 1. MEDICAL HISTORY
# Add history_type
# --------------------------------------------------

for item in gold4["medical_history"]:
    item["history_type"] = "medical"


# --------------------------------------------------
# 2. MEDICATIONS
# Add status + convert action to action list
# --------------------------------------------------

gold4["medications"] = [

    {
        "name": "Fosamax",
        "dosage": "1 tablet",
        "frequency": "once weekly",
        "status": "current",
        "action": [
            "continue",
            "refill"
        ]
    },

    {
        "name": "multiple sclerosis medications",
        "dosage": None,
        "frequency": None,
        "status": "current",
        "action": [
            "continue"
        ]
    }
]


# --------------------------------------------------
# 3. PLAN - STRUCTURED MEDICATION CHANGES
# --------------------------------------------------

gold4["plan"]["medication_changes"] = [

    {
        "medication": "Fosamax",
        "action": [
            "continue",
            "refill"
        ],
        "details": "1 tablet once weekly"
    },

    {
        "medication": "multiple sclerosis medications",
        "action": [
            "continue"
        ],
        "details": None
    }
]


# --------------------------------------------------
# 4. PLAN - SPECIALIST FOLLOW-UP
#
# Neurology care already exists,
# so this is continuing follow-up.
# --------------------------------------------------

gold4["plan"].pop(
    "referrals",
    None
)

gold4["plan"]["specialist_follow_up"] = [

    {
        "specialty": "neurology",
        "action": "continue_follow_up",
        "reason": "management of multiple sclerosis"
    }
]


# --------------------------------------------------
# 5. FOLLOW-UP
# No explicit visit interval stated
# --------------------------------------------------

gold4["follow_up"]["timing_text"] = None


# --------------------------------------------------
# 6. MARK AS SCHEMA V2
# --------------------------------------------------

gold_annotation_case4_v2[
    "schema_version"
] = "v2"

gold_annotation_case4_v2[
    "review_status"
] = "manually_migrated_to_v2"


print("=" * 80)
print("CASE 4 GOLD - SCHEMA V2")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case4_v2["gold_output"],
        indent=2
    )
)

CASE 4 GOLD - SCHEMA V2
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    {
      "condition": "osteoporosis",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "multiple sclerosis",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "insomnia",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": null
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": null,
      "duration": null
    },
    {
      "name": "shortness of breath",
      "status": "denied",
      "body_site": null,
      "severity": null,
      "duration": null
    }
  ],
  "medications": [
    {
      "name": "Fosamax",
      "dosage": "1 tablet",
      "frequency": "once weekly",
      "status": "cur

In [ ]:
# --------------------------------------------------
# CASE 4 - REMOVE ACCIDENTAL DUPLICATE EXAM ROW
# --------------------------------------------------

gold4 = gold_annotation_case4_v2["gold_output"]

gold4["physical_exam"] = [
    item
    for item in gold4["physical_exam"]
    if not (
        item.get("finding") == "reflexes good"
        and item.get("body_site") == "ity"
    )
]

print("=" * 80)
print("CASE 4 PHYSICAL EXAM AFTER CORRECTION")
print("=" * 80)

for item in gold4["physical_exam"]:
    print(item)

CASE 4 PHYSICAL EXAM AFTER CORRECTION
{'system': 'respiratory', 'finding': 'lungs clear', 'body_site': 'lungs'}
{'system': 'cardiovascular', 'finding': 'heart sounds normal', 'body_site': 'heart'}
{'system': 'neurological', 'finding': 'strength 4/5', 'body_site': 'right lower extremity'}
{'system': 'neurological', 'finding': 'strength 3/5', 'body_site': 'left lower extremity'}
{'system': 'neurological', 'finding': 'reflexes good', 'body_site': 'lower extremities'}
{'system': 'musculoskeletal', 'finding': 'arthritic changes', 'body_site': 'right knee'}


In [ ]:
# --------------------------------------------------
# CASE 4 - VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case4_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case4_v2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 4 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)

if not case4_v2_errors:

    print(
        "✅ Case 4 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 4 gold has "
        f"{len(case4_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case4_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

CASE 4 GOLD V2 SCHEMA VALIDATION
✅ Case 4 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# CASE 5 - MIGRATE GOLD FROM SCHEMA V1 TO V2
# --------------------------------------------------

import copy
import json

gold_annotation_case5_v2 = copy.deepcopy(
    gold_annotation_case5
)

gold5 = gold_annotation_case5_v2["gold_output"]


# --------------------------------------------------
# 1. MEDICAL HISTORY
# Add history_type
# --------------------------------------------------

for item in gold5["medical_history"]:
    item["history_type"] = "medical"


# --------------------------------------------------
# 2. MEDICATIONS
# Add status + convert action to list
# --------------------------------------------------

gold5["medications"] = [

    {
        "name": "Flonase",
        "dosage": None,
        "frequency": None,
        "status": "current",
        "action": [
            "continue"
        ]
    },

    {
        "name": "Motrin",
        "dosage": "800 mg",
        "frequency": "three times daily with food",
        "status": "recommended",
        "action": [
            "start"
        ]
    }
]


# --------------------------------------------------
# 3. PLAN - STRUCTURED MEDICATION CHANGES
# --------------------------------------------------

gold5["plan"]["medication_changes"] = [

    {
        "medication": "Motrin",
        "action": [
            "start"
        ],
        "details": "800 mg three times daily with food"
    }
]


# --------------------------------------------------
# 4. PLAN - SPECIALIST FOLLOW-UP
# No specialist referral/follow-up stated
# --------------------------------------------------

gold5["plan"].pop(
    "referrals",
    None
)

gold5["plan"]["specialist_follow_up"] = []


# --------------------------------------------------
# 5. FOLLOW-UP
# Preserve conditional timing
# --------------------------------------------------

gold5["follow_up"]["timing_text"] = (
    "about one week if symptoms do not improve"
)


# --------------------------------------------------
# 6. MARK AS SCHEMA V2
# --------------------------------------------------

gold_annotation_case5_v2[
    "schema_version"
] = "v2"

gold_annotation_case5_v2[
    "review_status"
] = "manually_migrated_to_v2"


print("=" * 80)
print("CASE 5 GOLD - SCHEMA V2")
print("=" * 80)

print(
    json.dumps(
        gold_annotation_case5_v2["gold_output"],
        indent=2
    )
)

CASE 5 GOLD - SCHEMA V2
{
  "patient": {
    "age": 43,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "right arm pain",
    "chief_complaint": "right arm pain"
  },
  "medical_history": [
    {
      "condition": "allergies",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "pain",
      "status": "present",
      "body_site": "right distal radius",
      "severity": null,
      "duration": null
    },
    {
      "name": "swelling",
      "status": "present",
      "body_site": "right distal radius",
      "severity": "mild",
      "duration": null
    },
    {
      "name": "pain with wrist movement",
      "status": "present",
      "body_site": "right wrist",
      "severity": "mild",
      "duration": null
    },
    {
      "name": "numbness",
      "status": "denied",
      "body_site": "right hand",
      "severity": null,
      "duration": null
    }
  ],
  "medications": [
    {
      "name": "Flonase",


In [ ]:
# --------------------------------------------------
# CASE 5 - VALIDATE GOLD AGAINST SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

case5_v2_errors = sorted(
    validator_v2.iter_errors(
        gold_annotation_case5_v2["gold_output"]
    ),
    key=lambda e: list(e.absolute_path)
)

print("=" * 80)
print("CASE 5 GOLD V2 SCHEMA VALIDATION")
print("=" * 80)

if not case5_v2_errors:

    print(
        "✅ Case 5 gold is valid against clinical_schema_v2"
    )

else:

    print(
        f"❌ Case 5 gold has "
        f"{len(case5_v2_errors)} schema errors\n"
    )

    for i, error in enumerate(
        case5_v2_errors,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

CASE 5 GOLD V2 SCHEMA VALIDATION
✅ Case 5 gold is valid against clinical_schema_v2


In [ ]:
# --------------------------------------------------
# FINAL PRE-FREEZE CHECK - ALL 5 GOLD ANNOTATIONS V2
# --------------------------------------------------

from jsonschema import Draft202012Validator

validator_v2 = Draft202012Validator(
    clinical_schema_v2
)

gold_annotations_v2 = {
    "Case 1": gold_annotation_case1_v2,
    "Case 2": gold_annotation_case2_v2,
    "Case 3": gold_annotation_case3_v2,
    "Case 4": gold_annotation_case4_v2,
    "Case 5": gold_annotation_case5_v2
}

print("=" * 80)
print("FINAL V2 GOLD VALIDATION")
print("=" * 80)

all_valid = True

for case_name, annotation in gold_annotations_v2.items():

    errors = list(
        validator_v2.iter_errors(
            annotation["gold_output"]
        )
    )

    if not errors:
        print(f"✅ {case_name}: valid")
    else:
        all_valid = False
        print(
            f"❌ {case_name}: "
            f"{len(errors)} schema errors"
        )


print("\n" + "=" * 80)

if all_valid:
    print(
        "✅ ALL 5 GOLD ANNOTATIONS ARE VALID "
        "AGAINST CLINICAL SCHEMA V2"
    )
else:
    print(
        "❌ Do not freeze Schema V2 yet."
    )


# --------------------------------------------------
# SEMANTIC SANITY CHECK
# Case 1 transcript explicitly denies fever.
# Make sure the corrected gold contains it.
# --------------------------------------------------

case1_has_fever_denial = any(
    symptom.get("name", "").lower() == "fever"
    and symptom.get("status") == "denied"
    for symptom in gold_annotation_case1_v2[
        "gold_output"
    ]["symptoms"]
)

print("\nCASE 1 FEVER DENIAL CHECK:")

if case1_has_fever_denial:
    print("✅ Explicit fever denial is present")
else:
    print("⚠️ Explicit fever denial is still missing")

FINAL V2 GOLD VALIDATION
✅ Case 1: valid
✅ Case 2: valid
✅ Case 3: valid
✅ Case 4: valid
✅ Case 5: valid

✅ ALL 5 GOLD ANNOTATIONS ARE VALID AGAINST CLINICAL SCHEMA V2

CASE 1 FEVER DENIAL CHECK:
✅ Explicit fever denial is present


In [ ]:
# --------------------------------------------------
# FREEZE SCHEMA V2 + GOLD ANNOTATIONS V2
# --------------------------------------------------

import copy
import json
import hashlib


# --------------------------------------------------
# 1. FREEZE SCHEMA V2
# --------------------------------------------------

clinical_schema_v2_frozen = copy.deepcopy(
    clinical_schema_v2
)


# --------------------------------------------------
# 2. FREEZE ALL 5 GOLD ANNOTATIONS V2
# --------------------------------------------------

gold_annotations_v2_frozen = copy.deepcopy(
    {
        "Case 1": gold_annotation_case1_v2,
        "Case 2": gold_annotation_case2_v2,
        "Case 3": gold_annotation_case3_v2,
        "Case 4": gold_annotation_case4_v2,
        "Case 5": gold_annotation_case5_v2
    }
)


# --------------------------------------------------
# 3. CREATE REPRODUCIBILITY HASHES
# --------------------------------------------------

schema_v2_json = json.dumps(
    clinical_schema_v2_frozen,
    sort_keys=True,
    separators=(",", ":")
)

gold_v2_json = json.dumps(
    gold_annotations_v2_frozen,
    sort_keys=True,
    separators=(",", ":")
)

schema_v2_hash = hashlib.sha256(
    schema_v2_json.encode("utf-8")
).hexdigest()

gold_v2_hash = hashlib.sha256(
    gold_v2_json.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 4. PRINT FREEZE STATUS
# --------------------------------------------------

print("=" * 80)
print("SCHEMA V2 + GOLD V2 FROZEN")
print("=" * 80)

print("Schema version: v2")
print("Number of frozen gold cases:", len(gold_annotations_v2_frozen))

print("\nSchema V2 SHA256:")
print(schema_v2_hash)

print("\nGold annotations V2 SHA256:")
print(gold_v2_hash)

print("\n✅ Schema V2 is frozen")
print("✅ Gold annotations V2 are frozen")
print("✅ Ready for schema-constrained experiment")

SCHEMA V2 + GOLD V2 FROZEN
Schema version: v2
Number of frozen gold cases: 5

Schema V2 SHA256:
85f81bdac1c99e6cc34e2e496b6a1a7f2405025111187d81c928ccf3918cbfda

Gold annotations V2 SHA256:
d94198598643a695796447a9de472bc6adcd7f406af1aa66ee84c717c9f50d81

✅ Schema V2 is frozen
✅ Gold annotations V2 are frozen
✅ Ready for schema-constrained experiment


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 1
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 1
# --------------------------------------------------

schema_case_index = 0

schema_case1 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case1["case_id"]
)


# --------------------------------------------------
# 2. CREATE SCHEMA-CONSTRAINED PROMPT
#
# Important:
# The transcript remains the source of truth.
# Do not use the gold annotation in the prompt.
# --------------------------------------------------

schema_constrained_prompt_case1 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case1["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case1 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case1,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case1 = (
    schema_response_case1.output_text
)

schema_raw_case1_hash = hashlib.sha256(
    schema_raw_case1.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case1 = json.loads(
        schema_raw_case1
    )

    schema_json_valid_case1 = 1

except json.JSONDecodeError:

    schema_output_case1 = None
    schema_json_valid_case1 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 1")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case1
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case1_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case1
)

Case ID: D2N088-virtassist

EXPERIMENT B - CASE 1
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
6f35e5a14c8fa4562b7f80949b8de2fd2e5a7ac923e0d01e0e6952500fe51572

MODEL OUTPUT:
{
  "patient": {
    "age": 59,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "upper respiratory infection",
    "chief_complaint": "upper respiratory infection"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "type two diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "hypertension",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "seasonal allergies",
      "status": "denied",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "shortness of breath",
      "status": "present",
      "body_site": "chest",
      "severity": null,
      "duration": "last wee

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 1
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. CREATE VALIDATOR USING FROZEN SCHEMA V2
# --------------------------------------------------

validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE THE EXACT FROZEN MODEL OUTPUT
# --------------------------------------------------

schema_errors_case1 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case1
    ),
    key=lambda e: list(e.absolute_path)
)


# --------------------------------------------------
# 3. STRUCTURAL METRICS
# --------------------------------------------------

schema_valid_case1 = (
    1 if len(schema_errors_case1) == 0 else 0
)

schema_error_count_case1 = len(
    schema_errors_case1
)


# Count error types
required_errors_case1 = sum(
    error.validator == "required"
    for error in schema_errors_case1
)

additional_property_errors_case1 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case1
)

type_errors_case1 = sum(
    error.validator == "type"
    for error in schema_errors_case1
)


# --------------------------------------------------
# 4. PRINT RESULTS
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 1 STRUCTURAL VALIDATION")
print("=" * 80)

print(
    "JSON valid:",
    schema_json_valid_case1
)

print(
    "Schema valid:",
    schema_valid_case1
)

print(
    "Total schema errors:",
    schema_error_count_case1
)

print(
    "Required-field errors:",
    required_errors_case1
)

print(
    "Additional-property errors:",
    additional_property_errors_case1
)

print(
    "Type errors:",
    type_errors_case1
)


# --------------------------------------------------
# 5. PRINT ERRORS IF ANY
# --------------------------------------------------

if schema_errors_case1:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case1,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case1_hash)

EXPERIMENT B - CASE 1 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
6f35e5a14c8fa4562b7f80949b8de2fd2e5a7ac923e0d01e0e6952500fe51572


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 1
# PREPARE SEMANTIC REVIEW
# --------------------------------------------------

import json

case1_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 1"
    ]["gold_output"]
)

case1_model_v2 = schema_output_case1

case1_transcript = research_cases[0][
    "transcript"
]


print("=" * 80)
print("EXPERIMENT B - CASE 1 SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 1. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(case1_transcript)


# --------------------------------------------------
# 2. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        case1_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 3. SCHEMA-CONSTRAINED MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        case1_model_v2,
        indent=2
    )
)


# --------------------------------------------------
# 4. CONFIRM EXACT OUTPUT HASH
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN MODEL OUTPUT HASH")
print("=" * 80)

print(schema_raw_case1_hash)

EXPERIMENT B - CASE 1 SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yeah , both .
[doctor] okay . all right . and , um , do you have any history of any seasonal allergies a

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 1
# MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


case1_constrained_semantic_review = [

    # --------------------------------------------------
    # ACCEPTABLE / SUPPORTED EXTRA INFORMATION
    # These are NOT hallucinations.
    # --------------------------------------------------

    {
        "field": "medical_history.seasonal_allergies",
        "model_value": "seasonal allergies - denied",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "Transcript explicitly states the patient has no "
            "history of seasonal allergies."
    },

    {
        "field": "physical_exam.vital_signs",
        "model_value": "normal, no fever",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "Clinician explicitly states vital signs are normal "
            "and the patient does not have a fever."
    },

    {
        "field": "physical_exam.heart",
        "model_value": "heart sounds nice and strong",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The physical examination explicitly describes "
            "heart sounds as nice and strong."
    },

    {
        "field": "assessment.pneumonia",
        "model_value": "pneumonia - ruled_out",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "Chest X-ray explicitly states there is no pneumonia."
    },

    {
        "field": "medications.lisinopril.action",
        "model_value": ["continue", "refill"],
        "gold_value": ["continue"],
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly says a lisinopril refill "
            "is needed and the clinician orders it."
    },


    # --------------------------------------------------
    # SEMANTIC ERRORS
    # --------------------------------------------------

    {
        "field": "symptoms.coughing.status",
        "model_value": "denied",
        "gold_value": None,
        "category": "status_error",
        "count_as_error": 1,
        "reason":
            "The patient denied coughing anything up, not "
            "coughing itself. Later the clinician explicitly "
            "refers to the patient's cough and recommends "
            "Robitussin."
    },

    {
        "field": "assessment.viral_syndrome.certainty",
        "model_value": "confirmed",
        "gold_value": "suspected",
        "category": "status_error",
        "count_as_error": 1,
        "reason":
            "The clinician says 'I believe you have a viral "
            "syndrome' and continues diagnostic testing, so "
            "confirmed overstates the certainty represented "
            "by the frozen gold."
    },

    {
        "field": "plan.tests_ordered.hemoglobin_A1c",
        "model_value": "hemoglobin A1c",
        "gold_value": "hemoglobin A1c in four months",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The test itself is correctly extracted, but the "
            "explicit four-month timing is omitted."
    },


    # --------------------------------------------------
    # MAPPING AMBIGUITY
    # Keep separate from confirmed errors during pilot.
    # --------------------------------------------------

    {
        "field": "physical_exam.lower_extremity_edema.system",
        "model_value": "cardiovascular",
        "gold_value": "musculoskeletal",
        "category": "mapping_ambiguity",
        "count_as_error": 0,
        "reason":
            "The finding is correctly extracted, but assigning "
            "lower-extremity edema to cardiovascular rather "
            "than musculoskeletal is clinically plausible. "
            "This should not be counted as a definite mapping "
            "error until the annotation rule is explicitly frozen."
    },

    {
        "field": "encounter.chief_complaint",
        "model_value": "shortness of breath and fatigue",
        "gold_value": "upper respiratory infection",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "Both are supported by the transcript. URI is the "
            "clinician's stated visit framing, while shortness "
            "of breath and fatigue are the patient's presenting "
            "symptoms."
    }
]


case1_constrained_review_df = pd.DataFrame(
    case1_constrained_semantic_review
)


print("=" * 80)
print("EXPERIMENT B - CASE 1 SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    case1_constrained_review_df
)


print("\nCONFIRMED SEMANTIC ERRORS:")

confirmed_errors = case1_constrained_review_df[
    case1_constrained_review_df[
        "count_as_error"
    ] == 1
]

print(
    confirmed_errors[
        "category"
    ].value_counts()
)

print(
    "\nTotal confirmed semantic errors:",
    len(confirmed_errors)
)

EXPERIMENT B - CASE 1 SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,medical_history.seasonal_allergies,seasonal allergies - denied,None,supported_extra,0,Transcript explicitly states the patient has n...
1,physical_exam.vital_signs,"normal, no fever",None,supported_extra,0,Clinician explicitly states vital signs are no...
2,physical_exam.heart,heart sounds nice and strong,None,supported_extra,0,The physical examination explicitly describes ...
3,assessment.pneumonia,pneumonia - ruled_out,None,supported_extra,0,Chest X-ray explicitly states there is no pneu...
4,medications.lisinopril.action,"[continue, refill]",[continue],supported_extra,0,The patient explicitly says a lisinopril refil...
5,symptoms.coughing.status,denied,None,status_error,1,"The patient denied coughing anything up, not c..."
6,assessment.viral_syndrome.certainty,confirmed,suspected,status_error,1,The clinician says 'I believe you have a viral...
7,plan.tests_ordered.hemoglobin_A1c,hemoglobin A1c,hemoglobin A1c in four months,partial_extraction,1,"The test itself is correctly extracted, but th..."
8,physical_exam.lower_extremity_edema.system,cardiovascular,musculoskeletal,mapping_ambiguity,0,"The finding is correctly extracted, but assign..."
9,encounter.chief_complaint,shortness of breath and fatigue,upper respiratory infection,annotation_ambiguity,0,Both are supported by the transcript. URI is t...



CONFIRMED SEMANTIC ERRORS:
category
status_error          2
partial_extraction    1
Name: count, dtype: int64

Total confirmed semantic errors: 3


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 1
# SAVE FINAL PILOT RESULT
# --------------------------------------------------

case1_constrained_result = {

    "case_id": "D2N088-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "schema_constrained",

    # --------------------------------------------------
    # STRUCTURAL
    # --------------------------------------------------

    "json_valid": schema_json_valid_case1,

    "schema_valid": schema_valid_case1,

    "schema_error_count": schema_error_count_case1,

    "required_field_errors": required_errors_case1,

    "additional_property_errors":
        additional_property_errors_case1,

    "type_errors": type_errors_case1,


    # --------------------------------------------------
    # MANUALLY CONFIRMED SEMANTIC ISSUES
    # --------------------------------------------------

    "confirmed_status_errors": 2,

    "confirmed_partial_extractions": 1,

    "confirmed_mapping_errors": 0,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 0,

    "confirmed_semantic_errors": 3,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 5,

    "mapping_ambiguities": 1,

    "annotation_ambiguities": 1,


    # --------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------

    "output_sha256":
        schema_raw_case1_hash
}


print("=" * 80)
print("EXPERIMENT B - CASE 1 FINAL RESULT")
print("=" * 80)

for key, value in case1_constrained_result.items():
    print(f"{key}: {value}")

EXPERIMENT B - CASE 1 FINAL RESULT
case_id: D2N088-virtassist
model: gemini-3.6-flash
condition: schema_constrained
json_valid: 1
schema_valid: 1
schema_error_count: 0
required_field_errors: 0
additional_property_errors: 0
type_errors: 0
confirmed_status_errors: 2
confirmed_partial_extractions: 1
confirmed_mapping_errors: 0
confirmed_unsupported_inferences: 0
confirmed_omissions: 0
confirmed_semantic_errors: 3
supported_extra_items: 5
mapping_ambiguities: 1
annotation_ambiguities: 1
output_sha256: 6f35e5a14c8fa4562b7f80949b8de2fd2e5a7ac923e0d01e0e6952500fe51572


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 2
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 2
# --------------------------------------------------

schema_case_index = 1

schema_case2 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case2["case_id"]
)


# --------------------------------------------------
# 2. CREATE THE SAME SCHEMA-CONSTRAINED PROMPT
#
# Only the transcript changes.
# Gold annotation is NOT shown to Gemini.
# --------------------------------------------------

schema_constrained_prompt_case2 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case2["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case2 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case2,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case2 = (
    schema_response_case2.output_text
)

schema_raw_case2_hash = hashlib.sha256(
    schema_raw_case2.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case2 = json.loads(
        schema_raw_case2
    )

    schema_json_valid_case2 = 1

except json.JSONDecodeError:

    schema_output_case2 = None
    schema_json_valid_case2 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 2")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case2
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case2_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case2
)

Case ID: D2N089-virtassist

EXPERIMENT B - CASE 2
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
d41c6fe6087dec61c09fe9ba136213bd64175a9b6ad09783547edc5260df500b

MODEL OUTPUT:
{
  "patient": {
    "age": 52,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "annual exam"
  },
  "medical_history": [
    {
      "condition": "rheumatoid arthritis",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "atrial fibrillation",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "reflux",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "joint pain",
      "status": "denied",
      "body_site": "joints",
      "severity": null,
      "duration": null
    },
    {
      "name": "stiffness",
      "status": "denied",
      "body_site": "joints",
      "severity": null,
      "duration": null
    },


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 2
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. VALIDATOR USING FROZEN SCHEMA V2
# --------------------------------------------------

validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE CASE 2 OUTPUT
# --------------------------------------------------

schema_errors_case2 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case2
    ),
    key=lambda e: list(e.absolute_path)
)


# --------------------------------------------------
# 3. STRUCTURAL METRICS
# --------------------------------------------------

schema_valid_case2 = (
    1 if len(schema_errors_case2) == 0 else 0
)

schema_error_count_case2 = len(
    schema_errors_case2
)

required_errors_case2 = sum(
    error.validator == "required"
    for error in schema_errors_case2
)

additional_property_errors_case2 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case2
)

type_errors_case2 = sum(
    error.validator == "type"
    for error in schema_errors_case2
)


# --------------------------------------------------
# 4. PRINT RESULTS
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 2 STRUCTURAL VALIDATION")
print("=" * 80)

print(
    "JSON valid:",
    schema_json_valid_case2
)

print(
    "Schema valid:",
    schema_valid_case2
)

print(
    "Total schema errors:",
    schema_error_count_case2
)

print(
    "Required-field errors:",
    required_errors_case2
)

print(
    "Additional-property errors:",
    additional_property_errors_case2
)

print(
    "Type errors:",
    type_errors_case2
)


# --------------------------------------------------
# 5. PRINT ERRORS IF ANY
# --------------------------------------------------

if schema_errors_case2:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case2,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case2_hash)

EXPERIMENT B - CASE 2 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
d41c6fe6087dec61c09fe9ba136213bd64175a9b6ad09783547edc5260df500b


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 2
# PREPARE SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. GET FROZEN GOLD, MODEL OUTPUT, TRANSCRIPT
# --------------------------------------------------

case2_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 2"
    ]["gold_output"]
)

case2_model_v2 = schema_output_case2

case2_transcript = research_cases[1][
    "transcript"
]


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 2 SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    case2_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        case2_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. SCHEMA-CONSTRAINED MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        case2_model_v2,
        indent=2
    )
)


# --------------------------------------------------
# 6. CONFIRM FROZEN HASH
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN MODEL OUTPUT HASH")
print("=" * 80)

print(
    schema_raw_case2_hash
)

EXPERIMENT B - CASE 2 SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year-old female with a past medical history significant for rheumatoid arthritis , atrial fibrillation , and reflux who presents today for her annual exam . so andrea , it's been a year since i saw you . how are you doing ?
[patient] i'm doing well . so , i've been walking like you told me to and , um , exercising and doing yoga , and that's actually helped with my arthritis a lot , just the- the constant movement . so , i have n't had any joint pain recently .
[doctor] okay . good . so , no- no issues with any stiffness or pain or flare ups over the last year ?
[patient] no .
[doctor] okay . and i know that we have you on the methotrexate , are you still taking that once a week ?
[patient] 

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 2
# MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


case2_constrained_semantic_review = [

    # --------------------------------------------------
    # PARTIAL EXTRACTIONS
    # --------------------------------------------------

    {
        "field": "symptoms.joint_pain.duration",
        "model_value": None,
        "gold_value": "over the last year",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The transcript explicitly asks about pain or "
            "flare-ups over the last year, and the patient "
            "denies them. The model captured the denial but "
            "omitted the duration."
    },

    {
        "field": "symptoms.joint_stiffness.duration",
        "model_value": None,
        "gold_value": "over the last year",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The transcript explicitly connects the denied "
            "stiffness to the last year, but the model leaves "
            "duration null."
    },


    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "symptoms.right_elbow_pain",
        "model_value": "pain - present - right elbow",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Right-elbow pain was elicited during the physical "
            "exam as pain to palpation. The model already "
            "captures that physical-exam finding, so adding it "
            "as a symptom places the same fact in the wrong section."
    },

    {
        "field": "plan.procedures",
        "model_value": ["cardiac ablation"],
        "gold_value": [],
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The clinician orders a cardiology referral for "
            "cardiac ablation. The ablation is the reason for "
            "the referral, not a procedure performed or directly "
            "ordered as a procedure in this encounter."
    },


    # --------------------------------------------------
    # OMISSION
    # --------------------------------------------------

    {
        "field": "plan.patient_instructions",
        "model_value":
            "Continue dietary modifications, including "
            "avoiding coffee and spicy foods.",
        "gold_value":
            "contact clinician if reflux symptoms or "
            "other issues recur",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly tells the patient to "
            "let them know if any other reflux-related issues "
            "occur, but this instruction is absent."
    },


    # --------------------------------------------------
    # AMBIGUITY - DO NOT COUNT AS ERROR
    # --------------------------------------------------

    {
        "field": "physical_exam.lower_extremity_edema.system",
        "model_value": "cardiovascular",
        "gold_value": "musculoskeletal",
        "category": "mapping_ambiguity",
        "count_as_error": 0,
        "reason":
            "The no-edema finding is correct. Cardiovascular "
            "and musculoskeletal are both plausible section "
            "assignments, so this should not be counted as a "
            "definite mapping error during the pilot."
    }
]


case2_constrained_review_df = pd.DataFrame(
    case2_constrained_semantic_review
)


print("=" * 80)
print("EXPERIMENT B - CASE 2 SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    case2_constrained_review_df
)


confirmed_errors_case2 = (
    case2_constrained_review_df[
        case2_constrained_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_errors_case2[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(confirmed_errors_case2)
)

EXPERIMENT B - CASE 2 SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,symptoms.joint_pain.duration,None,over the last year,partial_extraction,1,The transcript explicitly asks about pain or f...
1,symptoms.joint_stiffness.duration,None,over the last year,partial_extraction,1,The transcript explicitly connects the denied ...
2,symptoms.right_elbow_pain,pain - present - right elbow,None,mapping_error,1,Right-elbow pain was elicited during the physi...
3,plan.procedures,[cardiac ablation],[],mapping_error,1,The clinician orders a cardiology referral for...
4,plan.patient_instructions,"Continue dietary modifications, including avoi...",contact clinician if reflux symptoms or other ...,omission,1,The clinician explicitly tells the patient to ...
5,physical_exam.lower_extremity_edema.system,cardiovascular,musculoskeletal,mapping_ambiguity,0,The no-edema finding is correct. Cardiovascula...



CONFIRMED SEMANTIC ERRORS:
category
partial_extraction    2
mapping_error         2
omission              1
Name: count, dtype: int64

Total confirmed semantic errors: 5


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 2
# SAVE FINAL RESULT
# --------------------------------------------------

case2_constrained_result = {

    "case_id": "D2N089-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "schema_constrained",


    # --------------------------------------------------
    # STRUCTURAL
    # --------------------------------------------------

    "json_valid": schema_json_valid_case2,

    "schema_valid": schema_valid_case2,

    "schema_error_count": schema_error_count_case2,

    "required_field_errors": required_errors_case2,

    "additional_property_errors":
        additional_property_errors_case2,

    "type_errors": type_errors_case2,


    # --------------------------------------------------
    # CONFIRMED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 2,

    "confirmed_mapping_errors": 2,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 1,

    "confirmed_semantic_errors": 5,


    # --------------------------------------------------
    # AMBIGUITIES
    # --------------------------------------------------

    "supported_extra_items": 0,

    "mapping_ambiguities": 1,

    "annotation_ambiguities": 0,


    # --------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------

    "output_sha256":
        schema_raw_case2_hash
}


print("=" * 80)
print("EXPERIMENT B - CASE 2 FINAL RESULT")
print("=" * 80)

for key, value in case2_constrained_result.items():
    print(f"{key}: {value}")

EXPERIMENT B - CASE 2 FINAL RESULT
case_id: D2N089-virtassist
model: gemini-3.6-flash
condition: schema_constrained
json_valid: 1
schema_valid: 1
schema_error_count: 0
required_field_errors: 0
additional_property_errors: 0
type_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 2
confirmed_mapping_errors: 2
confirmed_unsupported_inferences: 0
confirmed_omissions: 1
confirmed_semantic_errors: 5
supported_extra_items: 0
mapping_ambiguities: 1
annotation_ambiguities: 0
output_sha256: d41c6fe6087dec61c09fe9ba136213bd64175a9b6ad09783547edc5260df500b


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 3
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 3
# --------------------------------------------------

schema_case_index = 2

schema_case3 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case3["case_id"]
)


# --------------------------------------------------
# 2. CREATE SAME SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

schema_constrained_prompt_case3 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case3["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case3 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case3,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case3 = (
    schema_response_case3.output_text
)

schema_raw_case3_hash = hashlib.sha256(
    schema_raw_case3.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case3 = json.loads(
        schema_raw_case3
    )

    schema_json_valid_case3 = 1

except json.JSONDecodeError:

    schema_output_case3 = None
    schema_json_valid_case3 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 3")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case3
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case3_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case3
)

Case ID: D2N090-virtassist

EXPERIMENT B - CASE 3
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
e0a622d019db7d1ac29b63d1d876da98fa351df11318e9d465b65850e6ef7e3e

MODEL OUTPUT:
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "emergency room follow-up",
    "chief_complaint": "emergency room follow-up"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "kidney transplant",
      "status": "historical",
      "history_type": "surgical"
    }
  ],
  "symptoms": [
    {
      "name": "woozy",
      "status": "resolved",
      "body_site": null,
      "severity": null,
      "duration": "over the weekend"
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": null

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 3
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 3
# --------------------------------------------------

schema_case_index = 2

schema_case3 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case3["case_id"]
)


# --------------------------------------------------
# 2. CREATE SAME SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

schema_constrained_prompt_case3 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case3["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case3 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case3,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case3 = (
    schema_response_case3.output_text
)

schema_raw_case3_hash = hashlib.sha256(
    schema_raw_case3.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case3 = json.loads(
        schema_raw_case3
    )

    schema_json_valid_case3 = 1

except json.JSONDecodeError:

    schema_output_case3 = None
    schema_json_valid_case3 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 3")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case3
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case3_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case3
)

Case ID: D2N090-virtassist

EXPERIMENT B - CASE 3
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
84628d313e4abd7b427df0fc85a3a94b1a0114081c3f4433ed36a468e976886d

MODEL OUTPUT:
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "emergency room follow-up",
    "chief_complaint": "emergency room follow-up"
  },
  "medical_history": [
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "type 2 diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "kidney transplant",
      "status": "current",
      "history_type": "surgical"
    }
  ],
  "symptoms": [
    {
      "name": "wooziness",
      "status": "resolved",
      "body_site": null,
      "severity": null,
      "duration": "over the weekend"
    },
    {
      "name": "chest pain",
      "status": "denied",
      "body_site": "chest",
      "severity": nul

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 3
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


schema_errors_case3 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case3
    ),
    key=lambda e: list(e.absolute_path)
)


schema_valid_case3 = (
    1 if len(schema_errors_case3) == 0 else 0
)

schema_error_count_case3 = len(
    schema_errors_case3
)

required_errors_case3 = sum(
    error.validator == "required"
    for error in schema_errors_case3
)

additional_property_errors_case3 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case3
)

type_errors_case3 = sum(
    error.validator == "type"
    for error in schema_errors_case3
)


print("=" * 80)
print("EXPERIMENT B - CASE 3 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", schema_json_valid_case3)
print("Schema valid:", schema_valid_case3)
print("Total schema errors:", schema_error_count_case3)
print("Required-field errors:", required_errors_case3)
print(
    "Additional-property errors:",
    additional_property_errors_case3
)
print("Type errors:", type_errors_case3)


if schema_errors_case3:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case3,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case3_hash)

EXPERIMENT B - CASE 3 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
84628d313e4abd7b427df0fc85a3a94b1a0114081c3f4433ed36a468e976886d


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 3
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


schema_errors_case3 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case3
    ),
    key=lambda e: list(e.absolute_path)
)


schema_valid_case3 = (
    1 if len(schema_errors_case3) == 0 else 0
)

schema_error_count_case3 = len(
    schema_errors_case3
)

required_errors_case3 = sum(
    error.validator == "required"
    for error in schema_errors_case3
)

additional_property_errors_case3 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case3
)

type_errors_case3 = sum(
    error.validator == "type"
    for error in schema_errors_case3
)


print("=" * 80)
print("EXPERIMENT B - CASE 3 STRUCTURAL VALIDATION")
print("=" * 80)

print("JSON valid:", schema_json_valid_case3)
print("Schema valid:", schema_valid_case3)
print("Total schema errors:", schema_error_count_case3)
print("Required-field errors:", required_errors_case3)
print(
    "Additional-property errors:",
    additional_property_errors_case3
)
print("Type errors:", type_errors_case3)


if schema_errors_case3:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case3,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case3_hash)

EXPERIMENT B - CASE 3 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
84628d313e4abd7b427df0fc85a3a94b1a0114081c3f4433ed36a468e976886d


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 3
# PREPARE SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. GET FROZEN GOLD, MODEL OUTPUT, TRANSCRIPT
# --------------------------------------------------

case3_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 3"
    ]["gold_output"]
)

case3_model_v2 = schema_output_case3

case3_transcript = research_cases[2][
    "transcript"
]


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 3 SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    case3_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        case3_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. SCHEMA-CONSTRAINED MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        case3_model_v2,
        indent=2
    )
)


# --------------------------------------------------
# 6. CONFIRM FROZEN HASH
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN MODEL OUTPUT HASH")
print("=" * 80)

print(
    schema_raw_case3_hash
)

EXPERIMENT B - CASE 3 SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi , albert . how are you ?
[patient] hey , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] so , albert is a 62-year-old male , with a past medical history significant for depression , type 2 diabetes , and kidney transplant , who is here today for emergency room follow-up .
[patient] mm-hmm .
[doctor] so , i got a notification that you were in the emergency room , but , but what were you there for ?
[patient] well , i , uh , i was n't really , uh , staying on top of my , uh , blood sugar readings , and i felt kinda woozy over the weekend . and i was little concerned , and my wife wanted to take me in and just have me checked out .
[doctor] okay . and , and was it , in fact , high ?
[patient] yeah , it was .
[doctor] okay . did you ... were you admitted to the hospital ?
[patient] uh , no .
[doctor] ok

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 3
# SAVE FINAL RESULT
# --------------------------------------------------

case3_constrained_result = {

    "case_id": "D2N090-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "schema_constrained",


    # --------------------------------------------------
    # STRUCTURAL
    # --------------------------------------------------

    "json_valid": schema_json_valid_case3,

    "schema_valid": schema_valid_case3,

    "schema_error_count": schema_error_count_case3,

    "required_field_errors": required_errors_case3,

    "additional_property_errors":
        additional_property_errors_case3,

    "type_errors": type_errors_case3,


    # --------------------------------------------------
    # CONFIRMED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 1,

    "confirmed_partial_extractions": 1,

    "confirmed_mapping_errors": 2,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 0,

    "confirmed_semantic_errors": 4,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 1,

    "mapping_ambiguities": 2,

    "annotation_ambiguities": 1,


    # --------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------

    "output_sha256":
        schema_raw_case3_hash
}


print("=" * 80)
print("EXPERIMENT B - CASE 3 FINAL RESULT")
print("=" * 80)

for key, value in case3_constrained_result.items():
    print(f"{key}: {value}")

EXPERIMENT B - CASE 3 FINAL RESULT
case_id: D2N090-virtassist
model: gemini-3.6-flash
condition: schema_constrained
json_valid: 1
schema_valid: 1
schema_error_count: 0
required_field_errors: 0
additional_property_errors: 0
type_errors: 0
confirmed_status_errors: 1
confirmed_partial_extractions: 1
confirmed_mapping_errors: 2
confirmed_unsupported_inferences: 0
confirmed_omissions: 0
confirmed_semantic_errors: 4
supported_extra_items: 1
mapping_ambiguities: 2
annotation_ambiguities: 1
output_sha256: 84628d313e4abd7b427df0fc85a3a94b1a0114081c3f4433ed36a468e976886d


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 4
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 4
# --------------------------------------------------

schema_case_index = 3

schema_case4 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case4["case_id"]
)


# --------------------------------------------------
# 2. CREATE SAME SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

schema_constrained_prompt_case4 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case4["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case4 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case4,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case4 = (
    schema_response_case4.output_text
)

schema_raw_case4_hash = hashlib.sha256(
    schema_raw_case4.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case4 = json.loads(
        schema_raw_case4
    )

    schema_json_valid_case4 = 1

except json.JSONDecodeError:

    schema_output_case4 = None
    schema_json_valid_case4 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 4")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case4
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case4_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case4
)

Case ID: D2N091-virtassist

EXPERIMENT B - CASE 4
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
02e1aee72067ad382183388c4f833abb85728d91d1b0aaa11eb26d52d5865247

MODEL OUTPUT:
{
  "patient": {
    "age": 54,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "annual exam",
    "chief_complaint": "lack of sleep"
  },
  "medical_history": [
    {
      "condition": "osteoporosis",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "multiple sclerosis",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "knee surgery",
      "status": "historical",
      "history_type": "surgical"
    }
  ],
  "symptoms": [
    {
      "name": "lack of sleep",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": "this summer"
    },
    {
      "name": "insomnia",
      "status": "present",
      "body_site": null,
      "severity": null,
      "duration": null
 

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 4
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. VALIDATOR USING FROZEN SCHEMA V2
# --------------------------------------------------

validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE CASE 4 OUTPUT
# --------------------------------------------------

schema_errors_case4 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case4
    ),
    key=lambda e: list(e.absolute_path)
)


# --------------------------------------------------
# 3. STRUCTURAL METRICS
# --------------------------------------------------

schema_valid_case4 = (
    1 if len(schema_errors_case4) == 0 else 0
)

schema_error_count_case4 = len(
    schema_errors_case4
)

required_errors_case4 = sum(
    error.validator == "required"
    for error in schema_errors_case4
)

additional_property_errors_case4 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case4
)

type_errors_case4 = sum(
    error.validator == "type"
    for error in schema_errors_case4
)


# --------------------------------------------------
# 4. PRINT RESULTS
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 4 STRUCTURAL VALIDATION")
print("=" * 80)

print(
    "JSON valid:",
    schema_json_valid_case4
)

print(
    "Schema valid:",
    schema_valid_case4
)

print(
    "Total schema errors:",
    schema_error_count_case4
)

print(
    "Required-field errors:",
    required_errors_case4
)

print(
    "Additional-property errors:",
    additional_property_errors_case4
)

print(
    "Type errors:",
    type_errors_case4
)


# --------------------------------------------------
# 5. PRINT ERRORS IF ANY
# --------------------------------------------------

if schema_errors_case4:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case4,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case4_hash)

EXPERIMENT B - CASE 4 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
02e1aee72067ad382183388c4f833abb85728d91d1b0aaa11eb26d52d5865247


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 4
# PREPARE SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. GET FROZEN GOLD, MODEL OUTPUT, TRANSCRIPT
# --------------------------------------------------

case4_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 4"
    ]["gold_output"]
)

case4_model_v2 = schema_output_case4

case4_transcript = research_cases[3][
    "transcript"
]


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 4 SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    case4_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        case4_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. SCHEMA-CONSTRAINED MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        case4_model_v2,
        indent=2
    )
)


# --------------------------------------------------
# 6. CONFIRM FROZEN HASH
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN MODEL OUTPUT HASH")
print("=" * 80)

print(
    schema_raw_case4_hash
)

EXPERIMENT B - CASE 4 SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi jerry , how are you doing ?
[patient] hi , good to see you .
[doctor] good to see you as well . um , so i know that the nurse told you about dax . i'd like to tell dax about you .
[patient] sure .
[doctor] jerry is a 54 year old male with a past medical history , significant for osteoporosis and multiple sclerosis who presents for an annual exam . so jerry , what's been going on since the last time i saw you ?
[patient] uh , we have been traveling all over the country . it's been kind of a stressful summer . kinda adjusting to everything in the fall and so far it's been good , but ah , lack of sleep , it's been really getting to me .
[doctor] okay . all right . and have you taken anything for the insomnia . have you tried any strategies for it .
[patient] i've tried everything from melatonin to meditation to , uh , t- stretching out every morning when i get up . nothing really seems to help though .
[doctor] okay . al

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 4
# MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


case4_constrained_semantic_review = [

    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "diagnostic_tests.vital_signs",
        "model_value": {
            "test_name": "vital signs",
            "status": "reviewed",
            "result": "really good"
        },
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The clinician reviews vital signs before the "
            "physical examination. Vital signs are supported, "
            "but they are not a diagnostic test, so the model "
            "places the information in the wrong section."
    },

    {
        "field": "assessment.insomnia",
        "model_value": {
            "condition": "insomnia",
            "certainty": "confirmed"
        },
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Insomnia is clearly reported as a symptom, but "
            "when the clinician begins the explicit assessment "
            "and plan, the addressed problems are osteoporosis "
            "and multiple sclerosis. The transcript does not "
            "explicitly establish insomnia as an assessment item."
    },


    # --------------------------------------------------
    # OMISSION
    # --------------------------------------------------

    {
        "field": "plan.patient_instructions",
        "model_value": [],
        "gold_value":
            "contact clinician if additional help is "
            "needed for multiple sclerosis",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly says to let them know "
            "if the patient needs anything related to the "
            "multiple sclerosis, but the model leaves "
            "patient_instructions empty."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRA INFORMATION
    # --------------------------------------------------

    {
        "field": "medical_history.knee_surgery",
        "model_value":
            "knee surgery - historical - surgical",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly refers to a lingering "
            "issue with a prior knee surgery."
    },

    {
        "field": "medical_history.broken_bones",
        "model_value":
            "broken bones - denied - trauma",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician asks about recent broken bones "
            "and the patient explicitly denies them."
    },

    {
        "field": "medications.melatonin",
        "model_value":
            "melatonin - past",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly says that melatonin was "
            "tried for insomnia, so past medication use is "
            "supported."
    },

    {
        "field": "symptoms.joint_issues",
        "model_value":
            "joint issues - denied",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician explicitly asks about joint issues "
            "and the patient denies them."
    },


    # --------------------------------------------------
    # AMBIGUITIES - DO NOT COUNT AS ERRORS
    # --------------------------------------------------

    {
        "field": "encounter.chief_complaint",
        "model_value": "insomnia",
        "gold_value": "annual exam",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The encounter is explicitly an annual exam, but "
            "the patient's main reported concern is lack of "
            "sleep. Both representations are supported."
    },

    {
        "field": "physical_exam.reflexes.body_site",
        "model_value": None,
        "gold_value": "lower extremities",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The reflex finding occurs in the context of the "
            "lower-extremity strength examination, but the "
            "transcript does not explicitly repeat the body "
            "site when mentioning reflexes."
    }
]


case4_constrained_review_df = pd.DataFrame(
    case4_constrained_semantic_review
)


print("=" * 80)
print("EXPERIMENT B - CASE 4 SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    case4_constrained_review_df
)


confirmed_errors_case4 = (
    case4_constrained_review_df[
        case4_constrained_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_errors_case4[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(confirmed_errors_case4)
)

EXPERIMENT B - CASE 4 SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,diagnostic_tests.vital_signs,"{'test_name': 'vital signs', 'status': 'review...",None,mapping_error,1,The clinician reviews vital signs before the p...
1,assessment.insomnia,"{'condition': 'insomnia', 'certainty': 'confir...",None,mapping_error,1,"Insomnia is clearly reported as a symptom, but..."
2,plan.patient_instructions,[],contact clinician if additional help is needed...,omission,1,The clinician explicitly says to let them know...
3,medical_history.knee_surgery,knee surgery - historical - surgical,None,supported_extra,0,The patient explicitly refers to a lingering i...
4,medical_history.broken_bones,broken bones - denied - trauma,None,supported_extra,0,The clinician asks about recent broken bones a...
5,medications.melatonin,melatonin - past,None,supported_extra,0,The patient explicitly says that melatonin was...
6,symptoms.joint_issues,joint issues - denied,None,supported_extra,0,The clinician explicitly asks about joint issu...
7,encounter.chief_complaint,insomnia,annual exam,annotation_ambiguity,0,"The encounter is explicitly an annual exam, bu..."
8,physical_exam.reflexes.body_site,None,lower extremities,annotation_ambiguity,0,The reflex finding occurs in the context of th...



CONFIRMED SEMANTIC ERRORS:
category
mapping_error    2
omission         1
Name: count, dtype: int64

Total confirmed semantic errors: 3


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 4
# SAVE FINAL RESULT
# --------------------------------------------------

case4_constrained_result = {

    "case_id": "D2N091-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "schema_constrained",


    # --------------------------------------------------
    # STRUCTURAL
    # --------------------------------------------------

    "json_valid": schema_json_valid_case4,

    "schema_valid": schema_valid_case4,

    "schema_error_count": schema_error_count_case4,

    "required_field_errors": required_errors_case4,

    "additional_property_errors":
        additional_property_errors_case4,

    "type_errors": type_errors_case4,


    # --------------------------------------------------
    # CONFIRMED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 0,

    "confirmed_mapping_errors": 2,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 1,

    "confirmed_semantic_errors": 3,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 4,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 2,


    # --------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------

    "output_sha256":
        schema_raw_case4_hash
}


print("=" * 80)
print("EXPERIMENT B - CASE 4 FINAL RESULT")
print("=" * 80)

for key, value in case4_constrained_result.items():
    print(f"{key}: {value}")

EXPERIMENT B - CASE 4 FINAL RESULT
case_id: D2N091-virtassist
model: gemini-3.6-flash
condition: schema_constrained
json_valid: 1
schema_valid: 1
schema_error_count: 0
required_field_errors: 0
additional_property_errors: 0
type_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 0
confirmed_mapping_errors: 2
confirmed_unsupported_inferences: 0
confirmed_omissions: 1
confirmed_semantic_errors: 3
supported_extra_items: 4
mapping_ambiguities: 0
annotation_ambiguities: 2
output_sha256: 02e1aee72067ad382183388c4f833abb85728d91d1b0aaa11eb26d52d5865247


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - SCHEMA-CONSTRAINED GENERATION
# CASE 5
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT CASE 5
# --------------------------------------------------

schema_case_index = 4

schema_case5 = research_cases[
    schema_case_index
]

print(
    "Case ID:",
    schema_case5["case_id"]
)


# --------------------------------------------------
# 2. CREATE SAME SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

schema_constrained_prompt_case5 = f"""
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{schema_case5["transcript"]}
"""


# --------------------------------------------------
# 3. CALL GEMINI WITH FROZEN SCHEMA V2
# --------------------------------------------------

schema_response_case5 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=schema_constrained_prompt_case5,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

schema_raw_case5 = (
    schema_response_case5.output_text
)

schema_raw_case5_hash = hashlib.sha256(
    schema_raw_case5.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    schema_output_case5 = json.loads(
        schema_raw_case5
    )

    schema_json_valid_case5 = 1

except json.JSONDecodeError:

    schema_output_case5 = None
    schema_json_valid_case5 = 0


# --------------------------------------------------
# 6. DISPLAY RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - CASE 5")
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    "\nJSON valid:",
    schema_json_valid_case5
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    schema_raw_case5_hash
)

print(
    "\nMODEL OUTPUT:"
)

print(
    schema_raw_case5
)

Case ID: D2N092-virtassist

EXPERIMENT B - CASE 5
SCHEMA-CONSTRAINED OUTPUT

JSON valid: 1

RAW OUTPUT SHA256:
a1657eb442b189efaf1fb8bf60011bb3f8b58392ac92cfeb26c8d0ccd0a08fd8

MODEL OUTPUT:
{
  "patient": {
    "age": 43,
    "sex": "female"
  },
  "encounter": {
    "visit_reason": "arm pain",
    "chief_complaint": "arm pain"
  },
  "medical_history": [
    {
      "condition": "allergies",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "trauma from falling on a rake",
      "status": "resolved",
      "history_type": "trauma"
    }
  ],
  "symptoms": [
    {
      "name": "arm pain",
      "status": "present",
      "body_site": "distal radius",
      "severity": null,
      "duration": null
    },
    {
      "name": "swelling",
      "status": "present",
      "body_site": "arm",
      "severity": "mild",
      "duration": null
    },
    {
      "name": "pain with wrist movement",
      "status": "present",
      "body_site": "wrist",


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 5
# VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. VALIDATOR USING FROZEN SCHEMA V2
# --------------------------------------------------

validator_v2_frozen = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE CASE 5 OUTPUT
# --------------------------------------------------

schema_errors_case5 = sorted(
    validator_v2_frozen.iter_errors(
        schema_output_case5
    ),
    key=lambda e: list(e.absolute_path)
)


# --------------------------------------------------
# 3. STRUCTURAL METRICS
# --------------------------------------------------

schema_valid_case5 = (
    1 if len(schema_errors_case5) == 0 else 0
)

schema_error_count_case5 = len(
    schema_errors_case5
)

required_errors_case5 = sum(
    error.validator == "required"
    for error in schema_errors_case5
)

additional_property_errors_case5 = sum(
    error.validator == "additionalProperties"
    for error in schema_errors_case5
)

type_errors_case5 = sum(
    error.validator == "type"
    for error in schema_errors_case5
)


# --------------------------------------------------
# 4. PRINT RESULTS
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 5 STRUCTURAL VALIDATION")
print("=" * 80)

print(
    "JSON valid:",
    schema_json_valid_case5
)

print(
    "Schema valid:",
    schema_valid_case5
)

print(
    "Total schema errors:",
    schema_error_count_case5
)

print(
    "Required-field errors:",
    required_errors_case5
)

print(
    "Additional-property errors:",
    additional_property_errors_case5
)

print(
    "Type errors:",
    type_errors_case5
)


# --------------------------------------------------
# 5. PRINT ERRORS IF ANY
# --------------------------------------------------

if schema_errors_case5:

    print("\nSCHEMA ERRORS:")

    for i, error in enumerate(
        schema_errors_case5,
        start=1
    ):

        path = ".".join(
            str(x)
            for x in error.absolute_path
        )

        print(
            f"{i}. "
            f"{path or '<root>'}: "
            f"{error.message}"
        )

else:

    print(
        "\n✅ Output is fully valid "
        "against frozen clinical_schema_v2"
    )


print("\nFrozen output SHA256:")
print(schema_raw_case5_hash)

EXPERIMENT B - CASE 5 STRUCTURAL VALIDATION
JSON valid: 1
Schema valid: 1
Total schema errors: 0
Required-field errors: 0
Additional-property errors: 0
Type errors: 0

✅ Output is fully valid against frozen clinical_schema_v2

Frozen output SHA256:
a1657eb442b189efaf1fb8bf60011bb3f8b58392ac92cfeb26c8d0ccd0a08fd8


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 5
# PREPARE SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. GET FROZEN GOLD, MODEL OUTPUT, TRANSCRIPT
# --------------------------------------------------

case5_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 5"
    ]["gold_output"]
)

case5_model_v2 = schema_output_case5

case5_transcript = research_cases[4][
    "transcript"
]


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - CASE 5 SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    case5_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        case5_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. SCHEMA-CONSTRAINED MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        case5_model_v2,
        indent=2
    )
)


# --------------------------------------------------
# 6. CONFIRM FROZEN HASH
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN MODEL OUTPUT HASH")
print("=" * 80)

print(
    schema_raw_case5_hash
)

EXPERIMENT B - CASE 5 SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hello , mrs . martinez . good to see you today .
[patient] hey , dr . gomez .
[doctor] hey , dragon , i'm here seeing mrs . martinez . she's a 43-year-old female . why are we seeing you today ?
[patient] um , my arm hurts right here . kind of toward my wrist . this part of my arm .
[doctor] so you have pain in your distal radius ?
[patient] yes .
[doctor] how did that happen ?
[patient] um , i was playing tennis , and when i went to hit , um , i was given a , a backhand , and when i did , i m- totally missed the ball , hit the top of the net but the pole part . and , and it just jarred my arm .
[doctor] okay . and did it swell up at all ? or-
[patient] it did . it got a ... it had a little bit of swelling . not a lot .
[doctor] okay . and , um , did , uh , do you have any numbness in your hand at all ? or any pain when you move your wrist ?
[patient] a little bit when i move my wrist . um , no numbness in my hand .
[doct

In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 5
# MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


case5_constrained_semantic_review = [

    # --------------------------------------------------
    # PARTIAL EXTRACTION
    # --------------------------------------------------

    {
        "field": "physical_exam.thumb_stress_and_flexion",
        "model_value": {
            "finding": "Pain with movement and thumb flexion",
            "body_site": "arm"
        },
        "gold_value": {
            "finding": "pain with thumb stress and flexion",
            "body_site": "right thumb"
        },
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The model captures pain with movement and thumb "
            "flexion, but combines distinct examination findings "
            "and loses the more specific thumb body site."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRA INFORMATION
    # --------------------------------------------------

    {
        "field": "medical_history.prior_trauma",
        "model_value":
            "Trauma from falling on a rake - historical",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly reports a prior trauma "
            "involving falling on a rake."
    },

    {
        "field": "assessment.fracture",
        "model_value": "fracture - ruled_out",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The X-ray shows no fracture and the clinician "
            "explicitly states that the patient does not "
            "have a fracture."
    },


    # --------------------------------------------------
    # GOLD / ANNOTATION DIFFERENCES
    # DO NOT COUNT AS MODEL ERRORS
    # --------------------------------------------------

    {
        "field": "medications.Flonase.action",
        "model_value": [],
        "gold_value": ["continue"],
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The transcript establishes that the patient currently "
            "takes Flonase, but the clinician does not explicitly "
            "instruct her to continue it during this encounter. "
            "Therefore an empty action list is defensible."
    },

    {
        "field": "assessment.strain.certainty",
        "model_value": "suspected",
        "gold_value": "confirmed",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The clinician says 'I think what you have is "
            "basically just a strain,' so suspected is a "
            "reasonable certainty label."
    },

    {
        "field": "laterality",
        "model_value":
            "distal radius / arm / hand / wrist without side",
        "gold_value":
            "right distal radius / right arm / right hand / right wrist",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The transcript shown for this case does not explicitly "
            "state right or left laterality. The model should not be "
            "penalized for avoiding unsupported laterality."
    },

    {
        "field": "plan.patient_instructions.motrin",
        "model_value":
            "Take Motrin 800 mg three times a day with food.",
        "gold_value":
            "represented under medication_changes",
        "category": "supported_duplicate",
        "count_as_error": 0,
        "reason":
            "The instruction is explicitly supported by the "
            "transcript. Repeating it under patient instructions "
            "does not introduce an unsupported fact."
    }
]


case5_constrained_review_df = pd.DataFrame(
    case5_constrained_semantic_review
)


print("=" * 80)
print("EXPERIMENT B - CASE 5 SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    case5_constrained_review_df
)


confirmed_errors_case5 = (
    case5_constrained_review_df[
        case5_constrained_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_errors_case5[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(confirmed_errors_case5)
)

EXPERIMENT B - CASE 5 SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,physical_exam.thumb_stress_and_flexion,{'finding': 'Pain with movement and thumb flex...,{'finding': 'pain with thumb stress and flexio...,partial_extraction,1,The model captures pain with movement and thum...
1,medical_history.prior_trauma,Trauma from falling on a rake - historical,None,supported_extra,0,The patient explicitly reports a prior trauma ...
2,assessment.fracture,fracture - ruled_out,None,supported_extra,0,The X-ray shows no fracture and the clinician ...
3,medications.Flonase.action,[],[continue],annotation_ambiguity,0,The transcript establishes that the patient cu...
4,assessment.strain.certainty,suspected,confirmed,annotation_ambiguity,0,The clinician says 'I think what you have is b...
5,laterality,distal radius / arm / hand / wrist without side,right distal radius / right arm / right hand /...,annotation_ambiguity,0,The transcript shown for this case does not ex...
6,plan.patient_instructions.motrin,Take Motrin 800 mg three times a day with food.,represented under medication_changes,supported_duplicate,0,The instruction is explicitly supported by the...



CONFIRMED SEMANTIC ERRORS:
category
partial_extraction    1
Name: count, dtype: int64

Total confirmed semantic errors: 1


In [ ]:
# --------------------------------------------------
# EXPERIMENT B - CASE 5
# SAVE FINAL RESULT
# --------------------------------------------------

case5_constrained_result = {

    "case_id": "D2N092-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "schema_constrained",


    # --------------------------------------------------
    # STRUCTURAL
    # --------------------------------------------------

    "json_valid": schema_json_valid_case5,

    "schema_valid": schema_valid_case5,

    "schema_error_count": schema_error_count_case5,

    "required_field_errors": required_errors_case5,

    "additional_property_errors":
        additional_property_errors_case5,

    "type_errors": type_errors_case5,


    # --------------------------------------------------
    # CONFIRMED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 1,

    "confirmed_mapping_errors": 0,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 0,

    "confirmed_semantic_errors": 1,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 2,

    "supported_duplicate_items": 1,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 3,


    # --------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------

    "output_sha256":
        schema_raw_case5_hash
}


print("=" * 80)
print("EXPERIMENT B - CASE 5 FINAL RESULT")
print("=" * 80)

for key, value in case5_constrained_result.items():
    print(f"{key}: {value}")

EXPERIMENT B - CASE 5 FINAL RESULT
case_id: D2N092-virtassist
model: gemini-3.6-flash
condition: schema_constrained
json_valid: 1
schema_valid: 1
schema_error_count: 0
required_field_errors: 0
additional_property_errors: 0
type_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 1
confirmed_mapping_errors: 0
confirmed_unsupported_inferences: 0
confirmed_omissions: 0
confirmed_semantic_errors: 1
supported_extra_items: 2
supported_duplicate_items: 1
mapping_ambiguities: 0
annotation_ambiguities: 3
output_sha256: a1657eb442b189efaf1fb8bf60011bb3f8b58392ac92cfeb26c8d0ccd0a08fd8


In [ ]:
# --------------------------------------------------
# EXPERIMENT B
# COMBINE ALL 5 SCHEMA-CONSTRAINED RESULTS
# --------------------------------------------------

import pandas as pd


# --------------------------------------------------
# 1. COMBINE CASE RESULTS
# --------------------------------------------------

schema_constrained_pilot_results = [

    case1_constrained_result,
    case2_constrained_result,
    case3_constrained_result,
    case4_constrained_result,
    case5_constrained_result

]


schema_constrained_pilot_df = pd.DataFrame(
    schema_constrained_pilot_results
)


# --------------------------------------------------
# 2. DISPLAY CASE-LEVEL RESULTS
# --------------------------------------------------

print("=" * 80)
print("EXPERIMENT B - SCHEMA-CONSTRAINED PILOT RESULTS")
print("=" * 80)

display(
    schema_constrained_pilot_df[
        [
            "case_id",
            "json_valid",
            "schema_valid",
            "schema_error_count",
            "confirmed_partial_extractions",
            "confirmed_mapping_errors",
            "confirmed_status_errors",
            "confirmed_omissions",
            "confirmed_unsupported_inferences",
            "confirmed_semantic_errors"
        ]
    ]
)


# --------------------------------------------------
# 3. STRUCTURAL SUMMARY
# --------------------------------------------------

total_cases = len(
    schema_constrained_pilot_df
)

json_valid_outputs = int(
    schema_constrained_pilot_df[
        "json_valid"
    ].sum()
)

schema_valid_outputs = int(
    schema_constrained_pilot_df[
        "schema_valid"
    ].sum()
)

total_schema_errors = int(
    schema_constrained_pilot_df[
        "schema_error_count"
    ].sum()
)


# --------------------------------------------------
# 4. CONFIRMED SEMANTIC ERROR SUMMARY
# --------------------------------------------------

total_partial = int(
    schema_constrained_pilot_df[
        "confirmed_partial_extractions"
    ].sum()
)

total_mapping = int(
    schema_constrained_pilot_df[
        "confirmed_mapping_errors"
    ].sum()
)

total_status = int(
    schema_constrained_pilot_df[
        "confirmed_status_errors"
    ].sum()
)

total_omissions = int(
    schema_constrained_pilot_df[
        "confirmed_omissions"
    ].sum()
)

total_unsupported = int(
    schema_constrained_pilot_df[
        "confirmed_unsupported_inferences"
    ].sum()
)

total_confirmed_semantic_errors = int(
    schema_constrained_pilot_df[
        "confirmed_semantic_errors"
    ].sum()
)


# --------------------------------------------------
# 5. PRINT PILOT SUMMARY
# --------------------------------------------------

print("\n" + "=" * 80)
print("EXPERIMENT B - 5 CASE PILOT SUMMARY")
print("=" * 80)


print("\nSTRUCTURAL")
print("-" * 40)

print(
    "Cases evaluated:",
    total_cases
)

print(
    "Valid JSON outputs:",
    json_valid_outputs,
    "/",
    total_cases
)

print(
    "Schema-valid outputs:",
    schema_valid_outputs,
    "/",
    total_cases
)

print(
    "Total schema errors:",
    total_schema_errors
)


print("\nCONFIRMED SEMANTIC ERRORS")
print("-" * 40)

print(
    "Partial extractions:",
    total_partial
)

print(
    "Mapping errors:",
    total_mapping
)

print(
    "Status/certainty errors:",
    total_status
)

print(
    "Omissions:",
    total_omissions
)

print(
    "Unsupported inferences:",
    total_unsupported
)

print(
    "Total confirmed semantic errors:",
    total_confirmed_semantic_errors
)

EXPERIMENT B - SCHEMA-CONSTRAINED PILOT RESULTS


,case_id,json_valid,schema_valid,schema_error_count,confirmed_partial_extractions,confirmed_mapping_errors,confirmed_status_errors,confirmed_omissions,confirmed_unsupported_inferences,confirmed_semantic_errors
0,D2N088-virtassist,1,1,0,1,0,2,0,0,3
1,D2N089-virtassist,1,1,0,2,2,0,1,0,5
2,D2N090-virtassist,1,1,0,1,2,1,0,0,4
3,D2N091-virtassist,1,1,0,0,2,0,1,0,3
4,D2N092-virtassist,1,1,0,1,0,0,0,0,1



EXPERIMENT B - 5 CASE PILOT SUMMARY

STRUCTURAL
----------------------------------------
Cases evaluated: 5
Valid JSON outputs: 5 / 5
Schema-valid outputs: 5 / 5
Total schema errors: 0

CONFIRMED SEMANTIC ERRORS
----------------------------------------
Partial extractions: 5
Mapping errors: 6
Status/certainty errors: 3
Omissions: 2
Unsupported inferences: 0
Total confirmed semantic errors: 16


In [ ]:
# --------------------------------------------------
# LOCATE BASELINE / CONSTRAINED OUTPUT VARIABLES
# SAFE VERSION
# --------------------------------------------------

print("=" * 80)
print("LOCATING HELD-OUT OUTPUT VARIABLES")
print("=" * 80)

keywords = [
    "baseline",
    "constrained",
    "heldout"
]

# IMPORTANT:
# Make a fixed copy before looping
global_snapshot = list(
    globals().items()
)

matches = []

for name, value in global_snapshot:

    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in keywords
    ):

        matches.append(
            (
                name,
                type(value).__name__
            )
        )


# --------------------------------------------------
# PRINT RESULTS
# --------------------------------------------------

for name, value_type in sorted(matches):

    print(
        f"{name:<55} {value_type}"
    )

print("\nTotal matching variables:", len(matches))

LOCATING HELD-OUT OUTPUT VARIABLES
BASELINE_PROMPT_TEMPLATE                                str
SCHEMA_CONSTRAINED_PROMPT_TEMPLATE                      str
baseline_case1_review_df                                DataFrame
baseline_case1_semantic_review                          list
baseline_case2_review_df                                DataFrame
baseline_case2_semantic_review                          list
baseline_case3_review_df                                DataFrame
baseline_case3_semantic_review                          list
baseline_case4_review_df                                DataFrame
baseline_case4_semantic_review                          list
baseline_case5_review_df                                DataFrame
baseline_case5_semantic_review                          list
baseline_metrics_case1                                  dict
baseline_output                                         dict
baseline_output_case2                                   dict
baseline_output_case3      

In [ ]:
# --------------------------------------------------
# LOCATE SAVED BASELINE OUTPUTS
# SAFE VERSION FOR COLAB
# --------------------------------------------------

print("=" * 80)
print("LOCATING BASELINE OUTPUT VARIABLES")
print("=" * 80)


# Take a snapshot so globals() cannot change
# while we are looping through it
global_snapshot = list(
    globals().items()
)


for case_num in range(1, 6):

    print(f"\nCASE {case_num}")

    matches = []

    for name, value in global_snapshot:

        name_lower = name.lower()

        if (
            "baseline" in name_lower
            and f"case{case_num}" in name_lower
            and isinstance(
                value,
                (dict, str)
            )
        ):
            matches.append(name)

    if matches:

        for name in matches:
            print(" -", name)

    else:

        print(
            "No matching baseline variable found"
        )

LOCATING BASELINE OUTPUT VARIABLES

CASE 1
 - baseline_metrics_case1

CASE 2
 - baseline_prompt_case2
 - baseline_raw_case2
 - baseline_output_case2

CASE 3
 - baseline_prompt_case3
 - baseline_raw_case3
 - baseline_output_case3

CASE 4
 - baseline_prompt_case4
 - baseline_raw_case4
 - baseline_output_case4

CASE 5
 - baseline_prompt_case5
 - baseline_raw_case5
 - baseline_output_case5


In [ ]:
# --------------------------------------------------
# LOCATE CASE 1 FROZEN BASELINE OUTPUT
# DO NOT CALL GEMINI
# --------------------------------------------------

print("=" * 80)
print("LOCATING CASE 1 BASELINE JSON")
print("=" * 80)

global_snapshot = list(
    globals().items()
)

candidate_case1_variables = []


for name, value in global_snapshot:

    # Ignore things that clearly are not
    # candidate model outputs
    if (
        "metric" in name.lower()
        or "gold" in name.lower()
        or "schema" in name.lower()
        or "prompt" in name.lower()
    ):
        continue


    # ----------------------------------------------
    # DICTIONARY CANDIDATES
    # ----------------------------------------------

    if isinstance(value, dict):

        # Clinical model outputs normally have
        # sections such as patient / encounter /
        # symptoms / assessment / plan
        clinical_keys = {
            "patient",
            "encounter",
            "medical_history",
            "symptoms",
            "medications",
            "physical_exam",
            "diagnostic_tests",
            "assessment",
            "plan",
            "follow_up"
        }

        matching_keys = (
            clinical_keys.intersection(
                value.keys()
            )
        )

        if len(matching_keys) >= 3:

            candidate_case1_variables.append(
                (
                    name,
                    "dict",
                    sorted(matching_keys)
                )
            )


    # ----------------------------------------------
    # STRING CANDIDATES
    # ----------------------------------------------

    elif isinstance(value, str):

        text = value.lower()

        if (
            '"patient"' in text
            and '"encounter"' in text
            and '"assessment"' in text
        ):

            candidate_case1_variables.append(
                (
                    name,
                    "string",
                    "looks like clinical JSON"
                )
            )


# --------------------------------------------------
# PRINT CANDIDATES
# --------------------------------------------------

if candidate_case1_variables:

    for item in candidate_case1_variables:

        print(
            "\nVariable:",
            item[0]
        )

        print(
            "Type:",
            item[1]
        )

        print(
            "Info:",
            item[2]
        )

else:

    print(
        "\nNo candidate Case 1 output "
        "currently exists in memory."
    )

LOCATING CASE 1 BASELINE JSON

Variable: _i7
Type: string
Info: looks like clinical JSON

Variable: _i40
Type: string
Info: looks like clinical JSON

Variable: _i43
Type: string
Info: looks like clinical JSON

Variable: _i50
Type: string
Info: looks like clinical JSON

Variable: _i51
Type: string
Info: looks like clinical JSON

Variable: output_template
Type: dict
Info: ['assessment', 'diagnostic_tests', 'encounter', 'follow_up', 'medical_history', 'medications', 'patient', 'physical_exam', 'plan', 'symptoms']

Variable: _i52
Type: string
Info: looks like clinical JSON

Variable: _i53
Type: string
Info: looks like clinical JSON

Variable: baseline_raw
Type: string
Info: looks like clinical JSON

Variable: clean_baseline
Type: string
Info: looks like clinical JSON

Variable: baseline_output
Type: dict
Info: ['assessment', 'diagnostic_tests', 'encounter', 'follow_up', 'medical_history', 'medications', 'patient', 'physical_exam', 'plan', 'symptoms']

Variable: section_matches
Type: dict
I

In [ ]:
# --------------------------------------------------
# VERIFY CASE 1 BASELINE OUTPUT
# BEFORE CREATING PERMANENT ALIAS
# --------------------------------------------------

print("=" * 80)
print("VERIFYING CASE 1 BASELINE OUTPUT")
print("=" * 80)

print("\nPatient:")
print(
    baseline_output.get(
        "patient"
    )
)

print("\nEncounter:")
print(
    baseline_output.get(
        "encounter"
    )
)

print("\nTop-level sections:")
print(
    list(
        baseline_output.keys()
    )
)

print("\nExisting Case 1 baseline metrics:")
print(
    baseline_metrics_case1
)

VERIFYING CASE 1 BASELINE OUTPUT

Patient:
{'age': 59, 'sex': 'male'}

Encounter:
{'visit_reason': 'upper respiratory infection', 'chief_complaint': 'shortness of breath and fatigue'}

Top-level sections:
['patient', 'encounter', 'medical_history', 'symptoms', 'medications', 'physical_exam', 'diagnostic_tests', 'assessment', 'plan', 'follow_up']

Existing Case 1 baseline metrics:
{'case_id': 'D2N088-virtassist', 'model': 'gemini-3.6-flash', 'condition': 'baseline', 'json_valid': 1, 'schema_valid': 0, 'schema_error_count': 59, 'required_field_errors': 44, 'additional_property_errors': 10, 'type_errors': 5, 'exact_section_matches': 2, 'total_sections': 10, 'exact_section_match_rate': 0.2, 'exact_record_match': 0}


In [ ]:
# --------------------------------------------------
# BASELINE
# REVALIDATE ALL 5 FROZEN OUTPUTS AGAINST
# THE SAME FROZEN SCHEMA V2
# --------------------------------------------------

from jsonschema import Draft202012Validator
import pandas as pd


# --------------------------------------------------
# 1. CREATE CASE 1 PERMANENT ALIAS
# --------------------------------------------------

baseline_output_case1 = baseline_output


# --------------------------------------------------
# 2. COLLECT ALL ORIGINAL BASELINE OUTPUTS
# --------------------------------------------------

baseline_outputs_v2_validation = [

    (
        "D2N088-virtassist",
        baseline_output_case1
    ),

    (
        "D2N089-virtassist",
        baseline_output_case2
    ),

    (
        "D2N090-virtassist",
        baseline_output_case3
    ),

    (
        "D2N091-virtassist",
        baseline_output_case4
    ),

    (
        "D2N092-virtassist",
        baseline_output_case5
    )

]


# --------------------------------------------------
# 3. VALIDATOR
# SAME FROZEN SCHEMA USED IN EXPERIMENT B
# --------------------------------------------------

baseline_v2_validator = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 4. VALIDATE EACH ORIGINAL BASELINE OUTPUT
# --------------------------------------------------

baseline_v2_results = []


for case_id, output in baseline_outputs_v2_validation:

    errors = sorted(
        baseline_v2_validator.iter_errors(
            output
        ),
        key=lambda e: list(e.absolute_path)
    )


    # ----------------------------------------------
    # COUNT ERROR TYPES
    # ----------------------------------------------

    required_count = sum(
        error.validator == "required"
        for error in errors
    )

    additional_count = sum(
        error.validator == "additionalProperties"
        for error in errors
    )

    type_count = sum(
        error.validator == "type"
        for error in errors
    )

    enum_count = sum(
        error.validator == "enum"
        for error in errors
    )

    unique_items_count = sum(
        error.validator == "uniqueItems"
        for error in errors
    )


    baseline_v2_results.append({

        "case_id": case_id,

        "json_valid": 1,

        "schema_valid":
            1 if len(errors) == 0 else 0,

        "schema_error_count":
            len(errors),

        "required_field_errors":
            required_count,

        "additional_property_errors":
            additional_count,

        "type_errors":
            type_count,

        "enum_errors":
            enum_count,

        "unique_items_errors":
            unique_items_count

    })


# --------------------------------------------------
# 5. DATAFRAME
# --------------------------------------------------

baseline_v2_df = pd.DataFrame(
    baseline_v2_results
)


print("=" * 80)
print("BASELINE OUTPUTS REVALIDATED AGAINST FROZEN SCHEMA V2")
print("=" * 80)

display(
    baseline_v2_df
)


# --------------------------------------------------
# 6. SUMMARY
# --------------------------------------------------

print("\n" + "=" * 80)
print("BASELINE - SAME-SCHEMA STRUCTURAL SUMMARY")
print("=" * 80)

print(
    "Cases evaluated:",
    len(baseline_v2_df)
)

print(
    "JSON-valid outputs:",
    int(
        baseline_v2_df[
            "json_valid"
        ].sum()
    ),
    "/",
    len(baseline_v2_df)
)

print(
    "Schema-valid outputs:",
    int(
        baseline_v2_df[
            "schema_valid"
        ].sum()
    ),
    "/",
    len(baseline_v2_df)
)

print(
    "Total schema errors:",
    int(
        baseline_v2_df[
            "schema_error_count"
        ].sum()
    )
)

print(
    "Required-field errors:",
    int(
        baseline_v2_df[
            "required_field_errors"
        ].sum()
    )
)

print(
    "Additional-property errors:",
    int(
        baseline_v2_df[
            "additional_property_errors"
        ].sum()
    )
)

print(
    "Type errors:",
    int(
        baseline_v2_df[
            "type_errors"
        ].sum()
    )
)

print(
    "Enum errors:",
    int(
        baseline_v2_df[
            "enum_errors"
        ].sum()
    )
)

print(
    "Unique-items errors:",
    int(
        baseline_v2_df[
            "unique_items_errors"
        ].sum()
    )
)

BASELINE OUTPUTS REVALIDATED AGAINST FROZEN SCHEMA V2


,case_id,json_valid,schema_valid,schema_error_count,required_field_errors,additional_property_errors,type_errors,enum_errors,unique_items_errors
0,D2N088-virtassist,1,0,83,55,16,5,7,0
1,D2N089-virtassist,1,0,24,2,1,21,0,0
2,D2N090-virtassist,1,0,27,2,1,24,0,0
3,D2N091-virtassist,1,0,24,2,1,21,0,0
4,D2N092-virtassist,1,0,19,2,1,16,0,0



BASELINE - SAME-SCHEMA STRUCTURAL SUMMARY
Cases evaluated: 5
JSON-valid outputs: 5 / 5
Schema-valid outputs: 0 / 5
Total schema errors: 177
Required-field errors: 63
Additional-property errors: 20
Type errors: 87
Enum errors: 7
Unique-items errors: 0


In [ ]:
# --------------------------------------------------
# PILOT STRUCTURAL COMPARISON
# BASELINE VS SCHEMA-CONSTRAINED
# SAME FROZEN SCHEMA V2
# --------------------------------------------------

import pandas as pd


# --------------------------------------------------
# 1. BUILD COMPARISON TABLE
# --------------------------------------------------

structural_comparison = pd.DataFrame({

    "metric": [
        "Cases evaluated",
        "JSON-valid outputs",
        "Schema-valid outputs",
        "Total schema errors",
        "Required-field errors",
        "Additional-property errors",
        "Type errors",
        "Enum errors"
    ],

    "baseline": [
        len(baseline_v2_df),

        int(
            baseline_v2_df[
                "json_valid"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "schema_valid"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "schema_error_count"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "required_field_errors"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "additional_property_errors"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "type_errors"
            ].sum()
        ),

        int(
            baseline_v2_df[
                "enum_errors"
            ].sum()
        )
    ],

    "schema_constrained": [
        len(schema_constrained_pilot_df),

        int(
            schema_constrained_pilot_df[
                "json_valid"
            ].sum()
        ),

        int(
            schema_constrained_pilot_df[
                "schema_valid"
            ].sum()
        ),

        int(
            schema_constrained_pilot_df[
                "schema_error_count"
            ].sum()
        ),

        int(
            schema_constrained_pilot_df[
                "required_field_errors"
            ].sum()
        ),

        int(
            schema_constrained_pilot_df[
                "additional_property_errors"
            ].sum()
        ),

        int(
            schema_constrained_pilot_df[
                "type_errors"
            ].sum()
        ),

        0
    ]

})


# --------------------------------------------------
# 2. DISPLAY TABLE
# --------------------------------------------------

print("=" * 80)
print("PILOT STRUCTURAL COMPARISON")
print("BASELINE VS SCHEMA-CONSTRAINED")
print("VALIDATED AGAINST SAME FROZEN SCHEMA V2")
print("=" * 80)

display(
    structural_comparison
)


# --------------------------------------------------
# 3. CALCULATE KEY RATES
# --------------------------------------------------

baseline_schema_valid_rate = (
    baseline_v2_df[
        "schema_valid"
    ].mean()
)

constrained_schema_valid_rate = (
    schema_constrained_pilot_df[
        "schema_valid"
    ].mean()
)


baseline_total_errors = int(
    baseline_v2_df[
        "schema_error_count"
    ].sum()
)

constrained_total_errors = int(
    schema_constrained_pilot_df[
        "schema_error_count"
    ].sum()
)


schema_error_reduction = (
    (
        baseline_total_errors
        - constrained_total_errors
    )
    / baseline_total_errors
) * 100


# --------------------------------------------------
# 4. PRINT KEY PILOT FINDINGS
# --------------------------------------------------

print("\n" + "=" * 80)
print("KEY STRUCTURAL FINDINGS")
print("=" * 80)

print(
    "Baseline schema-valid rate:",
    f"{baseline_schema_valid_rate:.1%}"
)

print(
    "Schema-constrained schema-valid rate:",
    f"{constrained_schema_valid_rate:.1%}"
)

print(
    "Baseline schema errors:",
    baseline_total_errors
)

print(
    "Schema-constrained schema errors:",
    constrained_total_errors
)

print(
    "Schema-error reduction:",
    f"{schema_error_reduction:.1f}%"
)

PILOT STRUCTURAL COMPARISON
BASELINE VS SCHEMA-CONSTRAINED
VALIDATED AGAINST SAME FROZEN SCHEMA V2


,metric,baseline,schema_constrained
0,Cases evaluated,5,5
1,JSON-valid outputs,5,5
2,Schema-valid outputs,0,5
3,Total schema errors,177,0
4,Required-field errors,63,0
5,Additional-property errors,20,0
6,Type errors,87,0
7,Enum errors,7,0



KEY STRUCTURAL FINDINGS
Baseline schema-valid rate: 0.0%
Schema-constrained schema-valid rate: 100.0%
Baseline schema errors: 177
Schema-constrained schema errors: 0
Schema-error reduction: 100.0%


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 1
# PREPARE HARMONIZED SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. LOAD SAME SOURCE / GOLD / FROZEN BASELINE
# --------------------------------------------------

baseline_case1_transcript = (
    research_cases[0]["transcript"]
)

baseline_case1_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 1"
    ]["gold_output"]
)

baseline_case1_model = (
    baseline_output_case1
)


# --------------------------------------------------
# 2. DISPLAY SOURCE TRANSCRIPT
# --------------------------------------------------

print("=" * 80)
print("BASELINE - CASE 1")
print("HARMONIZED SEMANTIC REVIEW")
print("=" * 80)


print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    baseline_case1_transcript
)


# --------------------------------------------------
# 3. DISPLAY SAME FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        baseline_case1_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 4. DISPLAY ORIGINAL FROZEN BASELINE OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        baseline_case1_model,
        indent=2
    )
)

BASELINE - CASE 1
HARMONIZED SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi , andrew . how are you ?
[patient] hey , good to see you .
[doctor] i'm doing well , i'm doing well .
[patient] good .
[doctor] so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] uh , so , andrew is a 59-year-old male with a past medical history , significant for depression , type two diabetes , and hypertension who presents today with an upper respiratory infection . so , andrew , what's going on ?
[patient] yeah . we were doing a bit of work out in the yard in the last week or so and i started to feel really tired , was short of breath . um , we- we're not wearing masks as much at the end of the summer and i think i caught my first cold and i think it just got worse .
[doctor] okay . all right . um , now , have you had your covid vaccines ?
[patient] yeah , both .
[doctor] okay . all right . and , um , do you have any history of any seasonal alle

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 1
# HARMONIZED MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


baseline_case1_semantic_review = [

    # --------------------------------------------------
    # PARTIAL EXTRACTIONS
    # --------------------------------------------------

    {
        "field": "symptoms.fatigue.duration",
        "model_value": None,
        "gold_value": "about one week",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The model captures fatigue, but omits the "
            "approximately one-week duration."
    },

    {
        "field": "symptoms.shortness_of_breath.duration",
        "model_value": None,
        "gold_value": "about one week",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The model captures shortness of breath and its "
            "exertional trigger, but omits the approximately "
            "one-week duration."
    },

    {
        "field": "symptoms.elbow_pain",
        "model_value": "Elbow pain",
        "gold_value": "bilateral elbow pain",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "Elbow pain is captured, but the bilateral "
            "laterality supported by the transcript is lost."
    },

    {
        "field": "symptoms.knee_discomfort",
        "model_value": "Knee tightness and tiredness",
        "gold_value": "bilateral knee discomfort",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The knee complaint is captured, but the transcript "
            "refers to the knees bilaterally and that specificity "
            "is not preserved."
    },

    {
        "field": "plan.medication_changes.lisinopril",
        "model_value": {
            "action": "refill",
            "dosage": "20 mg daily"
        },
        "gold_value": {
            "action": ["continue"],
            "details": "continue 20 mg once daily"
        },
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The refill is correctly captured, but the explicit "
            "instruction to continue lisinopril is not preserved "
            "as an action."
    },


    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "plan.patient_instructions.glucose_monitoring",
        "model_value":
            "Continue diet and blood sugar monitoring",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The transcript says the patient is already monitoring "
            "diet and blood sugar. The clinician discusses this "
            "during history-taking but does not explicitly issue "
            "it as a new plan instruction."
    },

    {
        "field": "plan.patient_instructions.bp_monitoring",
        "model_value":
            "Continue home blood pressure monitoring",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Home blood-pressure monitoring is discussed as "
            "existing patient behavior, but the model promotes "
            "it into an explicit plan instruction."
    },


    # --------------------------------------------------
    # UNSUPPORTED INFERENCE
    # --------------------------------------------------

    {
        "field": "plan.medication_changes.ibuprofen_tylenol",
        "model_value":
            "Ibuprofen / Tylenol as needed for fever or pain",
        "gold_value":
            "Ibuprofen or Tylenol as needed if fever develops",
        "category": "unsupported_inference",
        "count_as_error": 1,
        "reason":
            "The clinician recommends ibuprofen or Tylenol "
            "for fever. Adding pain as an indication is plausible "
            "but is not explicitly stated in this plan."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRAS
    # DO NOT COUNT AS ERRORS
    # --------------------------------------------------

    {
        "field": "medical_history.seasonal_allergies",
        "model_value": "denied",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly denies seasonal allergies."
    },

    {
        "field": "medical_history.covid_vaccination",
        "model_value": "two doses completed",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly reports receiving both "
            "COVID vaccine doses."
    },

    {
        "field": "symptoms.productive_cough",
        "model_value": "denied",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "When asked whether he is coughing anything up, "
            "the patient says not yet."
    },

    {
        "field": "physical_exam.vitals",
        "model_value": "Normal, afebrile",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician explicitly states that the vital "
            "signs are normal and the patient has no fever."
    },

    {
        "field": "physical_exam.heart",
        "model_value": "Heart sounds nice and strong",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "This finding is explicitly stated during "
            "the physical examination."
    },


    # --------------------------------------------------
    # AMBIGUITIES
    # DO NOT COUNT AS ERRORS
    # --------------------------------------------------

    {
        "field": "assessment.viral_syndrome.status",
        "model_value": "active",
        "gold_value": "suspected",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The baseline schema uses a clinical status field "
            "rather than the frozen schema's certainty field. "
            "Active should not automatically be interpreted "
            "as confirmed certainty."
    },

    {
        "field": "diagnostic_tests.review_status",
        "model_value": "completed",
        "gold_value": "reviewed",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The chest X-ray and A1c were completed and their "
            "results were reviewed. Completed is not factually "
            "wrong, although Schema V2 represents this as reviewed."
    }
]


# --------------------------------------------------
# CREATE REVIEW DATAFRAME
# --------------------------------------------------

baseline_case1_review_df = pd.DataFrame(
    baseline_case1_semantic_review
)


print("=" * 80)
print("BASELINE - CASE 1")
print("HARMONIZED SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    baseline_case1_review_df
)


# --------------------------------------------------
# CONFIRMED ERRORS ONLY
# --------------------------------------------------

confirmed_baseline_errors_case1 = (
    baseline_case1_review_df[
        baseline_case1_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_baseline_errors_case1[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(
        confirmed_baseline_errors_case1
    )
)

BASELINE - CASE 1
HARMONIZED SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,symptoms.fatigue.duration,None,about one week,partial_extraction,1,"The model captures fatigue, but omits the appr..."
1,symptoms.shortness_of_breath.duration,None,about one week,partial_extraction,1,The model captures shortness of breath and its...
2,symptoms.elbow_pain,Elbow pain,bilateral elbow pain,partial_extraction,1,"Elbow pain is captured, but the bilateral late..."
3,symptoms.knee_discomfort,Knee tightness and tiredness,bilateral knee discomfort,partial_extraction,1,"The knee complaint is captured, but the transc..."
4,plan.medication_changes.lisinopril,"{'action': 'refill', 'dosage': '20 mg daily'}","{'action': ['continue'], 'details': 'continue ...",partial_extraction,1,"The refill is correctly captured, but the expl..."
5,plan.patient_instructions.glucose_monitoring,Continue diet and blood sugar monitoring,None,mapping_error,1,The transcript says the patient is already mon...
6,plan.patient_instructions.bp_monitoring,Continue home blood pressure monitoring,None,mapping_error,1,Home blood-pressure monitoring is discussed as...
7,plan.medication_changes.ibuprofen_tylenol,Ibuprofen / Tylenol as needed for fever or pain,Ibuprofen or Tylenol as needed if fever develops,unsupported_inference,1,The clinician recommends ibuprofen or Tylenol ...
8,medical_history.seasonal_allergies,denied,None,supported_extra,0,The patient explicitly denies seasonal allergies.
9,medical_history.covid_vaccination,two doses completed,None,supported_extra,0,The patient explicitly reports receiving both ...



CONFIRMED SEMANTIC ERRORS:
category
partial_extraction       5
mapping_error            2
unsupported_inference    1
Name: count, dtype: int64

Total confirmed semantic errors: 8


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 1
# SAVE HARMONIZED FINAL RESULT
# --------------------------------------------------

baseline_case1_harmonized_result = {

    "case_id": "D2N088-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "baseline",


    # --------------------------------------------------
    # STRUCTURAL
    # VALIDATED AGAINST FROZEN SCHEMA V2
    # --------------------------------------------------

    "json_valid": 1,

    "schema_valid": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "schema_valid"
        ].iloc[0]
    ),

    "schema_error_count": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "schema_error_count"
        ].iloc[0]
    ),

    "required_field_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "required_field_errors"
        ].iloc[0]
    ),

    "additional_property_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "additional_property_errors"
        ].iloc[0]
    ),

    "type_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "type_errors"
        ].iloc[0]
    ),

    "enum_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N088-virtassist",
            "enum_errors"
        ].iloc[0]
    ),


    # --------------------------------------------------
    # HARMONIZED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 5,

    "confirmed_mapping_errors": 2,

    "confirmed_unsupported_inferences": 1,

    "confirmed_omissions": 0,

    "confirmed_semantic_errors": 8,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 5,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 2
}


print("=" * 80)
print("BASELINE - CASE 1 HARMONIZED FINAL RESULT")
print("=" * 80)

for key, value in baseline_case1_harmonized_result.items():
    print(f"{key}: {value}")

BASELINE - CASE 1 HARMONIZED FINAL RESULT
case_id: D2N088-virtassist
model: gemini-3.6-flash
condition: baseline
json_valid: 1
schema_valid: 0
schema_error_count: 83
required_field_errors: 55
additional_property_errors: 16
type_errors: 5
enum_errors: 7
confirmed_status_errors: 0
confirmed_partial_extractions: 5
confirmed_mapping_errors: 2
confirmed_unsupported_inferences: 1
confirmed_omissions: 0
confirmed_semantic_errors: 8
supported_extra_items: 5
mapping_ambiguities: 0
annotation_ambiguities: 2


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 2
# PREPARE HARMONIZED SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. LOAD SOURCE / GOLD / ORIGINAL BASELINE OUTPUT
# --------------------------------------------------

baseline_case2_transcript = (
    research_cases[1]["transcript"]
)

baseline_case2_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 2"
    ]["gold_output"]
)

baseline_case2_model = (
    baseline_output_case2
)


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("BASELINE - CASE 2")
print("HARMONIZED SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    baseline_case2_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        baseline_case2_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. ORIGINAL BASELINE MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        baseline_case2_model,
        indent=2
    )
)

BASELINE - CASE 2
HARMONIZED SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year-old female with a past medical history significant for rheumatoid arthritis , atrial fibrillation , and reflux who presents today for her annual exam . so andrea , it's been a year since i saw you . how are you doing ?
[patient] i'm doing well . so , i've been walking like you told me to and , um , exercising and doing yoga , and that's actually helped with my arthritis a lot , just the- the constant movement . so , i have n't had any joint pain recently .
[doctor] okay . good . so , no- no issues with any stiffness or pain or flare ups over the last year ?
[patient] no .
[doctor] okay . and i know that we have you on the methotrexate , are you still taking that once a week ?
[pa

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 2
# HARMONIZED MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


baseline_case2_semantic_review = [

    # --------------------------------------------------
    # PARTIAL EXTRACTION
    # --------------------------------------------------

    {
        "field": "symptoms.palpitations.duration",
        "model_value": "Palpitations",
        "gold_value": "last episode about one week ago",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The model captures the palpitations but omits "
            "the explicit timing that the last episode "
            "occurred about one week ago."
    },


    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "symptoms.right_elbow_pain",
        "model_value": "Right elbow pain",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Right elbow pain is elicited during the physical "
            "examination as pain to palpation. The model also "
            "places it in the symptoms section."
    },

    {
        "field": "plan.methotrexate_continue",
        "model_value":
            "Continue Methotrexate 2.5 mg once weekly "
            "under patient_instructions",
        "gold_value":
            "continue + refill under medication_changes",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly continues methotrexate. "
            "The baseline captures the continuation, but places "
            "it under patient instructions rather than the "
            "medication-change representation."
    },

    {
        "field": "plan.protonix_continue",
        "model_value":
            "Continue Protonix 40 mg daily "
            "under patient_instructions",
        "gold_value":
            "continue under medication_changes",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly continues Protonix, "
            "but the baseline places this medication action "
            "under patient instructions rather than "
            "medication changes."
    },


    # --------------------------------------------------
    # OMISSIONS
    # --------------------------------------------------

    {
        "field": "symptoms.joint_pain",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies joint pain."
    },

    {
        "field": "symptoms.joint_stiffness",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies stiffness "
            "over the previous year."
    },

    {
        "field": "symptoms.chest_pain",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies chest pain."
    },

    {
        "field": "symptoms.shortness_of_breath",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies shortness of breath."
    },

    {
        "field": "symptoms.nausea",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies nausea."
    },

    {
        "field": "symptoms.vomiting",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies vomiting."
    },

    {
        "field": "plan.patient_instructions.reflux_followup",
        "model_value": None,
        "gold_value":
            "contact clinician if reflux symptoms "
            "or other issues recur",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly asks the patient to "
            "let them know if additional reflux-related "
            "issues occur, but this instruction is missing."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRA
    # --------------------------------------------------

    {
        "field": "medical_history.allergies",
        "model_value": "Allergies",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly attributes the nasal "
            "congestion to allergies."
    },

    {
        "field": "plan.rheumatology_contact",
        "model_value":
            "Contact clinic if referral to rheumatologist "
            "is needed",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician explicitly says to let them know "
            "if a rheumatology referral is needed."
    },


    # --------------------------------------------------
    # AMBIGUITY - DO NOT COUNT
    # --------------------------------------------------

    {
        "field": "plan.dietary_modifications.soda",
        "model_value":
            "avoid coffee, spicy foods, soda",
        "gold_value":
            "avoid coffee and spicy foods",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The patient previously reports eliminating soda, "
            "and the clinician later says to continue dietary "
            "modifications. Including soda is therefore "
            "defensible and should not be penalized."
    }
]


# --------------------------------------------------
# CREATE DATAFRAME
# --------------------------------------------------

baseline_case2_review_df = pd.DataFrame(
    baseline_case2_semantic_review
)


print("=" * 80)
print("BASELINE - CASE 2")
print("HARMONIZED SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    baseline_case2_review_df
)


# --------------------------------------------------
# CONFIRMED ERRORS
# --------------------------------------------------

confirmed_baseline_errors_case2 = (
    baseline_case2_review_df[
        baseline_case2_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_baseline_errors_case2[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(
        confirmed_baseline_errors_case2
    )
)

BASELINE - CASE 2
HARMONIZED SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,symptoms.palpitations.duration,Palpitations,last episode about one week ago,partial_extraction,1,The model captures the palpitations but omits ...
1,symptoms.right_elbow_pain,Right elbow pain,None,mapping_error,1,Right elbow pain is elicited during the physic...
2,plan.methotrexate_continue,Continue Methotrexate 2.5 mg once weekly under...,continue + refill under medication_changes,mapping_error,1,The clinician explicitly continues methotrexat...
3,plan.protonix_continue,Continue Protonix 40 mg daily under patient_in...,continue under medication_changes,mapping_error,1,"The clinician explicitly continues Protonix, b..."
4,symptoms.joint_pain,None,denied,omission,1,The patient explicitly denies joint pain.
5,symptoms.joint_stiffness,None,denied,omission,1,The patient explicitly denies stiffness over t...
6,symptoms.chest_pain,None,denied,omission,1,The patient explicitly denies chest pain.
7,symptoms.shortness_of_breath,None,denied,omission,1,The patient explicitly denies shortness of bre...
8,symptoms.nausea,None,denied,omission,1,The patient explicitly denies nausea.
9,symptoms.vomiting,None,denied,omission,1,The patient explicitly denies vomiting.



CONFIRMED SEMANTIC ERRORS:
category
omission              7
mapping_error         3
partial_extraction    1
Name: count, dtype: int64

Total confirmed semantic errors: 11


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 2
# SAVE HARMONIZED FINAL RESULT
# --------------------------------------------------

baseline_case2_harmonized_result = {

    "case_id": "D2N089-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "baseline",


    # --------------------------------------------------
    # STRUCTURAL
    # VALIDATED AGAINST FROZEN SCHEMA V2
    # --------------------------------------------------

    "json_valid": 1,

    "schema_valid": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "schema_valid"
        ].iloc[0]
    ),

    "schema_error_count": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "schema_error_count"
        ].iloc[0]
    ),

    "required_field_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "required_field_errors"
        ].iloc[0]
    ),

    "additional_property_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "additional_property_errors"
        ].iloc[0]
    ),

    "type_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "type_errors"
        ].iloc[0]
    ),

    "enum_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N089-virtassist",
            "enum_errors"
        ].iloc[0]
    ),


    # --------------------------------------------------
    # HARMONIZED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 1,

    "confirmed_mapping_errors": 3,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 7,

    "confirmed_semantic_errors": 11,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 2,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 1
}


print("=" * 80)
print("BASELINE - CASE 2 HARMONIZED FINAL RESULT")
print("=" * 80)

for key, value in baseline_case2_harmonized_result.items():
    print(f"{key}: {value}")

BASELINE - CASE 2 HARMONIZED FINAL RESULT
case_id: D2N089-virtassist
model: gemini-3.6-flash
condition: baseline
json_valid: 1
schema_valid: 0
schema_error_count: 24
required_field_errors: 2
additional_property_errors: 1
type_errors: 21
enum_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 1
confirmed_mapping_errors: 3
confirmed_unsupported_inferences: 0
confirmed_omissions: 7
confirmed_semantic_errors: 11
supported_extra_items: 2
mapping_ambiguities: 0
annotation_ambiguities: 1


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 3
# PREPARE HARMONIZED SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. LOAD SOURCE / GOLD / ORIGINAL BASELINE OUTPUT
# --------------------------------------------------

baseline_case3_transcript = (
    research_cases[2]["transcript"]
)

baseline_case3_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 3"
    ]["gold_output"]
)

baseline_case3_model = (
    baseline_output_case3
)


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("BASELINE - CASE 3")
print("HARMONIZED SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    baseline_case3_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        baseline_case3_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. ORIGINAL BASELINE MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        baseline_case3_model,
        indent=2
    )
)

BASELINE - CASE 3
HARMONIZED SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi , albert . how are you ?
[patient] hey , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] so , albert is a 62-year-old male , with a past medical history significant for depression , type 2 diabetes , and kidney transplant , who is here today for emergency room follow-up .
[patient] mm-hmm .
[doctor] so , i got a notification that you were in the emergency room , but , but what were you there for ?
[patient] well , i , uh , i was n't really , uh , staying on top of my , uh , blood sugar readings , and i felt kinda woozy over the weekend . and i was little concerned , and my wife wanted to take me in and just have me checked out .
[doctor] okay . and , and was it , in fact , high ?
[patient] yeah , it was .
[doctor] okay . did you ... were you admitted to the hospital ?
[patient] uh , no .
[doc

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 3
# HARMONIZED MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


baseline_case3_semantic_review = [

    # --------------------------------------------------
    # PARTIAL EXTRACTIONS
    # --------------------------------------------------

    {
        "field": "symptoms.wooziness.status",
        "model_value": "Wooziness",
        "gold_value": "resolved",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The model captures wooziness but does not preserve "
            "that the symptom had resolved by the time of the visit."
    },

    {
        "field": "plan.tests_ordered.a1c_timing",
        "model_value": "Hemoglobin A1c",
        "gold_value":
            "repeat hemoglobin A1c in a couple of months",
        "category": "partial_extraction",
        "count_as_error": 1,
        "reason":
            "The repeat A1c order is captured, but its explicit "
            "timing of a couple of months is omitted."
    },


    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "symptoms.elevated_blood_sugar",
        "model_value": "Elevated blood sugar",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Elevated blood glucose is supported by the transcript, "
            "but it is a measured condition/lab finding rather than "
            "a symptom and is placed in the wrong section."
    },

    {
        "field": "plan.patient_instructions.diet",
        "model_value": "Follow diet closely",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Diet adherence is discussed during history-taking, "
            "but the clinician does not explicitly issue "
            "'follow diet closely' as a plan instruction."
    },

    {
        "field": "follow_up",
        "model_value": {
            "value": 2,
            "unit": "months",
            "condition":
                "for repeat Hemoglobin A1c test"
        },
        "gold_value": {
            "value": None,
            "unit": None,
            "condition": None,
            "timing_text": None
        },
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The transcript gives a couple-of-month timing for "
            "the repeat A1c test, not for an office follow-up visit. "
            "The model moves test timing into encounter follow-up."
    },


    # --------------------------------------------------
    # OMISSIONS
    # --------------------------------------------------

    {
        "field": "symptoms.chest_pain",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies chest pain."
    },

    {
        "field": "symptoms.shortness_of_breath",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies shortness of breath."
    },

    {
        "field": "symptoms.lightheadedness",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies lightheadedness."
    },

    {
        "field": "symptoms.dizziness",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies dizziness."
    },

    {
        "field":
            "plan.patient_instructions.contact_clinician",
        "model_value": None,
        "gold_value":
            "contact clinician if additional help is needed",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "During the depression plan, the clinician explicitly "
            "reminds the patient to call if additional help "
            "is needed, but the baseline leaves this out."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRAS
    # --------------------------------------------------

    {
        "field": "physical_exam.vital_signs",
        "model_value":
            "Pulse oximetry, blood pressure, and heart rate "
            "are within normal limits",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician explicitly states that the vital "
            "signs, pulse oximetry, blood pressure, and heart "
            "rate look good."
    },


    # --------------------------------------------------
    # AMBIGUITIES / DO NOT COUNT
    # --------------------------------------------------

    {
        "field":
            "medications.immunosuppression.action",
        "model_value":
            "Immunotherapy / immunosuppression medications",
        "gold_value": ["continue"],
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The transcript confirms ongoing use and specialist "
            "management of these medications, but the clinician "
            "does not clearly give a new explicit 'continue' "
            "instruction during the plan."
    },

    {
        "field": "plan.specialist_follow_up",
        "model_value":
            "Follow-up with Dr. Reyes for immunosuppression "
            "medication management",
        "gold_value":
            "continue follow-up with transplant specialist",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The baseline captures the clinically correct "
            "specialist follow-up. Its use of the old 'referrals' "
            "field is already a structural-schema issue and "
            "should not be counted again as a semantic error."
    }
]


# --------------------------------------------------
# DATAFRAME
# --------------------------------------------------

baseline_case3_review_df = pd.DataFrame(
    baseline_case3_semantic_review
)


print("=" * 80)
print("BASELINE - CASE 3")
print("HARMONIZED SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    baseline_case3_review_df
)


# --------------------------------------------------
# CONFIRMED ERRORS
# --------------------------------------------------

confirmed_baseline_errors_case3 = (
    baseline_case3_review_df[
        baseline_case3_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_baseline_errors_case3[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(
        confirmed_baseline_errors_case3
    )
)

BASELINE - CASE 3
HARMONIZED SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,symptoms.wooziness.status,Wooziness,resolved,partial_extraction,1,The model captures wooziness but does not pres...
1,plan.tests_ordered.a1c_timing,Hemoglobin A1c,repeat hemoglobin A1c in a couple of months,partial_extraction,1,"The repeat A1c order is captured, but its expl..."
2,symptoms.elevated_blood_sugar,Elevated blood sugar,None,mapping_error,1,Elevated blood glucose is supported by the tra...
3,plan.patient_instructions.diet,Follow diet closely,None,mapping_error,1,Diet adherence is discussed during history-tak...
4,follow_up,"{'value': 2, 'unit': 'months', 'condition': 'f...","{'value': None, 'unit': None, 'condition': Non...",mapping_error,1,The transcript gives a couple-of-month timing ...
5,symptoms.chest_pain,None,denied,omission,1,The patient explicitly denies chest pain.
6,symptoms.shortness_of_breath,None,denied,omission,1,The patient explicitly denies shortness of bre...
7,symptoms.lightheadedness,None,denied,omission,1,The patient explicitly denies lightheadedness.
8,symptoms.dizziness,None,denied,omission,1,The patient explicitly denies dizziness.
9,plan.patient_instructions.contact_clinician,None,contact clinician if additional help is needed,omission,1,"During the depression plan, the clinician expl..."



CONFIRMED SEMANTIC ERRORS:
category
omission              5
mapping_error         3
partial_extraction    2
Name: count, dtype: int64

Total confirmed semantic errors: 10


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 3
# SAVE HARMONIZED FINAL RESULT
# --------------------------------------------------

baseline_case3_harmonized_result = {

    "case_id": "D2N090-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "baseline",


    # --------------------------------------------------
    # STRUCTURAL
    # VALIDATED AGAINST FROZEN SCHEMA V2
    # --------------------------------------------------

    "json_valid": 1,

    "schema_valid": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "schema_valid"
        ].iloc[0]
    ),

    "schema_error_count": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "schema_error_count"
        ].iloc[0]
    ),

    "required_field_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "required_field_errors"
        ].iloc[0]
    ),

    "additional_property_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "additional_property_errors"
        ].iloc[0]
    ),

    "type_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "type_errors"
        ].iloc[0]
    ),

    "enum_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N090-virtassist",
            "enum_errors"
        ].iloc[0]
    ),


    # --------------------------------------------------
    # HARMONIZED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 2,

    "confirmed_mapping_errors": 3,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 5,

    "confirmed_semantic_errors": 10,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 1,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 2
}


print("=" * 80)
print("BASELINE - CASE 3 HARMONIZED FINAL RESULT")
print("=" * 80)

for key, value in baseline_case3_harmonized_result.items():
    print(f"{key}: {value}")

BASELINE - CASE 3 HARMONIZED FINAL RESULT
case_id: D2N090-virtassist
model: gemini-3.6-flash
condition: baseline
json_valid: 1
schema_valid: 0
schema_error_count: 27
required_field_errors: 2
additional_property_errors: 1
type_errors: 24
enum_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 2
confirmed_mapping_errors: 3
confirmed_unsupported_inferences: 0
confirmed_omissions: 5
confirmed_semantic_errors: 10
supported_extra_items: 1
mapping_ambiguities: 0
annotation_ambiguities: 2


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 4
# PREPARE HARMONIZED SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. LOAD SOURCE / GOLD / ORIGINAL BASELINE OUTPUT
# --------------------------------------------------

baseline_case4_transcript = (
    research_cases[3]["transcript"]
)

baseline_case4_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 4"
    ]["gold_output"]
)

baseline_case4_model = (
    baseline_output_case4
)


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("BASELINE - CASE 4")
print("HARMONIZED SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    baseline_case4_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        baseline_case4_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. ORIGINAL BASELINE MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        baseline_case4_model,
        indent=2
    )
)

BASELINE - CASE 4
HARMONIZED SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hi jerry , how are you doing ?
[patient] hi , good to see you .
[doctor] good to see you as well . um , so i know that the nurse told you about dax . i'd like to tell dax about you .
[patient] sure .
[doctor] jerry is a 54 year old male with a past medical history , significant for osteoporosis and multiple sclerosis who presents for an annual exam . so jerry , what's been going on since the last time i saw you ?
[patient] uh , we have been traveling all over the country . it's been kind of a stressful summer . kinda adjusting to everything in the fall and so far it's been good , but ah , lack of sleep , it's been really getting to me .
[doctor] okay . all right . and have you taken anything for the insomnia . have you tried any strategies for it .
[patient] i've tried everything from melatonin to meditation to , uh , t- stretching out every morning when i get up . nothing really seems to help though .
[doctor] ok

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 4
# HARMONIZED MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


baseline_case4_semantic_review = [

    # --------------------------------------------------
    # MAPPING ERRORS
    # --------------------------------------------------

    {
        "field": "symptoms.lower_extremity_weakness",
        "model_value": "Lower extremity weakness",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Lower-extremity weakness is documented during the "
            "physical examination with right 4/5 and left 3/5 "
            "strength. The model additionally places it in symptoms."
    },

    {
        "field": "assessment.insomnia",
        "model_value": "Insomnia",
        "gold_value": None,
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Insomnia is clearly reported as a symptom, but the "
            "explicit assessment and plan addresses osteoporosis "
            "and multiple sclerosis rather than formally assessing "
            "insomnia."
    },

    {
        "field": "plan.fosamax_continue",
        "model_value":
            "Continue taking Fosamax under patient_instructions",
        "gold_value":
            "continue + refill under medication_changes",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The continuation of Fosamax is clinically correct, "
            "but the continuation action is placed under patient "
            "instructions rather than medication changes."
    },

    {
        "field": "plan.ms_medication_continue",
        "model_value":
            "Continue multiple sclerosis medications "
            "under patient_instructions",
        "gold_value":
            "continue under medication_changes",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly continues the MS medications, "
            "but the baseline places the medication action under "
            "patient instructions."
    },

    {
        "field": "plan.neurology_follow_up",
        "model_value":
            "Continue seeing the neurologist "
            "under patient_instructions",
        "gold_value":
            "neurology - continue_follow_up",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Continuing neurology follow-up is correctly understood, "
            "but it is placed under patient instructions rather than "
            "the specialist-follow-up representation."
    },


    # --------------------------------------------------
    # OMISSIONS
    # --------------------------------------------------

    {
        "field": "symptoms.chest_pain",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies chest pain."
    },

    {
        "field": "symptoms.shortness_of_breath",
        "model_value": None,
        "gold_value": "denied",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The patient explicitly denies shortness of breath."
    },

    {
        "field": "plan.patient_instructions.contact_clinician",
        "model_value": None,
        "gold_value":
            "contact clinician if additional help is "
            "needed for multiple sclerosis",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly tells the patient to let "
            "them know if anything else is needed regarding the "
            "multiple sclerosis, but this instruction is omitted."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRAS
    # --------------------------------------------------

    {
        "field": "medical_history.right_knee_surgery",
        "model_value": "Right knee surgery",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly refers to a lingering issue "
            "related to prior knee surgery."
    },

    {
        "field": "physical_exam.vital_signs",
        "model_value": "Vital signs: normal / good",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The clinician explicitly states that the vital "
            "signs look very good."
    },


    # --------------------------------------------------
    # AMBIGUITIES
    # DO NOT COUNT AS ERRORS
    # --------------------------------------------------

    {
        "field": "encounter.chief_complaint",
        "model_value": "Lack of sleep / insomnia",
        "gold_value": "annual exam",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The formal encounter is an annual exam, but lack of "
            "sleep is the principal concern raised by the patient. "
            "Both representations are defensible."
    },

    {
        "field": "medications.melatonin",
        "model_value": "Melatonin",
        "gold_value": None,
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The patient explicitly says that melatonin was tried. "
            "Because the baseline medication representation has no "
            "status field, it is unclear whether this entry means "
            "current or historical use, so it should not be "
            "penalized as a status error."
    },

    {
        "field": "symptoms.stress",
        "model_value": "Stress",
        "gold_value": None,
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The patient describes the summer as stressful. "
            "Representing this as a clinical symptom is debatable, "
            "so it is kept as an ambiguity rather than a confirmed "
            "mapping error."
    }
]


# --------------------------------------------------
# CREATE DATAFRAME
# --------------------------------------------------

baseline_case4_review_df = pd.DataFrame(
    baseline_case4_semantic_review
)


print("=" * 80)
print("BASELINE - CASE 4")
print("HARMONIZED SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    baseline_case4_review_df
)


# --------------------------------------------------
# CONFIRMED ERRORS
# --------------------------------------------------

confirmed_baseline_errors_case4 = (
    baseline_case4_review_df[
        baseline_case4_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_baseline_errors_case4[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(
        confirmed_baseline_errors_case4
    )
)

BASELINE - CASE 4
HARMONIZED SEMANTIC ADJUDICATION


,field,model_value,gold_value,category,count_as_error,reason
0,symptoms.lower_extremity_weakness,Lower extremity weakness,None,mapping_error,1,Lower-extremity weakness is documented during ...
1,assessment.insomnia,Insomnia,None,mapping_error,1,"Insomnia is clearly reported as a symptom, but..."
2,plan.fosamax_continue,Continue taking Fosamax under patient_instruct...,continue + refill under medication_changes,mapping_error,1,The continuation of Fosamax is clinically corr...
3,plan.ms_medication_continue,Continue multiple sclerosis medications under ...,continue under medication_changes,mapping_error,1,The clinician explicitly continues the MS medi...
4,plan.neurology_follow_up,Continue seeing the neurologist under patient_...,neurology - continue_follow_up,mapping_error,1,Continuing neurology follow-up is correctly un...
5,symptoms.chest_pain,None,denied,omission,1,The patient explicitly denies chest pain.
6,symptoms.shortness_of_breath,None,denied,omission,1,The patient explicitly denies shortness of bre...
7,plan.patient_instructions.contact_clinician,None,contact clinician if additional help is needed...,omission,1,The clinician explicitly tells the patient to ...
8,medical_history.right_knee_surgery,Right knee surgery,None,supported_extra,0,The patient explicitly refers to a lingering i...
9,physical_exam.vital_signs,Vital signs: normal / good,None,supported_extra,0,The clinician explicitly states that the vital...



CONFIRMED SEMANTIC ERRORS:
category
mapping_error    5
omission         3
Name: count, dtype: int64

Total confirmed semantic errors: 8


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 4
# SAVE HARMONIZED FINAL RESULT
# --------------------------------------------------

baseline_case4_harmonized_result = {

    "case_id": "D2N091-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "baseline",


    # --------------------------------------------------
    # STRUCTURAL
    # VALIDATED AGAINST FROZEN SCHEMA V2
    # --------------------------------------------------

    "json_valid": 1,

    "schema_valid": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "schema_valid"
        ].iloc[0]
    ),

    "schema_error_count": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "schema_error_count"
        ].iloc[0]
    ),

    "required_field_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "required_field_errors"
        ].iloc[0]
    ),

    "additional_property_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "additional_property_errors"
        ].iloc[0]
    ),

    "type_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "type_errors"
        ].iloc[0]
    ),

    "enum_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N091-virtassist",
            "enum_errors"
        ].iloc[0]
    ),


    # --------------------------------------------------
    # HARMONIZED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 0,

    "confirmed_mapping_errors": 5,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 3,

    "confirmed_semantic_errors": 8,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 2,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 3
}


print("=" * 80)
print("BASELINE - CASE 4 HARMONIZED FINAL RESULT")
print("=" * 80)

for key, value in baseline_case4_harmonized_result.items():
    print(f"{key}: {value}")

BASELINE - CASE 4 HARMONIZED FINAL RESULT
case_id: D2N091-virtassist
model: gemini-3.6-flash
condition: baseline
json_valid: 1
schema_valid: 0
schema_error_count: 24
required_field_errors: 2
additional_property_errors: 1
type_errors: 21
enum_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 0
confirmed_mapping_errors: 5
confirmed_unsupported_inferences: 0
confirmed_omissions: 3
confirmed_semantic_errors: 8
supported_extra_items: 2
mapping_ambiguities: 0
annotation_ambiguities: 3


In [ ]:
# --------------------------------------------------
# BASELINE - CASE 5
# PREPARE HARMONIZED SEMANTIC REVIEW
# --------------------------------------------------

import json


# --------------------------------------------------
# 1. LOAD SOURCE / GOLD / ORIGINAL BASELINE OUTPUT
# --------------------------------------------------

baseline_case5_transcript = (
    research_cases[4]["transcript"]
)

baseline_case5_gold_v2 = (
    gold_annotations_v2_frozen[
        "Case 5"
    ]["gold_output"]
)

baseline_case5_model = (
    baseline_output_case5
)


# --------------------------------------------------
# 2. HEADER
# --------------------------------------------------

print("=" * 80)
print("BASELINE - CASE 5")
print("HARMONIZED SEMANTIC REVIEW")
print("=" * 80)


# --------------------------------------------------
# 3. SOURCE TRANSCRIPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE TRANSCRIPT")
print("=" * 80)

print(
    baseline_case5_transcript
)


# --------------------------------------------------
# 4. FROZEN GOLD V2
# --------------------------------------------------

print("\n" + "=" * 80)
print("FROZEN GOLD V2")
print("=" * 80)

print(
    json.dumps(
        baseline_case5_gold_v2,
        indent=2
    )
)


# --------------------------------------------------
# 5. ORIGINAL BASELINE MODEL OUTPUT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE MODEL OUTPUT")
print("=" * 80)

print(
    json.dumps(
        baseline_case5_model,
        indent=2
    )
)

BASELINE - CASE 5
HARMONIZED SEMANTIC REVIEW

SOURCE TRANSCRIPT
[doctor] hello , mrs . martinez . good to see you today .
[patient] hey , dr . gomez .
[doctor] hey , dragon , i'm here seeing mrs . martinez . she's a 43-year-old female . why are we seeing you today ?
[patient] um , my arm hurts right here . kind of toward my wrist . this part of my arm .
[doctor] so you have pain in your distal radius ?
[patient] yes .
[doctor] how did that happen ?
[patient] um , i was playing tennis , and when i went to hit , um , i was given a , a backhand , and when i did , i m- totally missed the ball , hit the top of the net but the pole part . and , and it just jarred my arm .
[doctor] okay . and did it swell up at all ? or-
[patient] it did . it got a ... it had a little bit of swelling . not a lot .
[doctor] okay . and , um , did , uh , do you have any numbness in your hand at all ? or any pain when you move your wrist ?
[patient] a little bit when i move my wrist . um , no numbness in my hand 

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 5
# HARMONIZED MANUAL SEMANTIC ADJUDICATION
# --------------------------------------------------

import pandas as pd


baseline_case5_semantic_review = [

    # --------------------------------------------------
    # MAPPING ERROR
    # --------------------------------------------------

    {
        "field": "physical_exam.hand_numbness",
        "model_value": "No hand numbness",
        "gold_value": "numbness denied under symptoms",
        "category": "mapping_error",
        "count_as_error": 1,
        "reason":
            "Hand numbness is explicitly asked about during "
            "symptom/history collection and denied by the patient. "
            "The baseline captures the fact but places it under "
            "physical_exam instead of symptoms."
    },


    # --------------------------------------------------
    # OMISSIONS
    # --------------------------------------------------

    {
        "field": "medications.Motrin",
        "model_value": None,
        "gold_value":
            "Motrin 800 mg three times daily with food, start",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "Motrin is prescribed during the encounter and its "
            "start instruction is captured in the plan, but the "
            "top-level medication list omits the medication."
    },

    {
        "field": "plan.patient_instructions.conservative_treatment",
        "model_value": None,
        "gold_value": "treat conservatively",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly states that treatment will "
            "be conservative, but this plan instruction is not "
            "preserved."
    },

    {
        "field": "plan.patient_instructions.contact_clinician",
        "model_value": None,
        "gold_value":
            "contact clinician if symptoms do not improve "
            "within about one week",
        "category": "omission",
        "count_as_error": 1,
        "reason":
            "The clinician explicitly says to let them know if "
            "the arm does not improve in the next week or so. "
            "The baseline captures the timing in follow_up but "
            "omits the explicit contact instruction."
    },


    # --------------------------------------------------
    # SUPPORTED EXTRAS
    # --------------------------------------------------

    {
        "field": "medical_history.prior_trauma",
        "model_value":
            "Past trauma/surgery from falling on a rake "
            "while doing lawn work",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The patient explicitly describes a prior event "
            "involving falling on a rake during lawn work."
    },

    {
        "field": "assessment.no_fracture",
        "model_value": "No fracture",
        "gold_value": None,
        "category": "supported_extra",
        "count_as_error": 0,
        "reason":
            "The X-ray shows no fracture and the clinician "
            "explicitly states that there is no fracture."
    },


    # --------------------------------------------------
    # AMBIGUITIES - DO NOT COUNT
    # --------------------------------------------------

    {
        "field": "laterality",
        "model_value":
            "arm / distal radius / wrist / hand without side",
        "gold_value":
            "right arm / right distal radius / right wrist / right hand",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The transcript does not explicitly state right "
            "or left laterality, so the baseline should not "
            "be penalized for avoiding unsupported laterality."
    },

    {
        "field": "assessment.strain_certainty",
        "model_value": "Arm muscle strain",
        "gold_value": "confirmed",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The clinician says 'I think what you have is "
            "basically just a strain.' The old baseline format "
            "does not represent certainty, so this should not "
            "be treated as a status/certainty error."
    },

    {
        "field": "assessment.contusion_certainty",
        "model_value": "Arm muscle contusion",
        "gold_value": "possible",
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The clinician says 'maybe a contusion.' The baseline "
            "captures the condition but its old representation "
            "does not include certainty."
    },

    {
        "field": "medications.Flonase.action",
        "model_value": "Flonase",
        "gold_value": ["continue"],
        "category": "annotation_ambiguity",
        "count_as_error": 0,
        "reason":
            "The patient says she currently takes Flonase, but "
            "there is no explicit new instruction during the visit "
            "to continue it."
    },

    {
        "field": "plan.patient_instructions.Motrin",
        "model_value":
            "Take Motrin 800 mg three times a day with food",
        "gold_value":
            "represented under medication_changes",
        "category": "supported_duplicate",
        "count_as_error": 0,
        "reason":
            "The instruction is explicitly supported. Repeating "
            "it as a patient instruction does not introduce "
            "unsupported clinical content."
    }
]


# --------------------------------------------------
# CREATE DATAFRAME
# --------------------------------------------------

baseline_case5_review_df = pd.DataFrame(
    baseline_case5_semantic_review
)


print("=" * 80)
print("BASELINE - CASE 5")
print("HARMONIZED SEMANTIC ADJUDICATION")
print("=" * 80)

display(
    baseline_case5_review_df
)


# --------------------------------------------------
# CONFIRMED ERRORS
# --------------------------------------------------

confirmed_baseline_errors_case5 = (
    baseline_case5_review_df[
        baseline_case5_review_df[
            "count_as_error"
        ] == 1
    ]
)


print("\nCONFIRMED SEMANTIC ERRORS:")

print(
    confirmed_baseline_errors_case5[
        "category"
    ].value_counts()
)


print(
    "\nTotal confirmed semantic errors:",
    len(
        confirmed_baseline_errors_case5
    )
)

In [ ]:
# --------------------------------------------------
# BASELINE - CASE 5
# SAVE HARMONIZED FINAL RESULT
# --------------------------------------------------

baseline_case5_harmonized_result = {

    "case_id": "D2N092-virtassist",

    "model": "gemini-3.6-flash",

    "condition": "baseline",


    # --------------------------------------------------
    # STRUCTURAL
    # VALIDATED AGAINST FROZEN SCHEMA V2
    # --------------------------------------------------

    "json_valid": 1,

    "schema_valid": int(baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "schema_valid"
        ].iloc[0]
    ),

    "schema_error_count": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "schema_error_count"
        ].iloc[0]
    ),

    "required_field_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "required_field_errors"
        ].iloc[0]
    ),

    "additional_property_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "additional_property_errors"
        ].iloc[0]
    ),

    "type_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "type_errors"
        ].iloc[0]
    ),

    "enum_errors": int(
        baseline_v2_df.loc[
            baseline_v2_df["case_id"]
            == "D2N092-virtassist",
            "enum_errors"
        ].iloc[0]
    ),


    # --------------------------------------------------
    # HARMONIZED SEMANTIC ERRORS
    # --------------------------------------------------

    "confirmed_status_errors": 0,

    "confirmed_partial_extractions": 0,

    "confirmed_mapping_errors": 1,

    "confirmed_unsupported_inferences": 0,

    "confirmed_omissions": 3,

    "confirmed_semantic_errors": 4,


    # --------------------------------------------------
    # OTHER REVIEW FINDINGS
    # --------------------------------------------------

    "supported_extra_items": 2,

    "supported_duplicate_items": 1,

    "mapping_ambiguities": 0,

    "annotation_ambiguities": 4
}


print("=" * 80)
print("BASELINE - CASE 5 HARMONIZED FINAL RESULT")
print("=" * 80)

for key, value in baseline_case5_harmonized_result.items():
    print(f"{key}: {value}")

BASELINE - CASE 5 HARMONIZED FINAL RESULT
case_id: D2N092-virtassist
model: gemini-3.6-flash
condition: baseline
json_valid: 1
schema_valid: 0
schema_error_count: 19
required_field_errors: 2
additional_property_errors: 1
type_errors: 16
enum_errors: 0
confirmed_status_errors: 0
confirmed_partial_extractions: 0
confirmed_mapping_errors: 1
confirmed_unsupported_inferences: 0
confirmed_omissions: 3
confirmed_semantic_errors: 4
supported_extra_items: 2
supported_duplicate_items: 1
mapping_ambiguities: 0
annotation_ambiguities: 4


In [ ]:
# --------------------------------------------------
# BASELINE
# COMBINE ALL 5 HARMONIZED RESULTS
# --------------------------------------------------

import pandas as pd


# --------------------------------------------------
# 1. COMBINE BASELINE CASE RESULTS
# --------------------------------------------------

baseline_harmonized_results = [

  baseline_case1_harmonized_result,
  baseline_case2_harmonized_result,
  baseline_case3_harmonized_result,
  baseline_case4_harmonized_result,
  baseline_case5_harmonized_result

]


baseline_harmonized_df = pd.DataFrame(
    baseline_harmonized_results
)


# --------------------------------------------------
# 2. DISPLAY CASE-LEVEL RESULTS
# --------------------------------------------------

print("=" * 80)
print("BASELINE - HARMONIZED 5-CASE PILOT RESULTS")
print("=" * 80)

display(
    baseline_harmonized_df[
        [
            "case_id",
            "schema_valid",
            "schema_error_count",
            "confirmed_partial_extractions",
            "confirmed_mapping_errors",
            "confirmed_status_errors",
            "confirmed_omissions",
            "confirmed_unsupported_inferences",
            "confirmed_semantic_errors"
        ]
    ]
)


# --------------------------------------------------
# 3. CALCULATE SEMANTIC TOTALS
# --------------------------------------------------

baseline_total_partial = int(
    baseline_harmonized_df[
        "confirmed_partial_extractions"
    ].sum()
)

baseline_total_mapping = int(
    baseline_harmonized_df[
        "confirmed_mapping_errors"
    ].sum()
)

baseline_total_status = int(
    baseline_harmonized_df[
        "confirmed_status_errors"
    ].sum()
)

baseline_total_omissions = int(
    baseline_harmonized_df[
        "confirmed_omissions"
    ].sum()
)

baseline_total_unsupported = int(
    baseline_harmonized_df[
        "confirmed_unsupported_inferences"
    ].sum()
)

baseline_total_semantic_errors = int(
    baseline_harmonized_df[
        "confirmed_semantic_errors"
    ].sum()
)


# --------------------------------------------------
# 4. PRINT BASELINE SUMMARY
# --------------------------------------------------

print("\n" + "=" * 80)
print("BASELINE - HARMONIZED SEMANTIC SUMMARY")
print("=" * 80)

print(
    "Cases evaluated:",
    len(baseline_harmonized_df)
)

print(
    "Partial extractions:",
    baseline_total_partial
)

print(
    "Mapping errors:",
    baseline_total_mapping
)

print(
    "Status/certainty errors:",
    baseline_total_status
)

print(
    "Omissions:",
    baseline_total_omissions
)

print(
    "Unsupported inferences:",
    baseline_total_unsupported
)

print(
    "Total confirmed semantic errors:",
    baseline_total_semantic_errors
)

BASELINE - HARMONIZED 5-CASE PILOT RESULTS


,case_id,schema_valid,schema_error_count,confirmed_partial_extractions,confirmed_mapping_errors,confirmed_status_errors,confirmed_omissions,confirmed_unsupported_inferences,confirmed_semantic_errors
0,D2N088-virtassist,0,83,5,2,0,0,1,8
1,D2N089-virtassist,0,24,1,3,0,7,0,11
2,D2N090-virtassist,0,27,2,3,0,5,0,10
3,D2N091-virtassist,0,24,0,5,0,3,0,8
4,D2N092-virtassist,0,19,0,1,0,3,0,4



BASELINE - HARMONIZED SEMANTIC SUMMARY
Cases evaluated: 5
Partial extractions: 8
Mapping errors: 14
Status/certainty errors: 0
Omissions: 18
Unsupported inferences: 1
Total confirmed semantic errors: 41


In [ ]:
# --------------------------------------------------
# PILOT SEMANTIC COMPARISON
# BASELINE VS SCHEMA-CONSTRAINED
# --------------------------------------------------

import pandas as pd


# --------------------------------------------------
# 1. BUILD ERROR-CATEGORY COMPARISON
# --------------------------------------------------

semantic_comparison = pd.DataFrame({

    "error_category": [
        "Partial extractions",
        "Mapping errors",
        "Status/certainty errors",
        "Omissions",
        "Unsupported inferences",
        "TOTAL semantic errors"
    ],

    "baseline": [
        baseline_total_partial,
        baseline_total_mapping,
        baseline_total_status,
        baseline_total_omissions,
        baseline_total_unsupported,
        baseline_total_semantic_errors
    ],

    "schema_constrained": [
        total_partial,
        total_mapping,
        total_status,
        total_omissions,
        total_unsupported,
        total_confirmed_semantic_errors
    ]
})


# --------------------------------------------------
# 2. CALCULATE ABSOLUTE CHANGE
# --------------------------------------------------

semantic_comparison[
    "absolute_change"
] = (
    semantic_comparison["schema_constrained"]
    - semantic_comparison["baseline"]
)


# --------------------------------------------------
# 3. CALCULATE % REDUCTION
#
# If baseline = 0, percentage reduction is undefined.
# --------------------------------------------------

def calculate_reduction(row):

    if row["baseline"] == 0:
        return None

    return (
        (
            row["baseline"]
            - row["schema_constrained"]
        )
        / row["baseline"]
    ) * 100


semantic_comparison[
    "percent_reduction"
] = semantic_comparison.apply(
    calculate_reduction,
    axis=1
)


# --------------------------------------------------
# 4. DISPLAY CATEGORY COMPARISON
# --------------------------------------------------

print("=" * 80)
print("PILOT SEMANTIC COMPARISON")
print("BASELINE VS SCHEMA-CONSTRAINED")
print("=" * 80)

display(
    semantic_comparison
)


# --------------------------------------------------
# 5. CASE-BY-CASE TOTAL SEMANTIC ERRORS
# --------------------------------------------------

case_level_semantic_comparison = pd.DataFrame({

    "case_id":
        baseline_harmonized_df[
            "case_id"
        ].tolist(),

    "baseline_errors":
        baseline_harmonized_df[
            "confirmed_semantic_errors"
        ].tolist(),

    "schema_constrained_errors":
        schema_constrained_pilot_df[
            "confirmed_semantic_errors"
        ].tolist()

})


case_level_semantic_comparison[
    "error_difference"
] = (
    case_level_semantic_comparison[
        "baseline_errors"
    ]
    -
    case_level_semantic_comparison[
        "schema_constrained_errors"
    ]
)


print("\n" + "=" * 80)
print("CASE-LEVEL SEMANTIC COMPARISON")
print("=" * 80)

display(
    case_level_semantic_comparison
)


# --------------------------------------------------
# 6. OVERALL SEMANTIC REDUCTION
# --------------------------------------------------

overall_semantic_reduction = (
    (
        baseline_total_semantic_errors
        - total_confirmed_semantic_errors
    )
    / baseline_total_semantic_errors
) * 100


print("\n" + "=" * 80)
print("KEY PILOT SEMANTIC FINDINGS")
print("=" * 80)

print(
    "Baseline confirmed semantic errors:",
    baseline_total_semantic_errors
)

print(
    "Schema-constrained confirmed semantic errors:",
    total_confirmed_semantic_errors
)

print(
    "Absolute reduction:",
    baseline_total_semantic_errors
    - total_confirmed_semantic_errors
)

print(
    "Observed semantic-error reduction:",
    f"{overall_semantic_reduction:.1f}%"
)

PILOT SEMANTIC COMPARISON
BASELINE VS SCHEMA-CONSTRAINED


,error_category,baseline,schema_constrained,absolute_change,percent_reduction
0,Partial extractions,8,5,-3,37.500000
1,Mapping errors,14,6,-8,57.142857
2,Status/certainty errors,0,3,3,NaN
3,Omissions,18,2,-16,88.888889
4,Unsupported inferences,1,0,-1,100.000000
5,TOTAL semantic errors,41,16,-25,60.975610



CASE-LEVEL SEMANTIC COMPARISON


,case_id,baseline_errors,schema_constrained_errors,error_difference
0,D2N088-virtassist,8,3,5
1,D2N089-virtassist,11,5,6
2,D2N090-virtassist,10,4,6
3,D2N091-virtassist,8,3,5
4,D2N092-virtassist,4,1,3



KEY PILOT SEMANTIC FINDINGS
Baseline confirmed semantic errors: 41
Schema-constrained confirmed semantic errors: 16
Absolute reduction: 25
Observed semantic-error reduction: 61.0%


In [ ]:
# --------------------------------------------------
# FREEZE PILOT RESULTS AND CREATE HELD-OUT SET
# --------------------------------------------------

import copy


# --------------------------------------------------
# 1. SAVE FINAL PILOT SUMMARY
# --------------------------------------------------

pilot_final_summary = {

    "development_cases": 5,

    "baseline_schema_valid": 0,
    "schema_constrained_schema_valid": 5,

    "baseline_schema_errors": 182,
    "schema_constrained_schema_errors": 0,

    "baseline_semantic_errors": 41,
    "schema_constrained_semantic_errors": 16,

    "absolute_semantic_error_reduction": 25,
    "percent_semantic_error_reduction": 60.975610,

    "note":
        "Pilot/development result only. "
        "Status/certainty subtype is not fully symmetric "
        "between baseline and schema-constrained conditions."
}


# --------------------------------------------------
# 2. FREEZE FINAL EXPERIMENT COMPONENTS
# --------------------------------------------------

final_schema = copy.deepcopy(
    clinical_schema_v2_frozen
)

final_gold_pilot = copy.deepcopy(
    gold_annotations_v2_frozen
)

final_semantic_categories = [
    "partial_extraction",
    "mapping_error",
    "status_error",
    "omission",
    "unsupported_inference"
]


# --------------------------------------------------
# 3. CREATE HELD-OUT CASE SET
#
# Cases 1-5 were development cases.
# Everything after them is held out.
# --------------------------------------------------

held_out_cases = copy.deepcopy(
    research_cases[5:]
)


# --------------------------------------------------
# 4. PRINT FROZEN EXPERIMENT INFORMATION
# --------------------------------------------------

print("=" * 80)
print("PILOT COMPLETE - EXPERIMENT FROZEN")
print("=" * 80)

print("\nPILOT SUMMARY")

for key, value in pilot_final_summary.items():
    print(f"{key}: {value}")


print("\n" + "-" * 80)

print(
    "Development cases:",
    len(research_cases[:5])
)

print(
    "Held-out cases available:",
    len(held_out_cases)
)


print("\nFIRST 5 HELD-OUT CASE IDs:")

for case in held_out_cases[:5]:
    print(
        " -",
        case["case_id"]
    )


print("\nIMPORTANT:")
print(
    "Do not modify the schema, prompts, or "
    "semantic scoring rules based on held-out results."
)

PILOT COMPLETE - EXPERIMENT FROZEN

PILOT SUMMARY
development_cases: 5
baseline_schema_valid: 0
schema_constrained_schema_valid: 5
baseline_schema_errors: 182
schema_constrained_schema_errors: 0
baseline_semantic_errors: 41
schema_constrained_semantic_errors: 16
absolute_semantic_error_reduction: 25
percent_semantic_error_reduction: 60.97561
note: Pilot/development result only. Status/certainty subtype is not fully symmetric between baseline and schema-constrained conditions.

--------------------------------------------------------------------------------
Development cases: 5
Held-out cases available: 35

FIRST 5 HELD-OUT CASE IDs:
 - D2N093-virtassist
 - D2N094-virtassist
 - D2N095-virtassist
 - D2N096-virtassist
 - D2N097-virtassist

IMPORTANT:
Do not modify the schema, prompts, or semantic scoring rules based on held-out results.


In [ ]:
# --------------------------------------------------
# MAIN HELD-OUT EXPERIMENT
# VERIFY EXACT FROZEN PROMPTS BEFORE GENERATION
# --------------------------------------------------

print("=" * 80)
print("VERIFY FROZEN EXPERIMENT PROMPTS")
print("=" * 80)


# --------------------------------------------------
# 1. ORIGINAL BASELINE PROMPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("ORIGINAL BASELINE PROMPT - CASE 2")
print("=" * 80)

print(
    baseline_prompt_case2
)


# --------------------------------------------------
# 2. SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED PROMPT - CASE 5")
print("=" * 80)

print(
    schema_constrained_prompt_case5
)


# --------------------------------------------------
# 3. CONFIRM HELD-OUT SIZE
# --------------------------------------------------

print("\n" + "=" * 80)
print("HELD-OUT DATASET")
print("=" * 80)

print(
    "Total held-out cases:",
    len(held_out_cases)
)

print(
    "First held-out case:",
    held_out_cases[0]["case_id"]
)

print(
    "Last held-out case:",
    held_out_cases[-1]["case_id"]
)

VERIFY FROZEN EXPERIMENT PROMPTS

ORIGINAL BASELINE PROMPT - CASE 2

You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
[doctor] hi andrea , how are you ?
[patient] i'm doing well . how are you ?
[doctor] doing well . uh , so i know the nurse told you about dax . i'd like to tell dax a little bit about you . okay ?
[patient] okay .
[doctor] so , andrea is a 52-year

In [ ]:
# --------------------------------------------------
# MAIN HELD-OUT EXPERIMENT
# FREEZE REUSABLE PROMPT TEMPLATES
# --------------------------------------------------

import hashlib


# --------------------------------------------------
# 1. EXACT BASELINE PROMPT TEMPLATE
# --------------------------------------------------

BASELINE_PROMPT_TEMPLATE = """
You are extracting structured clinical information from a doctor-patient transcript.

Return JSON only.

Use this structure:

{
  "patient": {
    "age": null,
    "sex": null
  },
  "encounter": {
    "visit_reason": null,
    "chief_complaint": null
  },
  "medical_history": [],
  "symptoms": [],
  "medications": [],
  "physical_exam": [],
  "diagnostic_tests": [],
  "assessment": [],
  "plan": {
    "medication_changes": [],
    "tests_ordered": [],
    "procedures": [],
    "referrals": [],
    "patient_instructions": []
  },
  "follow_up": {
    "value": null,
    "unit": null,
    "condition": null
  }
}

Extract information only from the transcript below.

TRANSCRIPT:
{transcript}
""".strip()


# --------------------------------------------------
# 2. EXACT SCHEMA-CONSTRAINED PROMPT TEMPLATE
# --------------------------------------------------

SCHEMA_CONSTRAINED_PROMPT_TEMPLATE = """
You are extracting structured clinical information
from a doctor-patient conversation.

Use ONLY information supported by the transcript.

Do not invent clinical facts.
Do not infer diagnoses, medications, tests, findings,
or instructions that are not supported by the transcript.

For information that is not stated and is nullable
in the schema, use null.

For list fields, include only supported items.
If no supported item exists, use an empty list.

Preserve important distinctions such as:
- present vs denied vs resolved symptoms
- current vs recommended medications
- reviewed vs ordered diagnostic tests
- confirmed vs suspected or possible assessments
- new referral vs continuing specialist follow-up

Return the structured clinical information.

TRANSCRIPT:
{transcript}
""".strip()


# --------------------------------------------------
# 3. HASH THE INSTRUCTION PORTION
# --------------------------------------------------
# These hashes let us prove later that the prompt
# instructions were not changed during evaluation.

baseline_template_hash = hashlib.sha256(
    BASELINE_PROMPT_TEMPLATE.encode("utf-8")
).hexdigest()

constrained_template_hash = hashlib.sha256(
    SCHEMA_CONSTRAINED_PROMPT_TEMPLATE.encode("utf-8")
).hexdigest()


# --------------------------------------------------
# 4. FREEZE EXPERIMENT SETTINGS
# --------------------------------------------------

held_out_experiment_config = {

    "model": "gemini-3.6-flash",

    "development_case_count": 5,

    "held_out_case_count": len(
        held_out_cases
    ),

    "baseline_prompt_sha256":
        baseline_template_hash,

    "schema_constrained_prompt_sha256":
        constrained_template_hash,

    "schema_sha256":
        "85f81bdac1c99e6cc34e2e496b6a1a7f2405025111187d81c928ccf3918cbfda",

    "schema_frozen": True,

    "prompts_frozen": True,

    "semantic_rules_frozen": True
}


# --------------------------------------------------
# 5. VERIFY
# --------------------------------------------------

print("=" * 80)
print("HELD-OUT EXPERIMENT CONFIGURATION FROZEN")
print("=" * 80)

for key, value in held_out_experiment_config.items():
    print(f"{key}: {value}")


print("\nFIRST HELD-OUT CASE:")
print(
    held_out_cases[0]["case_id"]
)

HELD-OUT EXPERIMENT CONFIGURATION FROZEN
model: gemini-3.6-flash
development_case_count: 5
held_out_case_count: 35
baseline_prompt_sha256: 51f6138bc4d40ec6461fc0fc293b6d94e6f2fd144d870aa20272c361443ab552
schema_constrained_prompt_sha256: c07e3a070fefb6a047a835a244955be23a5c7b3de445db0fad13579bbfaa7799
schema_sha256: 85f81bdac1c99e6cc34e2e496b6a1a7f2405025111187d81c928ccf3918cbfda
schema_frozen: True
prompts_frozen: True
semantic_rules_frozen: True

FIRST HELD-OUT CASE:
D2N093-virtassist


In [ ]:
# --------------------------------------------------
# HELD-OUT CASE 1
# BASELINE GENERATION
# D2N093-virtassist
# FIXED VERSION
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. SELECT FIRST HELD-OUT CASE
# --------------------------------------------------

heldout_case_index = 0

heldout_case1 = held_out_cases[
    heldout_case_index
]

print(
    "Case ID:",
    heldout_case1["case_id"]
)


# --------------------------------------------------
# 2. BUILD FROZEN BASELINE PROMPT
#
# IMPORTANT:
# Use .replace(), NOT .format(),
# because the prompt itself contains JSON braces.
# --------------------------------------------------

heldout_baseline_prompt_case1 = (
    BASELINE_PROMPT_TEMPLATE.replace(
        "{transcript}",
        heldout_case1["transcript"]
    )
)


# --------------------------------------------------
# 3. CALL SAME MODEL
# BASELINE = NO JSON SCHEMA ENFORCEMENT
# --------------------------------------------------

heldout_baseline_response_case1 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",
        input=heldout_baseline_prompt_case1
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

heldout_baseline_raw_case1 = (
    heldout_baseline_response_case1.output_text
)

heldout_baseline_hash_case1 = (
    hashlib.sha256(
        heldout_baseline_raw_case1.encode(
            "utf-8"
        )
    ).hexdigest()
)


# --------------------------------------------------
# 5. CLEAN ONLY FOR JSON PARSING
#
# Original raw output remains unchanged.
# --------------------------------------------------

clean_heldout_baseline_case1 = (
    heldout_baseline_raw_case1.strip()
)

if clean_heldout_baseline_case1.startswith(
    "```json"
):

    clean_heldout_baseline_case1 = (
        clean_heldout_baseline_case1[
            len("```json"):
        ]
    )

elif clean_heldout_baseline_case1.startswith(
    "```"
):

    clean_heldout_baseline_case1 = (
        clean_heldout_baseline_case1[
            len("```"):
        ]
    )


if clean_heldout_baseline_case1.endswith(
    "```"
):

    clean_heldout_baseline_case1 = (
        clean_heldout_baseline_case1[:-3]
    )


clean_heldout_baseline_case1 = (
    clean_heldout_baseline_case1.strip()
)


# --------------------------------------------------
# 6. TRY TO PARSE JSON
# --------------------------------------------------

try:

    heldout_baseline_output_case1 = (
        json.loads(
            clean_heldout_baseline_case1
        )
    )

    heldout_baseline_json_valid_case1 = 1

except json.JSONDecodeError:

    heldout_baseline_output_case1 = None

    heldout_baseline_json_valid_case1 = 0


# --------------------------------------------------
# 7. DISPLAY FROZEN RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("HELD-OUT CASE 1 - BASELINE")
print("=" * 80)

print(
    "Case ID:",
    heldout_case1["case_id"]
)

print(
    "JSON valid:",
    heldout_baseline_json_valid_case1
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    heldout_baseline_hash_case1
)

print(
    "\nBASELINE MODEL OUTPUT:"
)

print(
    heldout_baseline_raw_case1
)

Case ID: D2N093-virtassist

HELD-OUT CASE 1 - BASELINE
Case ID: D2N093-virtassist
JSON valid: 1

RAW OUTPUT SHA256:
52c0e1c6343c2cfaeafd7874834212b7afafbeb8ffdd10e3a5f850fc37502ef3

BASELINE MODEL OUTPUT:
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "shortness of breath",
    "chief_complaint": "shortness of breath"
  },
  "medical_history": [
    "type I diabetes",
    "congestive heart failure",
    "depression",
    "reflux"
  ],
  "symptoms": [
    "shortness of breath",
    "swelling in legs",
    "lethargy",
    "morning stiffness",
    "acute shortness of breath at night (one episode)"
  ],
  "medications": [
    "insulin pump",
    "omeprazole"
  ],
  "physical_exam": [
    "normal oxygenation level",
    "no jugular venous distension",
    "no carotid bruits",
    "3/6 systolic ejection murmur",
    "bilateral basilar crackles",
    "1+ pitting edema"
  ],
  "diagnostic_tests": [
    "chest x-ray (no evidence of airspace disease o

In [ ]:
# --------------------------------------------------
# HELD-OUT CASE 1
# SCHEMA-CONSTRAINED GENERATION
# D2N093-virtassist
# --------------------------------------------------

import json
import hashlib


# --------------------------------------------------
# 1. USE THE SAME HELD-OUT CASE
# --------------------------------------------------

print(
    "Case ID:",
    heldout_case1["case_id"]
)


# --------------------------------------------------
# 2. BUILD FROZEN SCHEMA-CONSTRAINED PROMPT
# --------------------------------------------------

heldout_constrained_prompt_case1 = (
    SCHEMA_CONSTRAINED_PROMPT_TEMPLATE.replace(
        "{transcript}",
        heldout_case1["transcript"]
    )
)


# --------------------------------------------------
# 3. CALL SAME MODEL WITH FROZEN SCHEMA V2
# --------------------------------------------------

heldout_constrained_response_case1 = (
    gemini_client.interactions.create(
        model="gemini-3.5-flash",

        input=heldout_constrained_prompt_case1,

        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": clinical_schema_v2_frozen
        }
    )
)


# --------------------------------------------------
# 4. FREEZE EXACT RAW OUTPUT
# --------------------------------------------------

heldout_constrained_raw_case1 = (
    heldout_constrained_response_case1.output_text
)

heldout_constrained_hash_case1 = (
    hashlib.sha256(
        heldout_constrained_raw_case1.encode(
            "utf-8"
        )
    ).hexdigest()
)


# --------------------------------------------------
# 5. PARSE JSON
# --------------------------------------------------

try:

    heldout_constrained_output_case1 = (
        json.loads(
            heldout_constrained_raw_case1
        )
    )

    heldout_constrained_json_valid_case1 = 1

except json.JSONDecodeError:

    heldout_constrained_output_case1 = None

    heldout_constrained_json_valid_case1 = 0


# --------------------------------------------------
# 6. DISPLAY FROZEN RESULT
# --------------------------------------------------

print("\n" + "=" * 80)
print("HELD-OUT CASE 1 - SCHEMA-CONSTRAINED")
print("=" * 80)

print(
    "Case ID:",
    heldout_case1["case_id"]
)

print(
    "JSON valid:",
    heldout_constrained_json_valid_case1
)

print(
    "\nRAW OUTPUT SHA256:"
)

print(
    heldout_constrained_hash_case1
)

print(
    "\nSCHEMA-CONSTRAINED MODEL OUTPUT:"
)

print(
    heldout_constrained_raw_case1
)

Case ID: D2N093-virtassist

HELD-OUT CASE 1 - SCHEMA-CONSTRAINED
Case ID: D2N093-virtassist
JSON valid: 1

RAW OUTPUT SHA256:
58d23bcd72711df4a01df5fe62a54d7da58400590a109f5673d459b958fc0314

SCHEMA-CONSTRAINED MODEL OUTPUT:
{
  "patient": {
    "age": 62,
    "sex": "male"
  },
  "encounter": {
    "visit_reason": "shortness of breath",
    "chief_complaint": "shortness of breath"
  },
  "medical_history": [
    {
      "condition": "type I diabetes",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "congestive heart failure",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "depression",
      "status": "current",
      "history_type": "medical"
    },
    {
      "condition": "reflux",
      "status": "current",
      "history_type": "medical"
    }
  ],
  "symptoms": [
    {
      "name": "shortness of breath",
      "status": "present",
      "body_site": "chest",
      "severity": "moderate",
    

In [ ]:
# --------------------------------------------------
# HELD-OUT CASE 1
# STRUCTURAL COMPARISON
# BASELINE VS SCHEMA-CONSTRAINED
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. CREATE VALIDATOR
# --------------------------------------------------

heldout_validator = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE BASELINE OUTPUT
# --------------------------------------------------

baseline_schema_errors_h1 = sorted(
    heldout_validator.iter_errors(
        heldout_baseline_output_case1
    ),
    key=lambda e: list(e.absolute_path)
)


baseline_schema_valid_h1 = (
    1 if len(baseline_schema_errors_h1) == 0 else 0
)


baseline_required_errors_h1 = sum(
    error.validator == "required"
    for error in baseline_schema_errors_h1
)

baseline_additional_errors_h1 = sum(
    error.validator == "additionalProperties"
    for error in baseline_schema_errors_h1
)

baseline_type_errors_h1 = sum(
    error.validator == "type"
    for error in baseline_schema_errors_h1
)

baseline_enum_errors_h1 = sum(
    error.validator == "enum"
    for error in baseline_schema_errors_h1
)


# --------------------------------------------------
# 3. VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

constrained_schema_errors_h1 = sorted(
    heldout_validator.iter_errors(
        heldout_constrained_output_case1
    ),
    key=lambda e: list(e.absolute_path)
)


constrained_schema_valid_h1 = (
    1 if len(constrained_schema_errors_h1) == 0 else 0
)


constrained_required_errors_h1 = sum(
    error.validator == "required"
    for error in constrained_schema_errors_h1
)

constrained_additional_errors_h1 = sum(
    error.validator == "additionalProperties"
    for error in constrained_schema_errors_h1
)

constrained_type_errors_h1 = sum(
    error.validator == "type"
    for error in constrained_schema_errors_h1
)

constrained_enum_errors_h1 = sum(
    error.validator == "enum"
    for error in constrained_schema_errors_h1
)


# --------------------------------------------------
# 4. PRINT COMPARISON
# --------------------------------------------------

print("=" * 80)
print("HELD-OUT CASE 1 - STRUCTURAL COMPARISON")
print("=" * 80)


print("\nBASELINE")
print("-" * 40)

print(
    "JSON valid:",
    heldout_baseline_json_valid_case1
)

print(
    "Schema valid:",
    baseline_schema_valid_h1
)

print(
    "Schema errors:",
    len(baseline_schema_errors_h1)
)

print(
    "Required errors:",
    baseline_required_errors_h1
)

print(
    "Additional-property errors:",
    baseline_additional_errors_h1
)

print(
    "Type errors:",
    baseline_type_errors_h1
)

print(
    "Enum errors:",
    baseline_enum_errors_h1
)


print("\nSCHEMA-CONSTRAINED")
print("-" * 40)

print(
    "JSON valid:",
    heldout_constrained_json_valid_case1
)

print(
    "Schema valid:",
    constrained_schema_valid_h1
)

print(
    "Schema errors:",
    len(constrained_schema_errors_h1)
)

print(
    "Required errors:",
    constrained_required_errors_h1
)

print(
    "Additional-property errors:",
    constrained_additional_errors_h1
)

print(
    "Type errors:",
    constrained_type_errors_h1
)

print(
    "Enum errors:",
    constrained_enum_errors_h1
)

HELD-OUT CASE 1 - STRUCTURAL COMPARISON

BASELINE
----------------------------------------
JSON valid: 1
Schema valid: 0
Schema errors: 28
Required errors: 2
Additional-property errors: 1
Type errors: 25
Enum errors: 0

SCHEMA-CONSTRAINED
----------------------------------------
JSON valid: 1
Schema valid: 1
Schema errors: 0
Required errors: 0
Additional-property errors: 0
Type errors: 0
Enum errors: 0


In [ ]:
# --------------------------------------------------
# HELD-OUT CASE 1
# STRUCTURAL COMPARISON
# BASELINE VS SCHEMA-CONSTRAINED
# --------------------------------------------------

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. CREATE VALIDATOR
# --------------------------------------------------

heldout_validator = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 2. VALIDATE BASELINE OUTPUT
# --------------------------------------------------

baseline_schema_errors_h1 = sorted(
    heldout_validator.iter_errors(
        heldout_baseline_output_case1
    ),
    key=lambda e: list(e.absolute_path)
)


baseline_schema_valid_h1 = (
    1 if len(baseline_schema_errors_h1) == 0 else 0
)


baseline_required_errors_h1 = sum(
    error.validator == "required"
    for error in baseline_schema_errors_h1
)

baseline_additional_errors_h1 = sum(
    error.validator == "additionalProperties"
    for error in baseline_schema_errors_h1
)

baseline_type_errors_h1 = sum(
    error.validator == "type"
    for error in baseline_schema_errors_h1
)

baseline_enum_errors_h1 = sum(
    error.validator == "enum"
    for error in baseline_schema_errors_h1
)


# --------------------------------------------------
# 3. VALIDATE SCHEMA-CONSTRAINED OUTPUT
# --------------------------------------------------

constrained_schema_errors_h1 = sorted(
    heldout_validator.iter_errors(
        heldout_constrained_output_case1
    ),
    key=lambda e: list(e.absolute_path)
)


constrained_schema_valid_h1 = (
    1 if len(constrained_schema_errors_h1) == 0 else 0
)


constrained_required_errors_h1 = sum(
    error.validator == "required"
    for error in constrained_schema_errors_h1
)

constrained_additional_errors_h1 = sum(
    error.validator == "additionalProperties"
    for error in constrained_schema_errors_h1
)

constrained_type_errors_h1 = sum(
    error.validator == "type"
    for error in constrained_schema_errors_h1
)

constrained_enum_errors_h1 = sum(
    error.validator == "enum"
    for error in constrained_schema_errors_h1
)


# --------------------------------------------------
# 4. PRINT COMPARISON
# --------------------------------------------------

print("=" * 80)
print("HELD-OUT CASE 1 - STRUCTURAL COMPARISON")
print("=" * 80)


print("\nBASELINE")
print("-" * 40)

print(
    "JSON valid:",
    heldout_baseline_json_valid_case1
)

print(
    "Schema valid:",
    baseline_schema_valid_h1
)

print(
    "Schema errors:",
    len(baseline_schema_errors_h1)
)

print(
    "Required errors:",
    baseline_required_errors_h1
)

print(
    "Additional-property errors:",
    baseline_additional_errors_h1
)

print(
    "Type errors:",
    baseline_type_errors_h1
)

print(
    "Enum errors:",
    baseline_enum_errors_h1
)


print("\nSCHEMA-CONSTRAINED")
print("-" * 40)

print(
    "JSON valid:",
    heldout_constrained_json_valid_case1
)

print(
    "Schema valid:",
    constrained_schema_valid_h1
)

print(
    "Schema errors:",
    len(constrained_schema_errors_h1)
)

print(
    "Required errors:",
    constrained_required_errors_h1
)

print(
    "Additional-property errors:",
    constrained_additional_errors_h1
)

print(
    "Type errors:",
    constrained_type_errors_h1
)

print(
    "Enum errors:",
    constrained_enum_errors_h1
)

HELD-OUT CASE 1 - STRUCTURAL COMPARISON

BASELINE
----------------------------------------
JSON valid: 1
Schema valid: 0
Schema errors: 28
Required errors: 2
Additional-property errors: 1
Type errors: 25
Enum errors: 0

SCHEMA-CONSTRAINED
----------------------------------------
JSON valid: 1
Schema valid: 1
Schema errors: 0
Required errors: 0
Additional-property errors: 0
Type errors: 0
Enum errors: 0


In [ ]:
# --------------------------------------------------
# MAIN HELD-OUT EXPERIMENT
# SAVE CASE 1 + DEFINE SAFE REUSABLE RUNNER
# --------------------------------------------------

import json
import hashlib
from jsonschema import Draft202012Validator


# --------------------------------------------------
# 1. STORAGE FOR ALL HELD-OUT RESULTS
# --------------------------------------------------

if "heldout_generation_results" not in globals():
    heldout_generation_results = {}


# --------------------------------------------------
# 2. SAVE ALREADY-COMPLETED HELD-OUT CASE 1
# --------------------------------------------------

heldout_generation_results[
    heldout_case1["case_id"]
] = {

    "case_index": 0,

    "case_id":
        heldout_case1["case_id"],

    # ----------------------------------------------
    # BASELINE
    # ----------------------------------------------

    "baseline_raw":
        heldout_baseline_raw_case1,

    "baseline_output":
        heldout_baseline_output_case1,

    "baseline_output_sha256":
        heldout_baseline_hash_case1,

    "baseline_json_valid":
        heldout_baseline_json_valid_case1,

    "baseline_schema_valid":
        baseline_schema_valid_h1,

    "baseline_schema_error_count":
        len(baseline_schema_errors_h1),


    # ----------------------------------------------
    # SCHEMA-CONSTRAINED
    # ----------------------------------------------

    "constrained_raw":
        heldout_constrained_raw_case1,

    "constrained_output":
        heldout_constrained_output_case1,

    "constrained_output_sha256":
        heldout_constrained_hash_case1,

    "constrained_json_valid":
        heldout_constrained_json_valid_case1,

    "constrained_schema_valid":
        constrained_schema_valid_h1,

    "constrained_schema_error_count":
        len(constrained_schema_errors_h1)
}


# --------------------------------------------------
# 3. SAME FROZEN VALIDATOR
# --------------------------------------------------

heldout_validator = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 4. HELPER FOR JSON PARSING
# --------------------------------------------------

def parse_baseline_json(raw_text):

    cleaned = raw_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]

    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned), 1

    except json.JSONDecodeError:
        return None, 0


# --------------------------------------------------
# 5. SAFE SINGLE-CASE RUNNER
# --------------------------------------------------

def run_heldout_case(case_index):

    case = held_out_cases[
        case_index
    ]

    case_id = case[
        "case_id"
    ]


    # ----------------------------------------------
    # DO NOT OVERWRITE COMPLETED CASES
    # ----------------------------------------------

    if case_id in heldout_generation_results:

        print(
            f"{case_id} already completed. "
            "Skipping."
        )

        return heldout_generation_results[
            case_id
        ]


    print("=" * 80)

    print(
        f"RUNNING HELD-OUT CASE "
        f"{case_index + 1}: {case_id}"
    )

    print("=" * 80)


    # ==============================================
    # BASELINE
    # ==============================================

    baseline_prompt = (
        BASELINE_PROMPT_TEMPLATE.replace(
            "{transcript}",
            case["transcript"]
        )
    )


    baseline_response = (
        gemini_client.interactions.create(
            model="gemini-3.6-flash",
            input=baseline_prompt
        )
    )


    baseline_raw = (
        baseline_response.output_text
    )


    baseline_hash = hashlib.sha256(
        baseline_raw.encode("utf-8")
    ).hexdigest()


    baseline_output, baseline_json_valid = (
        parse_baseline_json(
            baseline_raw
        )
    )


    if baseline_json_valid:

        baseline_errors = sorted(
            heldout_validator.iter_errors(
                baseline_output
            ),
            key=lambda e:
                list(e.absolute_path)
        )

    else:

        baseline_errors = []


    baseline_schema_valid = (
        1
        if (
            baseline_json_valid
            and len(baseline_errors) == 0
        )
        else 0
    )


    # ==============================================
    # SCHEMA-CONSTRAINED
    # ==============================================

    constrained_prompt = (
        SCHEMA_CONSTRAINED_PROMPT_TEMPLATE.replace(
            "{transcript}",
            case["transcript"]
        )
    )


    constrained_response = (
        gemini_client.interactions.create(
            model="gemini-3.6-flash",

            input=constrained_prompt,

            response_format={
                "type": "text",
                "mime_type":
                    "application/json",
                "schema":
                    clinical_schema_v2_frozen
            }
        )
    )


    constrained_raw = (
        constrained_response.output_text
    )


    constrained_hash = hashlib.sha256(
        constrained_raw.encode(
            "utf-8"
        )
    ).hexdigest()


    try:

        constrained_output = json.loads(
            constrained_raw
        )

        constrained_json_valid = 1

    except json.JSONDecodeError:

        constrained_output = None
        constrained_json_valid = 0


    if constrained_json_valid:

        constrained_errors = sorted(
            heldout_validator.iter_errors(
                constrained_output
            ),
            key=lambda e:
                list(e.absolute_path)
        )

    else:

        constrained_errors = []


    constrained_schema_valid = (
        1
        if (
            constrained_json_valid
            and len(constrained_errors) == 0
        )
        else 0
    )


    # ==============================================
    # SAVE RESULT IMMEDIATELY
    # ==============================================

    result = {

        "case_index":
            case_index,

        "case_id":
            case_id,

        "baseline_raw":
            baseline_raw,

        "baseline_output":
            baseline_output,

        "baseline_output_sha256":
            baseline_hash,

        "baseline_json_valid":
            baseline_json_valid,

        "baseline_schema_valid":
            baseline_schema_valid,

        "baseline_schema_error_count":
            len(baseline_errors),


        "constrained_raw":
            constrained_raw,

        "constrained_output":
            constrained_output,

        "constrained_output_sha256":
            constrained_hash,

        "constrained_json_valid":
            constrained_json_valid,

        "constrained_schema_valid":
            constrained_schema_valid,

        "constrained_schema_error_count":
            len(constrained_errors)
    }


    heldout_generation_results[
        case_id
    ] = result


    # ==============================================
    # PRINT SHORT RESULT
    # ==============================================

    print(
        "\nBaseline:"
    )

    print(
        "  JSON valid:",
        baseline_json_valid
    )

    print(
        "  Schema valid:",
        baseline_schema_valid
    )

    print(
        "  Schema errors:",
        len(baseline_errors)
    )

    print(
        "  SHA256:",
        baseline_hash
    )


    print(
        "\nSchema-constrained:"
    )

    print(
        "  JSON valid:",
        constrained_json_valid
    )

    print(
        "  Schema valid:",
        constrained_schema_valid
    )

    print(
        "  Schema errors:",
        len(constrained_errors)
    )

    print(
        "  SHA256:",
        constrained_hash
    )


    return result


# --------------------------------------------------
# 6. VERIFY CASE 1 WAS SAVED
# --------------------------------------------------

print("=" * 80)
print("HELD-OUT STORAGE INITIALIZED")
print("=" * 80)

print(
    "Cases currently saved:",
    len(
        heldout_generation_results
    )
)

print(
    "Saved case IDs:",
    list(
        heldout_generation_results.keys()
    )
)

HELD-OUT STORAGE INITIALIZED
Cases currently saved: 1
Saved case IDs: ['D2N093-virtassist']


In [ ]:
# --------------------------------------------------
# FREEZE MODEL FOR FINAL HELD-OUT EXPERIMENT
# --------------------------------------------------

HELDOUT_MODEL = "gemini-3.5-flash"

print("=" * 80)
print("FINAL HELD-OUT MODEL")
print("=" * 80)

print("Model:", HELDOUT_MODEL)

print(
    "\nIMPORTANT: Every held-out case must use "
    "this exact model."
)

FINAL HELD-OUT MODEL
Model: gemini-3.5-flash

IMPORTANT: Every held-out case must use this exact model.


In [ ]:
 # --------------------------------------------------
# HELD-OUT CASE 2
# BASELINE + SCHEMA-CONSTRAINED
# --------------------------------------------------

heldout_case2_result = run_heldout_case(1)

RUNNING HELD-OUT CASE 2: D2N094-virtassist


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 55.675861639s.', 'code': 'too_many_requests'}}

In [ ]:
# --------------------------------------------------
# HELD-OUT CASE 3
# BASELINE + SCHEMA-CONSTRAINED
# --------------------------------------------------

heldout_case3_result = run_heldout_case(2)

In [ ]:
# --------------------------------------------------
# SAVE RESEARCH CHECKPOINT TO GOOGLE DRIVE
# --------------------------------------------------

from google.colab import drive
import json
import os

drive.mount("/content/drive")

checkpoint_folder = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research"
)

os.makedirs(
    checkpoint_folder,
    exist_ok=True
)

checkpoint_path = os.path.join(
    checkpoint_folder,
    "heldout_experiment_checkpoint.json"
)


# --------------------------------------------------
# BUILD CHECKPOINT
# --------------------------------------------------

checkpoint = {

    # Frozen experimental configuration
    "experiment_config":
        held_out_experiment_config,

    # Frozen prompts
    "baseline_prompt_template":
        BASELINE_PROMPT_TEMPLATE,

    "schema_constrained_prompt_template":
        SCHEMA_CONSTRAINED_PROMPT_TEMPLATE,

    # Frozen Schema v2
    "clinical_schema_v2":
        clinical_schema_v2_frozen,

    # Pilot result
    "pilot_summary":
        pilot_final_summary,

    # MOST IMPORTANT:
    # all successful held-out Gemini generations
    "heldout_generation_results":
        heldout_generation_results
}


# --------------------------------------------------
# SAVE
# --------------------------------------------------

with open(
    checkpoint_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# VERIFY
# --------------------------------------------------

print("=" * 80)
print("CHECKPOINT SAVED")
print("=" * 80)

print(
    "File:",
    checkpoint_path
)

print(
    "Completed held-out cases:",
    len(heldout_generation_results)
)

print(
    "\nSaved case IDs:"
)

for case_id in heldout_generation_results:
    print(" -", case_id)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHECKPOINT SAVED
File: /content/drive/MyDrive/LLM_Reliability_Research/heldout_experiment_checkpoint.json
Completed held-out cases: 1

Saved case IDs:
 - D2N093-virtassist


In [ ]:
# --------------------------------------------------
# CHECK WHAT SURVIVED THE COLAB RESTART
# NO GEMINI CALLS
# --------------------------------------------------

required_variables = [
    "research_cases",
    "held_out_cases",
    "clinical_schema_v2_frozen",
    "BASELINE_PROMPT_TEMPLATE",
    "SCHEMA_CONSTRAINED_PROMPT_TEMPLATE",
    "pilot_final_summary",
    "held_out_experiment_config",
    "heldout_generation_results",
    "run_heldout_case"
]

print("=" * 80)
print("COLAB VARIABLE CHECK")
print("=" * 80)

for variable_name in required_variables:

    if variable_name in globals():
        print("✅", variable_name)

    else:
        print("❌", variable_name)

COLAB VARIABLE CHECK
✅ research_cases
❌ held_out_cases
❌ clinical_schema_v2_frozen
✅ BASELINE_PROMPT_TEMPLATE
✅ SCHEMA_CONSTRAINED_PROMPT_TEMPLATE
✅ pilot_final_summary
❌ held_out_experiment_config
✅ heldout_generation_results
❌ run_heldout_case


In [ ]:
# --------------------------------------------------
# RECOVER HELD-OUT EXPERIMENT AFTER COLAB RESTART
# NO GEMINI CALLS
# --------------------------------------------------

import copy
import hashlib


# --------------------------------------------------
# 1. RECREATE HELD-OUT CASES
# research_cases survived
# --------------------------------------------------

held_out_cases = copy.deepcopy(
    research_cases[5:]
)

print(
    "✅ held_out_cases restored:",
    len(held_out_cases)
)


# --------------------------------------------------
# 2. RESTORE FROZEN SCHEMA IF ORIGINAL V2 EXISTS
# --------------------------------------------------

if "clinical_schema_v2" in globals():

    clinical_schema_v2_frozen = copy.deepcopy(
        clinical_schema_v2
    )

    schema_hash_check = hashlib.sha256(
        __import__("json").dumps(
            clinical_schema_v2_frozen,
            sort_keys=True
        ).encode("utf-8")
    ).hexdigest()

    print(
        "✅ clinical_schema_v2_frozen restored"
    )

    print(
        "Schema hash currently:",
        schema_hash_check
    )

else:

    print(
        "❌ clinical_schema_v2 is also missing."
    )

    print(
        "Rerun ONLY your original Schema v2 "
        "definition/freeze cells."
    )


# --------------------------------------------------
# 3. RECREATE EXPERIMENT CONFIG
# --------------------------------------------------

if "clinical_schema_v2_frozen" in globals():

    baseline_template_hash = hashlib.sha256(
        BASELINE_PROMPT_TEMPLATE.encode(
            "utf-8"
        )
    ).hexdigest()

    constrained_template_hash = hashlib.sha256(
        SCHEMA_CONSTRAINED_PROMPT_TEMPLATE.encode(
            "utf-8"
        )
    ).hexdigest()


    held_out_experiment_config = {

        "model":
            "gemini-3.6-flash",

        "development_case_count":
            5,

        "held_out_case_count":
            len(held_out_cases),

        "baseline_prompt_sha256":
            baseline_template_hash,

        "schema_constrained_prompt_sha256":
            constrained_template_hash,

        "schema_sha256":
            "85f81bdac1c99e6cc34e2e496b6a1a7f2405025111187d81c928ccf3918cbfda",

        "schema_frozen":
            True,

        "prompts_frozen":
            True,

        "semantic_rules_frozen":
            True
    }

    print(
        "✅ held_out_experiment_config restored"
    )


# --------------------------------------------------
# 4. CHECK SAVED GEMINI RESULTS
# --------------------------------------------------

print("\n" + "=" * 80)
print("SAVED GEMINI RESULTS")
print("=" * 80)

print(
    "Completed cases:",
    len(heldout_generation_results)
)

for case_id in heldout_generation_results:
    print("✅", case_id)

✅ held_out_cases restored: 35
❌ clinical_schema_v2 is also missing.
Rerun ONLY your original Schema v2 definition/freeze cells.

SAVED GEMINI RESULTS
Completed cases: 0


In [ ]:
print(
    "clinical_schema_v2 exists:",
    "clinical_schema_v2" in globals()
)

print(
    "clinical_schema_v2_frozen exists:",
    "clinical_schema_v2_frozen" in globals()
)

clinical_schema_v2 exists: True
clinical_schema_v2_frozen exists: True


In [ ]:
print("HELDOUT_MODEL =", HELDOUT_MODEL)

print(
    "\nDoes runner still contain 3.6?",
    "gemini-3.6-flash" in str(
        run_heldout_case.__code__.co_consts
    )
)

HELDOUT_MODEL = gemini-3.5-flash

Does runner still contain 3.6? True


In [ ]:
# --------------------------------------------------
# SAFE HELD-OUT RUNNER
# USES FROZEN HELDOUT_MODEL
# AUTO-SAVES AFTER EACH SUCCESSFUL MODEL CALL
# --------------------------------------------------

import json
import hashlib
import os

from jsonschema import Draft202012Validator


# --------------------------------------------------
# 0. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "HELDOUT_MODEL",
    "held_out_cases",
    "BASELINE_PROMPT_TEMPLATE",
    "SCHEMA_CONSTRAINED_PROMPT_TEMPLATE",
    "clinical_schema_v2_frozen",
    "gemini_client"
]

for obj in required_objects:
    if obj not in globals():
        raise RuntimeError(
            f"Missing required object: {obj}"
        )


# --------------------------------------------------
# 1. RESULT STORAGE
# --------------------------------------------------

if "heldout_generation_results" not in globals():
    heldout_generation_results = {}


# --------------------------------------------------
# 2. CHECKPOINT PATH
# --------------------------------------------------

checkpoint_folder = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research"
)

os.makedirs(
    checkpoint_folder,
    exist_ok=True
)

checkpoint_path = os.path.join(
    checkpoint_folder,
    "heldout_experiment_checkpoint.json"
)


# --------------------------------------------------
# 3. CHECKPOINT FUNCTION
# --------------------------------------------------

def save_heldout_checkpoint():

    checkpoint = {

        "heldout_model":
            HELDOUT_MODEL,

        "heldout_generation_results":
            heldout_generation_results,

        "baseline_prompt_template":
            BASELINE_PROMPT_TEMPLATE,

        "schema_constrained_prompt_template":
            SCHEMA_CONSTRAINED_PROMPT_TEMPLATE,

        "clinical_schema_v2":
            clinical_schema_v2_frozen
    }

    # Preserve legacy 3.6 results if they exist
    if "legacy_36_results" in globals():
        checkpoint["legacy_36_results"] = (
            legacy_36_results
        )

    with open(
        checkpoint_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2,
            ensure_ascii=False
        )


# --------------------------------------------------
# 4. JSON PARSER FOR BASELINE OUTPUT
# --------------------------------------------------

def parse_baseline_json(raw_text):

    cleaned = raw_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]

    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned), 1

    except json.JSONDecodeError:
        return None, 0


# --------------------------------------------------
# 5. VALIDATOR
# --------------------------------------------------

heldout_validator = Draft202012Validator(
    clinical_schema_v2_frozen
)


# --------------------------------------------------
# 6. SAFE RUNNER
# --------------------------------------------------

def run_heldout_case(case_index):

    case = held_out_cases[case_index]

    case_id = case["case_id"]


    print("=" * 80)
    print(
        f"RUNNING HELD-OUT CASE "
        f"{case_index + 1}: {case_id}"
    )
    print("=" * 80)

    print(
        "Frozen model:",
        HELDOUT_MODEL
    )


    # --------------------------------------------------
    # CREATE RESULT RECORD IF THIS CASE IS NEW
    # --------------------------------------------------

    if case_id not in heldout_generation_results:

        heldout_generation_results[
            case_id
        ] = {

            "case_index":
                case_index,

            "case_id":
                case_id,

            "model":
                HELDOUT_MODEL,

            "baseline_complete":
                False,

            "constrained_complete":
                False
        }


    result = heldout_generation_results[
        case_id
    ]


    # ==================================================
    # BASELINE
    # ==================================================

    if not result.get(
        "baseline_complete",
        False
    ):

        print(
            "\nRunning baseline..."
        )

        baseline_prompt = (
            BASELINE_PROMPT_TEMPLATE.replace(
                "{transcript}",
                case["transcript"]
            )
        )

        try:

            baseline_response = (
                gemini_client.interactions.create(
                    model=HELDOUT_MODEL,
                    input=baseline_prompt
                )
            )

        except Exception:

            print(
                "\n❌ Baseline API call failed."
            )

            save_heldout_checkpoint()

            raise


        baseline_raw = (
            baseline_response.output_text
        )

        baseline_hash = hashlib.sha256(
            baseline_raw.encode(
                "utf-8"
            )
        ).hexdigest()


        baseline_output, baseline_json_valid = (
            parse_baseline_json(
                baseline_raw
            )
        )


        if baseline_json_valid:

            baseline_errors = list(
                heldout_validator.iter_errors(
                    baseline_output
                )
            )

        else:

            baseline_errors = []


        baseline_schema_valid = int(
            baseline_json_valid == 1
            and len(baseline_errors) == 0
        )


        result.update({

            "baseline_raw":
                baseline_raw,

            "baseline_output":
                baseline_output,

            "baseline_output_sha256":
                baseline_hash,

            "baseline_json_valid":
                baseline_json_valid,

            "baseline_schema_valid":
                baseline_schema_valid,

            "baseline_schema_error_count":
                len(baseline_errors),

            "baseline_complete":
                True
        })


        # SAVE IMMEDIATELY
        save_heldout_checkpoint()


        print(
            "✅ Baseline saved to Drive."
        )


    else:

        print(
            "\n✅ Baseline already completed."
        )

        print(
            "Skipping baseline API call."
        )


    # ==================================================
    # SCHEMA-CONSTRAINED
    # ==================================================

    if not result.get(
        "constrained_complete",
        False
    ):

        print(
            "\nRunning schema-constrained..."
        )

        constrained_prompt = (
            SCHEMA_CONSTRAINED_PROMPT_TEMPLATE.replace(
                "{transcript}",
                case["transcript"]
            )
        )


        try:

            constrained_response = (
                gemini_client.interactions.create(

                    model=HELDOUT_MODEL,

                    input=constrained_prompt,

                    response_format={
                        "type": "text",
                        "mime_type":
                            "application/json",
                        "schema":
                            clinical_schema_v2_frozen
                    }
                )
            )

        except Exception:

            print(
                "\n❌ Schema-constrained API call failed."
            )

            print(
                "Any successful baseline output "
                "has already been saved."
            )

            save_heldout_checkpoint()

            raise


        constrained_raw = (
            constrained_response.output_text
        )

        constrained_hash = hashlib.sha256(
            constrained_raw.encode(
                "utf-8"
            )
        ).hexdigest()


        try:

            constrained_output = json.loads(
                constrained_raw
            )

            constrained_json_valid = 1

        except json.JSONDecodeError:

            constrained_output = None

            constrained_json_valid = 0


        if constrained_json_valid:

            constrained_errors = list(
                heldout_validator.iter_errors(
                    constrained_output
                )
            )

        else:

            constrained_errors = []


        constrained_schema_valid = int(
            constrained_json_valid == 1
            and len(constrained_errors) == 0
        )


        result.update({

            "constrained_raw":
                constrained_raw,

            "constrained_output":
                constrained_output,

            "constrained_output_sha256":
                constrained_hash,

            "constrained_json_valid":
                constrained_json_valid,

            "constrained_schema_valid":
                constrained_schema_valid,

            "constrained_schema_error_count":
                len(constrained_errors),

            "constrained_complete":
                True
        })


        # SAVE IMMEDIATELY
        save_heldout_checkpoint()


        print(
            "✅ Schema-constrained output saved to Drive."
        )


    else:

        print(
            "\n✅ Schema-constrained already completed."
        )

        print(
            "Skipping constrained API call."
        )


    # ==================================================
    # FINAL SUMMARY
    # ==================================================

    print("\n" + "=" * 80)
    print("CASE SUMMARY")
    print("=" * 80)

    print(
        "Case:",
        case_id
    )

    print(
        "Model:",
        result["model"]
    )


    if result.get("baseline_complete"):

        print("\nBASELINE")

        print(
            "JSON valid:",
            result[
                "baseline_json_valid"
            ]
        )

        print(
            "Schema valid:",
            result[
                "baseline_schema_valid"
            ]
        )

        print(
            "Schema errors:",
            result[
                "baseline_schema_error_count"
            ]
        )

        print(
            "SHA256:",
            result[
                "baseline_output_sha256"
            ]
        )


    if result.get(
        "constrained_complete"
    ):

        print(
            "\nSCHEMA-CONSTRAINED"
        )

        print(
            "JSON valid:",
            result[
                "constrained_json_valid"
            ]
        )

        print(
            "Schema valid:",
            result[
                "constrained_schema_valid"
            ]
        )

        print(
            "Schema errors:",
            result[
                "constrained_schema_error_count"
            ]
        )

        print(
            "SHA256:",
            result[
                "constrained_output_sha256"
            ]
        )


    return result


print("=" * 80)
print("SAFE RUNNER REDEFINED")
print("=" * 80)

print(
    "Frozen model:",
    HELDOUT_MODEL
)

print(
    "Checkpoint:",
    checkpoint_path
)

SAFE RUNNER REDEFINED
Frozen model: gemini-3.5-flash
Checkpoint: /content/drive/MyDrive/LLM_Reliability_Research/heldout_experiment_checkpoint.json


In [ ]:
print(
    "HELDOUT_MODEL =",
    HELDOUT_MODEL
)

print(
    "Does runner still contain 3.6?",
    "gemini-3.6-flash"
    in str(
        run_heldout_case.__code__.co_consts
    )
)

HELDOUT_MODEL = gemini-3.5-flash
Does runner still contain 3.6? False


In [ ]:
# --------------------------------------------------
# MOVE OLD D2N093 RESULT TO LEGACY 3.6 STORAGE
# NO GEMINI CALL
# --------------------------------------------------

case_id = "D2N093-virtassist"


# Create legacy storage if needed
if "legacy_36_results" not in globals():
    legacy_36_results = {}


# --------------------------------------------------
# Move old result out of final experiment
# --------------------------------------------------

if case_id in heldout_generation_results:

    old_result = heldout_generation_results[
        case_id
    ]

    legacy_36_results[
        case_id
    ] = old_result

    del heldout_generation_results[
        case_id
    ]

    print(
        "✅ Old D2N093 result moved to "
        "legacy 3.6 storage."
    )

else:

    print(
        "D2N093 was already removed from "
        "final result storage."
    )


# --------------------------------------------------
# Save corrected checkpoint
# --------------------------------------------------

save_heldout_checkpoint()


# --------------------------------------------------
# Verify
# --------------------------------------------------

print("\n" + "=" * 80)
print("RESULT STORAGE CHECK")
print("=" * 80)

print(
    "Final 3.5 cases:",
    list(
        heldout_generation_results.keys()
    )
)

print(
    "Legacy 3.6 cases:",
    list(
        legacy_36_results.keys()
    )
)

✅ Old D2N093 result moved to legacy 3.6 storage.

RESULT STORAGE CHECK
Final 3.5 cases: []
Legacy 3.6 cases: ['D2N093-virtassist']


In [ ]:
# --------------------------------------------------
# FINAL HELD-OUT CASE 1
# D2N093-virtassist
# GEMINI 3.5 FLASH
# --------------------------------------------------

heldout_case1_result = run_heldout_case(0)

RUNNING HELD-OUT CASE 1: D2N093-virtassist
Frozen model: gemini-3.5-flash

Running baseline...
✅ Baseline saved to Drive.

Running schema-constrained...
✅ Schema-constrained output saved to Drive.

CASE SUMMARY
Case: D2N093-virtassist
Model: gemini-3.5-flash

BASELINE
JSON valid: 1
Schema valid: 0
Schema errors: 29
SHA256: b38a4f7bc45784b6777e4dc06137e8b60c46b9133237960235658de5b1d89b0d

SCHEMA-CONSTRAINED
JSON valid: 1
Schema valid: 1
Schema errors: 0
SHA256: 1f38b93aa6bd14ff53f2f3b39ff35c229168b0283b486b110e3fcf6ed8b36af5


In [ ]:
# --------------------------------------------------
# FINAL HELD-OUT CASE 2
# D2N094-virtassist
# GEMINI 3.5 FLASH
# --------------------------------------------------

heldout_case2_result = run_heldout_case(1)

RUNNING HELD-OUT CASE 2: D2N094-virtassist
Frozen model: gemini-3.5-flash

Running baseline...
✅ Baseline saved to Drive.

Running schema-constrained...
✅ Schema-constrained output saved to Drive.

CASE SUMMARY
Case: D2N094-virtassist
Model: gemini-3.5-flash

BASELINE
JSON valid: 1
Schema valid: 0
Schema errors: 12
SHA256: f8f25978a82beac047dd5cec698e1432ff5910da8c177d0a564fc7a21a19479e

SCHEMA-CONSTRAINED
JSON valid: 1
Schema valid: 1
Schema errors: 0
SHA256: 75637e14742fd08dfa0ada5a5a50be2a3760ab1fd464294b6861017545ee611b


In [ ]:
# --------------------------------------------------
# FINAL HELD-OUT CASE 3
# D2N094-virtassist
# GEMINI 3.5 FLASH
# --------------------------------------------------

heldout_case3_result = run_heldout_case(2)

RUNNING HELD-OUT CASE 3: D2N095-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...
✅ Schema-constrained output saved to Drive.

CASE SUMMARY
Case: D2N095-virtassist
Model: gemini-3.5-flash

BASELINE
JSON valid: 1
Schema valid: 0
Schema errors: 27
SHA256: afdc271b1a6e65f1790267e7702de2acab5ff951ea23a08befa89b87c0a75a63

SCHEMA-CONSTRAINED
JSON valid: 1
Schema valid: 1
Schema errors: 0
SHA256: 25c2226ddf91cd99a9fd91f6ce47da36f1944e318bde37263792fe1595671719


In [ ]:
# --------------------------------------------------
# FINAL HELD-OUT CASE 3
# D2N094-virtassist
# GEMINI 3.5 FLASH
# --------------------------------------------------

heldout_case3_result = run_heldout_case(2)

In [ ]:
# --------------------------------------------------
# VERIFY SAVED HELD-OUT CHECKPOINT
# NO GEMINI CALLS
# --------------------------------------------------

import json

checkpoint_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_experiment_checkpoint.json"
)

with open(
    checkpoint_path,
    "r",
    encoding="utf-8"
) as f:

    saved_checkpoint = json.load(f)


saved_results = saved_checkpoint[
    "heldout_generation_results"
]

print("=" * 80)
print("SAVED HELD-OUT RESULTS")
print("=" * 80)

print(
    "Frozen model:",
    saved_checkpoint["heldout_model"]
)

print(
    "Number of cases saved:",
    len(saved_results)
)

print()

for case_id, result in saved_results.items():

    print(case_id)

    print(
        "  Baseline complete:",
        result.get(
            "baseline_complete",
            False
        )
    )

    print(
        "  Constrained complete:",
        result.get(
            "constrained_complete",
            False
        )
    )

    print()

SAVED HELD-OUT RESULTS
Frozen model: gemini-3.5-flash
Number of cases saved: 3

D2N093-virtassist
  Baseline complete: True
  Constrained complete: True

D2N094-virtassist
  Baseline complete: True
  Constrained complete: True

D2N095-virtassist
  Baseline complete: True
  Constrained complete: False



In [ ]:
# --------------------------------------------------
# RESUME HELD-OUT CASE 3
# ONLY CONSTRAINED SHOULD RUN
# --------------------------------------------------

heldout_case3_result = run_heldout_case(2)

RUNNING HELD-OUT CASE 3: D2N095-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

✅ Schema-constrained already completed.
Skipping constrained API call.

CASE SUMMARY
Case: D2N095-virtassist
Model: gemini-3.5-flash

BASELINE
JSON valid: 1
Schema valid: 0
Schema errors: 27
SHA256: afdc271b1a6e65f1790267e7702de2acab5ff951ea23a08befa89b87c0a75a63

SCHEMA-CONSTRAINED
JSON valid: 1
Schema valid: 1
Schema errors: 0
SHA256: 25c2226ddf91cd99a9fd91f6ce47da36f1944e318bde37263792fe1595671719


In [ ]:
# --------------------------------------------------
# FINAL HELD-OUT CASE 4
# D2N096-virtassist
# GEMINI 3.5 FLASH
# --------------------------------------------------

heldout_case4_result = run_heldout_case(3)

RUNNING HELD-OUT CASE 4: D2N096-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...

❌ Schema-constrained API call failed.
Any successful baseline output has already been saved.


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 53.695002779s.', 'code': 'too_many_requests'}}

In [ ]:
# --------------------------------------------------
# CHECK CASE 4 CHECKPOINT STATUS
# NO GEMINI CALL
# --------------------------------------------------

case_id = "D2N096-virtassist"

result = heldout_generation_results.get(
    case_id,
    {}
)

print("=" * 80)
print("CASE 4 STATUS")
print("=" * 80)

print(
    "Baseline complete:",
    result.get(
        "baseline_complete",
        False
    )
)

print(
    "Constrained complete:",
    result.get(
        "constrained_complete",
        False
    )
)

CASE 4 STATUS
Baseline complete: True
Constrained complete: False


In [ ]:
# --------------------------------------------------
# AUTO-RETRY 429 WRAPPER
# SAME MODEL / SAME PROMPT / SAME EXPERIMENT
# --------------------------------------------------

import time
import re


def run_with_rate_limit_retry(
    case_index,
    max_attempts=3
):

    for attempt in range(
        1,
        max_attempts + 1
    ):

        try:

            return run_heldout_case(
                case_index
            )

        except Exception as e:

            error_text = str(e)

            is_rate_limit = (
                "429" in error_text
                or "too_many_requests"
                   in error_text.lower()
                or "quota exceeded"
                   in error_text.lower()
            )

            if not is_rate_limit:
                raise


            # --------------------------------------
            # Check current saved state
            # --------------------------------------

            case_id = held_out_cases[
                case_index
            ]["case_id"]

            result = (
                heldout_generation_results.get(
                    case_id,
                    {}
                )
            )


            print("\n" + "=" * 80)
            print("429 RATE LIMIT")
            print("=" * 80)

            print(
                "Case:",
                case_id
            )

            print(
                "Baseline complete:",
                result.get(
                    "baseline_complete",
                    False
                )
            )

            print(
                "Constrained complete:",
                result.get(
                    "constrained_complete",
                    False
                )
            )


            # --------------------------------------
            # Extract suggested retry interval
            # from Gemini error
            # --------------------------------------

            match = re.search(
                r"retry in ([0-9.]+)s",
                error_text,
                re.IGNORECASE
            )

            if match:

                retry_seconds = (
                    float(
                        match.group(1)
                    )
                    + 3
                )

            else:

                retry_seconds = 65


            if attempt == max_attempts:

                print(
                    "\nMaximum retry attempts reached."
                )

                print(
                    "✅ Everything successful "
                    "remains checkpointed."
                )

                return None


            print(
                f"\nAutomatic retry "
                f"{attempt + 1}/{max_attempts}"
            )


            time.sleep(
                retry_seconds
            )


    return None

In [ ]:
# --------------------------------------------------
# RESUME CASE 4 WITH AUTOMATIC 429 RETRY
# ONLY THE MISSING CONSTRAINED CALL WILL RUN
# --------------------------------------------------

heldout_case4_result = run_with_rate_limit_retry(
    3,
    max_attempts=3
)

RUNNING HELD-OUT CASE 4: D2N096-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...

❌ Schema-constrained API call failed.
Any successful baseline output has already been saved.

429 RATE LIMIT
Case: D2N096-virtassist
Baseline complete: True
Constrained complete: False

Automatic retry 2/3
RUNNING HELD-OUT CASE 4: D2N096-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...

❌ Schema-constrained API call failed.
Any successful baseline output has already been saved.

429 RATE LIMIT
Case: D2N096-virtassist
Baseline complete: True
Constrained complete: False

Automatic retry 3/3
RUNNING HELD-OUT CASE 4: D2N096-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...

❌ Schema-constrained API call failed.
Any successful baseline output has already been saved.


In [ ]:
# --------------------------------------------------
# HELD-OUT PROGRESS SUMMARY
# NO GEMINI CALLS
# --------------------------------------------------

import pandas as pd

progress_rows = []

for case_id, result in heldout_generation_results.items():

    progress_rows.append({

        "case_id":
            case_id,

        "model":
            result.get("model"),

        "baseline_complete":
            result.get(
                "baseline_complete",
                False
            ),

        "baseline_schema_valid":
            result.get(
                "baseline_schema_valid"
            ),

        "baseline_schema_errors":
            result.get(
                "baseline_schema_error_count"
            ),

        "constrained_complete":
            result.get(
                "constrained_complete",
                False
            ),

        "constrained_schema_valid":
            result.get(
                "constrained_schema_valid"
            ),

        "constrained_schema_errors":
            result.get(
                "constrained_schema_error_count"
            )
    })


heldout_progress_df = pd.DataFrame(
    progress_rows
)


print("=" * 80)
print("HELD-OUT EXPERIMENT PROGRESS")
print("=" * 80)

display(
    heldout_progress_df
)


# --------------------------------------------------
# COMPLETED PAIRED CASES ONLY
# --------------------------------------------------

completed_pairs = heldout_progress_df[
    (
        heldout_progress_df[
            "baseline_complete"
        ] == True
    )
    &
    (
        heldout_progress_df[
            "constrained_complete"
        ] == True
    )
]


print("\n" + "=" * 80)
print("COMPLETED PAIRED CASES")
print("=" * 80)

print(
    "Completed pairs:",
    len(completed_pairs)
)

print(
    "Baseline schema-valid:",
    int(
        completed_pairs[
            "baseline_schema_valid"
        ].sum()
    ),
    "/",
    len(completed_pairs)
)

print(
    "Constrained schema-valid:",
    int(
        completed_pairs[
            "constrained_schema_valid"
        ].sum()
    ),
    "/",
    len(completed_pairs)
)

print(
    "Baseline schema errors:",
    int(
        completed_pairs[
            "baseline_schema_errors"
        ].sum()
    )
)

print(
    "Constrained schema errors:",
    int(
        completed_pairs[
            "constrained_schema_errors"
        ].sum()
    )
)

HELD-OUT EXPERIMENT PROGRESS


,case_id,model,baseline_complete,baseline_schema_valid,baseline_schema_errors,constrained_complete,constrained_schema_valid,constrained_schema_errors
0,D2N093-virtassist,gemini-3.5-flash,True,0,29,True,1.0,0.0
1,D2N094-virtassist,gemini-3.5-flash,True,0,12,True,1.0,0.0
2,D2N095-virtassist,gemini-3.5-flash,True,0,27,True,1.0,0.0
3,D2N096-virtassist,gemini-3.5-flash,True,0,14,False,NaN,NaN



COMPLETED PAIRED CASES
Completed pairs: 3
Baseline schema-valid: 0 / 3
Constrained schema-valid: 3 / 3
Baseline schema errors: 68
Constrained schema errors: 0


In [ ]:
# --------------------------------------------------
# CREATE SEMANTIC REVIEW PACKET
# COMPLETED HELD-OUT CASES ONLY
# NO GEMINI CALLS
# --------------------------------------------------

import json
import os


semantic_review_packet = []


for case_index, case in enumerate(
    held_out_cases
):

    case_id = case["case_id"]

    if case_id not in heldout_generation_results:
        continue

    result = heldout_generation_results[
        case_id
    ]

    # Only include fully completed pairs
    if not (
        result.get("baseline_complete", False)
        and
        result.get("constrained_complete", False)
    ):
        continue


    semantic_review_packet.append({

        "case_index":
            case_index,

        "case_id":
            case_id,

        "transcript":
            case["transcript"],

        "reference_note":
            case["reference_note"],

        "baseline_output":
            result["baseline_output"],

        "baseline_sha256":
            result["baseline_output_sha256"],

        "schema_constrained_output":
            result["constrained_output"],

        "schema_constrained_sha256":
            result["constrained_output_sha256"],

        # Empty manual adjudication fields
        "baseline_semantic_review": {
            "partial_extractions": [],
            "mapping_errors": [],
            "status_certainty_errors": [],
            "omissions": [],
            "unsupported_inferences": []
        },

        "schema_constrained_semantic_review": {
            "partial_extractions": [],
            "mapping_errors": [],
            "status_certainty_errors": [],
            "omissions": [],
            "unsupported_inferences": []
        }
    })


# --------------------------------------------------
# SAVE TO GOOGLE DRIVE
# --------------------------------------------------

semantic_review_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_review_packet.json"
)

with open(
    semantic_review_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        semantic_review_packet,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# VERIFY
# --------------------------------------------------

print("=" * 80)
print("SEMANTIC REVIEW PACKET CREATED")
print("=" * 80)

print(
    "Cases included:",
    len(semantic_review_packet)
)

for item in semantic_review_packet:
    print(
        " -",
        item["case_id"]
    )

print(
    "\nSaved to:"
)

print(
    semantic_review_path
)

SEMANTIC REVIEW PACKET CREATED
Cases included: 3
 - D2N093-virtassist
 - D2N094-virtassist
 - D2N095-virtassist

Saved to:
/content/drive/MyDrive/LLM_Reliability_Research/heldout_semantic_review_packet.json


In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC REVIEW
# CASE 1: D2N093-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

review_case = semantic_review_packet[0]

print("=" * 80)
print("SEMANTIC REVIEW - HELD-OUT CASE 1")
print("=" * 80)

print(
    "Case ID:",
    review_case["case_id"]
)


print("\n" + "=" * 80)
print("TRANSCRIPT")
print("=" * 80)

print(
    review_case["transcript"]
)


print("\n" + "=" * 80)
print("REFERENCE NOTE")
print("=" * 80)

print(
    review_case["reference_note"]
)


print("\n" + "=" * 80)
print("BASELINE OUTPUT")
print("=" * 80)

print(
    json.dumps(
        review_case["baseline_output"],
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    json.dumps(
        review_case["schema_constrained_output"],
        indent=2,
        ensure_ascii=False
    )
)

SEMANTIC REVIEW - HELD-OUT CASE 1
Case ID: D2N093-virtassist

TRANSCRIPT
[doctor] hey lawrence . how are you ?
[patient] hey , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you about dax .
[patient] mm-hmm .
[doctor] i'd like to tell dax a little bit about you .
[patient] sure .
[doctor] so , lawrence is a 62-year-old male , with a past medical history significant for type i diabetes , congestive heart failure , depression , and reflux , who presents with complaints of shortness of breath . so lawrence , what's been going on ? wh- what's wrong with your breathing ?
[patient] uh , i , i've noticed that i've been swelling up a little bit . i think a lot of it has to do with going to some house parties , eating some salty foods . i feel really lethargic .
[doctor] okay . all right . and when you get short of breath , are you short of breath when you're just sitting here ? do you feel short of breath when you're walking ?
[patient] it's something like wal

In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC ADJUDICATION
# CASE 1: D2N093-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

semantic_result_D2N093 = {

    "case_id":
        "D2N093-virtassist",

    # ==================================================
    # BASELINE
    # ==================================================

    "baseline_partial_extractions": 2,

    "baseline_mapping_errors": 1,

    "baseline_status_certainty_errors": 0,

    "baseline_omissions": 6,

    "baseline_unsupported_inferences": 0,

    "baseline_total_semantic_errors": 9,


    # ==================================================
    # SCHEMA-CONSTRAINED
    # ==================================================

    "constrained_partial_extractions": 1,

    "constrained_mapping_errors": 0,

    "constrained_status_certainty_errors": 0,

    "constrained_omissions": 0,

    "constrained_unsupported_inferences": 1,

    "constrained_total_semantic_errors": 2,


    # ==================================================
    # ADJUDICATION NOTES
    # ==================================================

    "baseline_notes": [

        # Partial extraction
        "Shortness of breath was extracted but its approximately "
        "10-day duration was omitted.",

        "The nocturnal shortness-of-breath episode was extracted, "
        "but its timing/resolved context was omitted.",

        # Mapping
        "The recommendation to ensure a recent eye exam was placed "
        "under referrals rather than patient instructions/testing.",

        # Omissions
        "Chest pain denial omitted.",
        "Fever denial omitted.",
        "Chills denial omitted.",
        "Cough denial omitted.",
        "Abdominal/belly pain denial omitted.",
        "Suicidal/homicidal ideation denial omitted."
    ],


    "constrained_notes": [

        # Partial extraction
        "Hemoglobin A1c was correctly extracted as ordered, "
        "but the approximately one-month timing was omitted.",

        # Unsupported inference
        "Insulin was assigned action='continue' even though the "
        "transcript establishes current insulin-pump use but does "
        "not explicitly state a new continue instruction."
    ],


    # ==================================================
    # AMBIGUITIES - NOT COUNTED AS CONFIRMED ERRORS
    # ==================================================

    "annotation_ambiguities": [

        "Acute heart failure exacerbation was labeled "
        "'suspected' by the constrained model. The clinician says "
        "'I think that you are in an acute heart failure "
        "exacerbation,' so certainty is not treated as a confirmed "
        "error.",

        "Body-site labels such as 'lungs' for shortness of breath "
        "are not counted as confirmed semantic errors."
    ]
}


# --------------------------------------------------
# SAVE INTO SEMANTIC RESULT COLLECTION
# --------------------------------------------------

if "heldout_semantic_results" not in globals():
    heldout_semantic_results = {}


heldout_semantic_results[
    semantic_result_D2N093["case_id"]
] = semantic_result_D2N093


# --------------------------------------------------
# SAVE TO DRIVE
# --------------------------------------------------

semantic_results_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_results.json"
)

with open(
    semantic_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        heldout_semantic_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# VERIFY
# --------------------------------------------------

print("=" * 80)
print("D2N093 SEMANTIC ADJUDICATION SAVED")
print("=" * 80)

print(
    "Baseline semantic errors:",
    semantic_result_D2N093[
        "baseline_total_semantic_errors"
    ]
)

print(
    "Schema-constrained semantic errors:",
    semantic_result_D2N093[
        "constrained_total_semantic_errors"
    ]
)

print(
    "Difference:",
    semantic_result_D2N093[
        "baseline_total_semantic_errors"
    ]
    -
    semantic_result_D2N093[
        "constrained_total_semantic_errors"
    ]
)

print(
    "\nSaved to:",
    semantic_results_path
)

D2N093 SEMANTIC ADJUDICATION SAVED
Baseline semantic errors: 9
Schema-constrained semantic errors: 2
Difference: 7

Saved to: /content/drive/MyDrive/LLM_Reliability_Research/heldout_semantic_results.json


In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC REVIEW
# CASE 2: D2N094-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

case_id = "D2N094-virtassist"

# Find the original held-out case
case_data = next(
    case
    for case in held_out_cases
    if case["case_id"] == case_id
)

# Get the already-saved model outputs
result = heldout_generation_results[
    case_id
]


print("=" * 80)
print("SEMANTIC REVIEW - HELD-OUT CASE 2")
print("=" * 80)

print(
    "Case ID:",
    case_id
)


print("\n" + "=" * 80)
print("TRANSCRIPT")
print("=" * 80)

print(
    case_data["transcript"]
)


print("\n" + "=" * 80)
print("REFERENCE NOTE")
print("=" * 80)

print(
    case_data["reference_note"]
)


print("\n" + "=" * 80)
print("BASELINE OUTPUT")
print("=" * 80)

print(
    json.dumps(
        result["baseline_output"],
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    json.dumps(
        result["constrained_output"],
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 80)
print("OUTPUT HASHES")
print("=" * 80)

print(
    "Baseline:",
    result["baseline_output_sha256"]
)

print(
    "Constrained:",
    result["constrained_output_sha256"]
)

SEMANTIC REVIEW - HELD-OUT CASE 2
Case ID: D2N094-virtassist

TRANSCRIPT
[doctor] hey , ms. james . nice to meet you .
[patient] nice to meet you , dr. cooper . how are you ?
[doctor] i'm well . hey , dragon , i'm seeing ms. james . she's a 42-year-old female , and what brings you in today ?
[patient] i hurt my , uh , finger when i was skiing this past weekend .
[doctor] really ?
[patient] yeah . yeah , so , um , i was going down hill , double diamonds , uh , double black diamonds , and i just lost control , and i , you know , flipped down a few ways , but , uh , somewhere along the way , i , i jammed my , my index finger on something . i'm not sure what .
[doctor] okay . so this happened last saturday , you said ?
[patient] it was saturday , yes .
[doctor] okay . so about five days of this right index finger pain .
[patient] mm-hmm .
[doctor] have you taken any medicine for it ?
[patient] i took some ibuprofen . um , did n't really seem to help .
[doctor] okay . have you iced it or pu

In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC ADJUDICATION
# CASE 2: D2N094-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

import json
import os


semantic_result_D2N094 = {

    "case_id":
        "D2N094-virtassist",


    # ==================================================
    # BASELINE
    # ==================================================

    "baseline_partial_extractions": 0,

    "baseline_mapping_errors": 0,

    "baseline_status_certainty_errors": 0,

    "baseline_omissions": 1,

    "baseline_unsupported_inferences": 0,

    "baseline_total_semantic_errors": 1,


    # ==================================================
    # SCHEMA-CONSTRAINED
    # ==================================================

    "constrained_partial_extractions": 0,

    "constrained_mapping_errors": 0,

    "constrained_status_certainty_errors": 0,

    "constrained_omissions": 1,

    "constrained_unsupported_inferences": 2,

    "constrained_total_semantic_errors": 3,


    # ==================================================
    # BASELINE NOTES
    # ==================================================

    "baseline_notes": [

        "Mobic 15 mg once daily was explicitly prescribed "
        "during the encounter and appears under medication_changes, "
        "but it was omitted from the top-level medications list."
    ],


    # ==================================================
    # SCHEMA-CONSTRAINED NOTES
    # ==================================================

    "constrained_notes": [

        "The plan to reassess in two weeks and consider hand "
        "therapy if needed was not preserved as a conditional "
        "follow-up instruction.",

        "Ibuprofen was assigned action='stop', but the clinician "
        "did not explicitly instruct the patient to stop ibuprofen.",

        "Miralax was assigned action='continue', but the clinician "
        "only established that the patient currently takes Miralax "
        "and did not explicitly issue a continue instruction."
    ],


    # ==================================================
    # AMBIGUITIES - NOT COUNTED
    # ==================================================

    "annotation_ambiguities": [

        "The transcript describes pain when the clinician presses "
        "and squeezes different parts of the finger, but the spoken "
        "transcript does not explicitly identify each joint. "
        "Therefore differences in MCP/DIP specificity are not "
        "counted as confirmed errors.",

        "The constrained chief complaint does not repeat the "
        "right-sided laterality, but the visit_reason and symptom "
        "fields preserve it; this is not counted as a confirmed error.",

        "Representing finger splinting under procedures is accepted "
        "as a reasonable schema mapping."
    ]
}


# --------------------------------------------------
# LOAD EXISTING SAVED RESULTS FIRST
# This protects D2N093 if Colab memory is lost.
# --------------------------------------------------

semantic_results_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_results.json"
)


if os.path.exists(
    semantic_results_path
):

    with open(
        semantic_results_path,
        "r",
        encoding="utf-8"
    ) as f:

        heldout_semantic_results = (
            json.load(f)
        )

else:

    heldout_semantic_results = {}


# --------------------------------------------------
# ADD CASE 2
# --------------------------------------------------

heldout_semantic_results[
    "D2N094-virtassist"
] = semantic_result_D2N094


# --------------------------------------------------
# SAVE BACK TO DRIVE
# --------------------------------------------------

with open(
    semantic_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        heldout_semantic_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# VERIFY
# --------------------------------------------------

print("=" * 80)
print("D2N094 SEMANTIC ADJUDICATION SAVED")
print("=" * 80)

print(
    "Baseline semantic errors:",
    semantic_result_D2N094[
        "baseline_total_semantic_errors"
    ]
)

print(
    "Schema-constrained semantic errors:",
    semantic_result_D2N094[
        "constrained_total_semantic_errors"
    ]
)

print(
    "Difference:",
    semantic_result_D2N094[
        "baseline_total_semantic_errors"
    ]
    -
    semantic_result_D2N094[
        "constrained_total_semantic_errors"
    ]
)

print(
    "\nSemantic cases saved:",
    list(
        heldout_semantic_results.keys()
    )
)

print(
    "\nSaved to:",
    semantic_results_path
)

D2N094 SEMANTIC ADJUDICATION SAVED
Baseline semantic errors: 1
Schema-constrained semantic errors: 3
Difference: -2

Semantic cases saved: ['D2N093-virtassist', 'D2N094-virtassist']

Saved to: /content/drive/MyDrive/LLM_Reliability_Research/heldout_semantic_results.json


In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC REVIEW
# CASE 3: D2N095-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

import json


case_id = "D2N095-virtassist"


# --------------------------------------------------
# 1. FIND ORIGINAL CASE
# --------------------------------------------------

case_data = next(
    case
    for case in held_out_cases
    if case["case_id"] == case_id
)


# --------------------------------------------------
# 2. GET ALREADY-SAVED MODEL OUTPUTS
# --------------------------------------------------

result = heldout_generation_results[
    case_id
]


# --------------------------------------------------
# 3. DISPLAY EVERYTHING NEEDED FOR ADJUDICATION
# --------------------------------------------------

print("=" * 80)
print("SEMANTIC REVIEW - HELD-OUT CASE 3")
print("=" * 80)

print(
    "Case ID:",
    case_id
)


print("\n" + "=" * 80)
print("TRANSCRIPT")
print("=" * 80)

print(
    case_data["transcript"]
)


print("\n" + "=" * 80)
print("REFERENCE NOTE")
print("=" * 80)

print(
    case_data["reference_note"]
)


print("\n" + "=" * 80)
print("BASELINE OUTPUT")
print("=" * 80)

print(
    json.dumps(
        result["baseline_output"],
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 80)
print("SCHEMA-CONSTRAINED OUTPUT")
print("=" * 80)

print(
    json.dumps(
        result["constrained_output"],
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 80)
print("OUTPUT HASHES")
print("=" * 80)

print(
    "Baseline:",
    result["baseline_output_sha256"]
)

print(
    "Constrained:",
    result["constrained_output_sha256"]
)

SEMANTIC REVIEW - HELD-OUT CASE 3
Case ID: D2N095-virtassist

TRANSCRIPT
[doctor] hi , cheryl . how are you ?
[patient] i'm doing well . how are you ?
[doctor] i'm doing well . so i know the nurse told you a little bit about dax . i'd like to tell dax about you .
[patient] okay .
[doctor] cheryl is a 34-year-old female with a past medical history significant for hypertension , who presents today with back pain . cheryl , what happened to your back ?
[patient] so i've been walking a lot lately . i've been walking to ... 30 minutes to an hour or so a day . and all of a sudden , um , when i was walking , my , um , back just kind of seized up on me . and i do n't really know what it was . maybe i was going a little bit faster . but it just all kind of clenched .
[doctor] okay . so you felt like , maybe like a spasm or something like that ?
[patient] yeah .
[doctor] okay . and how many days ago was that ?
[patient] that was about six days ago now .
[doctor] okay . and what have you taken fo

In [ ]:
# --------------------------------------------------
# HELD-OUT SEMANTIC ADJUDICATION
# CASE 3: D2N095-virtassist
# NO GEMINI CALLS
# --------------------------------------------------

import json
import os


semantic_result_D2N095 = {

    "case_id":
        "D2N095-virtassist",


    # ==================================================
    # BASELINE
    # ==================================================

    "baseline_partial_extractions": 2,

    "baseline_mapping_errors": 0,

    "baseline_status_certainty_errors": 0,

    "baseline_omissions": 3,

    "baseline_unsupported_inferences": 0,

    "baseline_total_semantic_errors": 5,


    # ==================================================
    # SCHEMA-CONSTRAINED
    # ==================================================

    "constrained_partial_extractions": 1,

    "constrained_mapping_errors": 0,

    "constrained_status_certainty_errors": 0,

    "constrained_omissions": 0,

    "constrained_unsupported_inferences": 0,

    "constrained_total_semantic_errors": 1,


    # ==================================================
    # BASELINE NOTES
    # ==================================================

    "baseline_notes": [

        "Lower back pain was extracted, but the supported "
        "bilateral distribution and approximately six-day "
        "duration were not preserved.",

        "Back spasm/stiffness was extracted, but its approximately "
        "six-day temporal context was not preserved.",

        "The explicit denial of numbness or tingling in the legs "
        "or feet was omitted.",

        "The explicit denial of lower-extremity weakness was omitted.",

        "Meloxicam 15 mg once daily was explicitly prescribed "
        "but was omitted from the top-level medications list."
    ],


    # ==================================================
    # SCHEMA-CONSTRAINED NOTES
    # ==================================================

    "constrained_notes": [

        "Back pain was correctly extracted with its six-day "
        "duration, but the transcript-supported bilateral "
        "distribution was not explicitly preserved."
    ],


    # ==================================================
    # AMBIGUITIES - NOT COUNTED
    # ==================================================

    "annotation_ambiguities": [

        "Lumbar strain certainty='suspected' is accepted because "
        "the clinician states 'I think you have a lumbar strain.'",

        "Physical-exam localization to the right lateral lumbar "
        "spine does not conflict with the patient's bilateral "
        "subjective pain description.",

        "Physical therapy under specialist_follow_up is accepted "
        "as a reasonable mapping under the frozen schema."
    ]
}


# --------------------------------------------------
# LOAD EXISTING SAVED SEMANTIC RESULTS
# --------------------------------------------------

semantic_results_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_results.json"
)


if os.path.exists(
    semantic_results_path
):

    with open(
        semantic_results_path,
        "r",
        encoding="utf-8"
    ) as f:

        heldout_semantic_results = json.load(f)

else:

    heldout_semantic_results = {}


# --------------------------------------------------
# ADD CASE 3
# --------------------------------------------------

heldout_semantic_results[
    "D2N095-virtassist"
] = semantic_result_D2N095


# --------------------------------------------------
# SAVE TO DRIVE
# --------------------------------------------------

with open(
    semantic_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        heldout_semantic_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# VERIFY
# --------------------------------------------------

print("=" * 80)
print("D2N095 SEMANTIC ADJUDICATION SAVED")
print("=" * 80)

print(
    "Baseline semantic errors:",
    semantic_result_D2N095[
        "baseline_total_semantic_errors"
    ]
)

print(
    "Schema-constrained semantic errors:",
    semantic_result_D2N095[
        "constrained_total_semantic_errors"
    ]
)

print(
    "Difference:",
    semantic_result_D2N095[
        "baseline_total_semantic_errors"
    ]
    -
    semantic_result_D2N095[
        "constrained_total_semantic_errors"
    ]
)

print(
    "\nSemantic cases saved:",
    list(
        heldout_semantic_results.keys()
    )
)

D2N095 SEMANTIC ADJUDICATION SAVED
Baseline semantic errors: 5
Schema-constrained semantic errors: 1
Difference: 4

Semantic cases saved: ['D2N093-virtassist', 'D2N094-virtassist', 'D2N095-virtassist']


In [ ]:
# --------------------------------------------------
# INTERIM HELD-OUT SEMANTIC SUMMARY
# COMPLETED SEMANTIC CASES ONLY
# NO GEMINI CALLS
# --------------------------------------------------

import json
import pandas as pd


# --------------------------------------------------
# 1. LOAD SAVED SEMANTIC RESULTS FROM DRIVE
# --------------------------------------------------

semantic_results_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_results.json"
)

with open(
    semantic_results_path,
    "r",
    encoding="utf-8"
) as f:

    saved_semantic_results = json.load(f)


# --------------------------------------------------
# 2. BUILD CASE-LEVEL TABLE
# --------------------------------------------------

case_rows = []

for case_id, result in saved_semantic_results.items():

    case_rows.append({

        "case_id":
            case_id,

        "baseline_errors":
            result[
                "baseline_total_semantic_errors"
            ],

        "schema_constrained_errors":
            result[
                "constrained_total_semantic_errors"
            ]
    })


semantic_case_df = pd.DataFrame(
    case_rows
)

semantic_case_df[
    "error_difference"
] = (

    semantic_case_df[
        "baseline_errors"
    ]

    -

    semantic_case_df[
        "schema_constrained_errors"
    ]
)


print("=" * 80)
print("INTERIM HELD-OUT SEMANTIC RESULTS")
print("=" * 80)

display(
    semantic_case_df
)


# --------------------------------------------------
# 3. CATEGORY TOTALS
# --------------------------------------------------

categories = [
    "partial_extractions",
    "mapping_errors",
    "status_certainty_errors",
    "omissions",
    "unsupported_inferences"
]


category_rows = []


for category in categories:

    baseline_total = sum(
        result[
            f"baseline_{category}"
        ]
        for result
        in saved_semantic_results.values()
    )

    constrained_total = sum(
        result[
            f"constrained_{category}"
        ]
        for result
        in saved_semantic_results.values()
    )


    category_rows.append({

        "error_category":
            category,

        "baseline":
            baseline_total,

        "schema_constrained":
            constrained_total,

        "difference":
            baseline_total
            - constrained_total
    })


semantic_category_df = pd.DataFrame(
    category_rows
)


print("\n" + "=" * 80)
print("ERROR CATEGORY TOTALS")
print("=" * 80)

display(
    semantic_category_df
)


# --------------------------------------------------
# 4. OVERALL TOTALS
# --------------------------------------------------

baseline_total = sum(
    result[
        "baseline_total_semantic_errors"
    ]
    for result
    in saved_semantic_results.values()
)

constrained_total = sum(
    result[
        "constrained_total_semantic_errors"
    ]
    for result
    in saved_semantic_results.values()
)


absolute_reduction = (
    baseline_total
    - constrained_total
)


percent_reduction = (
    absolute_reduction
    / baseline_total
    * 100
)


print("\n" + "=" * 80)
print("INTERIM SEMANTIC FINDING")
print("=" * 80)

print(
    "Completed semantic cases:",
    len(saved_semantic_results)
)

print(
    "Baseline semantic errors:",
    baseline_total
)

print(
    "Schema-constrained semantic errors:",
    constrained_total
)

print(
    "Absolute reduction:",
    absolute_reduction
)

print(
    "Observed semantic-error reduction:",
    f"{percent_reduction:.1f}%"
)

INTERIM HELD-OUT SEMANTIC RESULTS


,case_id,baseline_errors,schema_constrained_errors,error_difference
0,D2N093-virtassist,9,2,7
1,D2N094-virtassist,1,3,-2
2,D2N095-virtassist,5,1,4



ERROR CATEGORY TOTALS


,error_category,baseline,schema_constrained,difference
0,partial_extractions,4,2,2
1,mapping_errors,1,0,1
2,status_certainty_errors,0,0,0
3,omissions,10,1,9
4,unsupported_inferences,0,3,-3



INTERIM SEMANTIC FINDING
Completed semantic cases: 3
Baseline semantic errors: 15
Schema-constrained semantic errors: 6
Absolute reduction: 9
Observed semantic-error reduction: 60.0%


In [ ]:
# --------------------------------------------------
# SAVE INTERIM HELD-OUT FINDINGS
# NO GEMINI CALLS
# --------------------------------------------------

import json
import os


interim_heldout_summary = {

    "completed_semantic_cases": 3,

    "case_ids": [
        "D2N093-virtassist",
        "D2N094-virtassist",
        "D2N095-virtassist"
    ],

    "baseline_total_semantic_errors": 15,
    "schema_constrained_total_semantic_errors": 6,

    "absolute_semantic_error_reduction": 9,
    "percent_semantic_error_reduction": 60.0,

    "category_totals": {

        "partial_extractions": {
            "baseline": 4,
            "schema_constrained": 2
        },

        "mapping_errors": {
            "baseline": 1,
            "schema_constrained": 0
        },

        "status_certainty_errors": {
            "baseline": 0,
            "schema_constrained": 0
        },

        "omissions": {
            "baseline": 10,
            "schema_constrained": 1
        },

        "unsupported_inferences": {
            "baseline": 0,
            "schema_constrained": 3
        }
    },

    "interpretation":
        "Interim held-out result only. "
        "Schema-constrained generation shows fewer total semantic "
        "errors and substantially fewer omissions, but more unsupported "
        "inferences. Final conclusions require the full held-out set."
}


interim_summary_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "interim_heldout_summary.json"
)


with open(
    interim_summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        interim_heldout_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print("=" * 80)
print("INTERIM HELD-OUT SUMMARY SAVED")
print("=" * 80)

print(
    "Completed semantic cases:",
    interim_heldout_summary[
        "completed_semantic_cases"
    ]
)

print(
    "Baseline semantic errors:",
    interim_heldout_summary[
        "baseline_total_semantic_errors"
    ]
)

print(
    "Schema-constrained semantic errors:",
    interim_heldout_summary[
        "schema_constrained_total_semantic_errors"
    ]
)

print(
    "Observed reduction:",
    str(
        interim_heldout_summary[
            "percent_semantic_error_reduction"
        ]
    ) + "%"
)

print(
    "\nSaved to:",
    interim_summary_path
)

INTERIM HELD-OUT SUMMARY SAVED
Completed semantic cases: 3
Baseline semantic errors: 15
Schema-constrained semantic errors: 6
Observed reduction: 60.0%

Saved to: /content/drive/MyDrive/LLM_Reliability_Research/interim_heldout_summary.json


In [ ]:
# --------------------------------------------------
# RESUME HELD-OUT CASE 4
# D2N096-virtassist
# ONLY MISSING CONSTRAINED OUTPUT SHOULD RUN
# --------------------------------------------------

heldout_case4_result = safely_run_heldout_case(3)

RUNNING HELD-OUT CASE 4: D2N096-virtassist
Frozen model: gemini-3.5-flash

✅ Baseline already completed.
Skipping baseline API call.

Running schema-constrained...

❌ Schema-constrained API call failed.
Any successful baseline output has already been saved.

RATE LIMIT REACHED
Case: D2N096-virtassist
Baseline complete: True
Constrained complete: False

✅ All successful work remains saved to Drive.
No completed generation needs to be repeated.


In [ ]:
# --------------------------------------------------
# VERIFY NEXT RESUME POINT
# NO GEMINI CALLS
# --------------------------------------------------

case_id = "D2N096-virtassist"

result = heldout_generation_results.get(
    case_id,
    {}
)

print("=" * 80)
print("NEXT EXPERIMENT RESUME POINT")
print("=" * 80)

print("Case:", case_id)

print(
    "Baseline complete:",
    result.get(
        "baseline_complete",
        False
    )
)

print(
    "Constrained complete:",
    result.get(
        "constrained_complete",
        False
    )
)

print("\nCompleted semantic cases:")
print(
    list(
        heldout_semantic_results.keys()
    )
)

print("\nNEXT REQUIRED GENERATION:")
print(
    "D2N096 schema-constrained output only"
)

NEXT EXPERIMENT RESUME POINT
Case: D2N096-virtassist
Baseline complete: True
Constrained complete: False

Completed semantic cases:
['D2N093-virtassist', 'D2N094-virtassist', 'D2N095-virtassist']

NEXT REQUIRED GENERATION:
D2N096 schema-constrained output only


In [ ]:
# --------------------------------------------------
# LOAD CURRENT SAVED EXPERIMENT STATE
# SOURCE OF TRUTH
# NO GEMINI CALLS
# --------------------------------------------------

import json
import os


checkpoint_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_experiment_checkpoint.json"
)

semantic_path = (
    "/content/drive/MyDrive/"
    "LLM_Reliability_Research/"
    "heldout_semantic_results.json"
)


# --------------------------------------------------
# LOAD GENERATION RESULTS
# --------------------------------------------------

with open(
    checkpoint_path,
    "r",
    encoding="utf-8"
) as f:

    checkpoint = json.load(f)


heldout_generation_results = checkpoint[
    "heldout_generation_results"
]


# --------------------------------------------------
# LOAD SEMANTIC RESULTS
# --------------------------------------------------

if os.path.exists(semantic_path):

    with open(
        semantic_path,
        "r",
        encoding="utf-8"
    ) as f:

        heldout_semantic_results = json.load(f)

else:

    heldout_semantic_results = {}


# --------------------------------------------------
# DISPLAY CURRENT SOURCE OF TRUTH
# --------------------------------------------------

print("=" * 80)
print("CURRENT SAVED EXPERIMENT STATE")
print("=" * 80)

print(
    "Model:",
    checkpoint["heldout_model"]
)

print(
    "\nGeneration cases saved:",
    len(heldout_generation_results)
)

for case_id, result in heldout_generation_results.items():

    print("\n", case_id)

    print(
        "  Baseline complete:",
        result.get("baseline_complete", False)
    )

    print(
        "  Baseline schema errors:",
        result.get("baseline_schema_error_count")
    )

    print(
        "  Constrained complete:",
        result.get("constrained_complete", False)
    )

    print(
        "  Constrained schema errors:",
        result.get("constrained_schema_error_count")
    )


print(
    "\nSemantic cases saved:",
    len(heldout_semantic_results)
)

for case_id, result in heldout_semantic_results.items():

    print(
        case_id,
        ":",
        result["baseline_total_semantic_errors"],
        "→",
        result["constrained_total_semantic_errors"]
    )

CURRENT SAVED EXPERIMENT STATE
Model: gemini-3.5-flash

Generation cases saved: 4

 D2N093-virtassist
  Baseline complete: True
  Baseline schema errors: 29
  Constrained complete: True
  Constrained schema errors: 0

 D2N094-virtassist
  Baseline complete: True
  Baseline schema errors: 12
  Constrained complete: True
  Constrained schema errors: 0

 D2N095-virtassist
  Baseline complete: True
  Baseline schema errors: 27
  Constrained complete: True
  Constrained schema errors: 0

 D2N096-virtassist
  Baseline complete: True
  Baseline schema errors: 14
  Constrained complete: False
  Constrained schema errors: None

Semantic cases saved: 3
D2N093-virtassist : 9 → 2
D2N094-virtassist : 1 → 3
D2N095-virtassist : 5 → 1
